In [1]:
# path: "C:\AxisRecordings\Optical_Flow\tracking_md\skeletal_siblings\OD_distinct_models\Step7_full.ipynb"

In [2]:
import json

def count_code_lines(notebook_path):
    with open(notebook_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    total_lines = 0
    per_cell = []

    for cell in nb["cells"]:
        if cell["cell_type"] == "code":
            lines = sum(1 for line in cell["source"] if line.strip())
            per_cell.append(lines)
            total_lines += lines

    return total_lines, per_cell


notebook = "Step7_full.ipynb"

total, per_cell = count_code_lines(notebook)

print("Total code lines:", total)
print("Lines per cell:", per_cell)


cells = [0,0,0,0,0,0,97,0,0,59,2,0,1,3,0,0,402,0,7,0,0,16,0,0,189,0,0,4,0,0,0,186,0,0,18,0,72,0,0,0,73,0,64,0,0,0,0,0,368,0,0,0,323,0,49,0,0,0,374,0,3,6,0,0,0,429,0,0,6,0,2,0,2,0,2,0,3,15,1,0,25,3,0,0,16,0,2,0,0,1,20,0,1,0,5,2,0,9,0,0,1,0,9,28,7,4,1,4,0,4,0,1,1,0,0,247,0,1,0,0,141,0,0,120,0,0,670,0,85,0,0,176,0,0,238,0,254,0,0,1,2,3,0,10,1,0,0,2,0,35,230,0,54,0,9,0,0,1,0,0,410,0,1,35,0,0,19,0,61,5,0,0,0,0]
print("\nLines of code in notebook:", sum(cells))


Total code lines: 6085
Lines per cell: [1, 18, 0, 0, 0, 0, 0, 0, 0, 97, 0, 0, 59, 2, 0, 1, 3, 0, 0, 445, 0, 99, 0, 7, 0, 0, 16, 0, 0, 189, 0, 0, 4, 0, 0, 0, 186, 0, 0, 18, 0, 72, 0, 0, 0, 73, 0, 64, 0, 0, 0, 0, 0, 368, 0, 0, 0, 323, 0, 49, 0, 0, 0, 374, 0, 3, 6, 0, 0, 0, 429, 0, 0, 6, 0, 2, 0, 2, 0, 2, 0, 7, 1, 0, 36, 26, 0, 8, 0, 0, 16, 0, 2, 0, 0, 1, 20, 0, 1, 0, 5, 2, 0, 9, 0, 0, 1, 0, 9, 28, 7, 4, 1, 4, 0, 4, 0, 1, 1, 0, 340, 0, 1, 0, 0, 141, 0, 0, 120, 0, 0, 671, 0, 85, 0, 0, 176, 0, 0, 238, 0, 284, 0, 0, 1, 2, 3, 0, 10, 0, 0, 1, 0, 34, 2, 0, 12, 0, 0, 2, 0, 35, 230, 0, 54, 0, 0, 0, 1, 0, 0, 410, 0, 0, 35, 0, 0, 19, 0, 61, 5, 0, 0, 0, 0]

Lines of code in notebook: 5731


# <font color = lime> Step 7: Stand-Alone (Gap-Tolerant Multi-Frame Tracking)

    System Description
    
- Step 7 implements a complete, production-ready multi-object tracking system using standard signal detection principles from radar tracking and video surveillance. The system operates as a self-contained pipeline that processes pre-recorded video to detect, track, and evaluate moving objects (specifically eagles in flight). It combines classical computer vision techniques (frame differencing, morphological operations, contour detection) with temporal association logic to build coherent trajectories across frames, even in the presence of detection gaps caused by occlusions or lighting variations. The architecture follows conventional multi-target tracking design patterns, making it maintainable, explainable, and suitable for deployment in wildlife monitoring applications.

- Core Methodology
The tracking algorithm uses a temporal window approach where tracks persist for MAX_MISS = FRAME_WINDOW(4) + BUFFER(1) = 5 consecutive frames without observation before termination. Motion prediction employs a Linear Constant Velocity Model (x(t+Δt) = x(t) + v·Δt) with adaptive search radius expansion to handle prediction uncertainty during gaps. Data association uses Greedy Nearest Neighbor matching, which is computationally efficient and appropriate for low-density scenarios (1-3 objects per frame). The system applies four temporal gates—minimum track length, observation density, displacement range, and directional constraints—to filter spurious detections. Quality scoring combines trajectory length, heading alignment with expected flight direction, and vertical corridor occupancy into a composite confidence metric that accounts for data completeness (observation density).


- Design Philosophy & Extensibility
Step 7 prioritizes simplicity and standards compliance over custom heuristics, explicitly surfacing design choices such as motion models and association strategies with clear upgrade paths. The linear motion model is documented as appropriate for straight-line flight with recommendations to upgrade to Kalman Filtering for curved trajectories or longer gaps. Similarly, the greedy association strategy includes conditions under which the Hungarian Algorithm could become an option used with (crossing tracks, high object density). The system includes a visual validation component (Cell 7.12) that generates annotated video of the best-ranked track, allowing human verification that the algorithm correctly identified the target object rather than noise. All configuration parameters are consolidated in Cell 7.2 with unused legacy variables explicitly marked.  This entire pipeline, named Step 7, can run independently without requiring previous processing steps, making it suitable for both research iteration and operational deployment.


DESIGN PHILOSOPHY:  (This might be improved ? )
------------------
This pipeline implementation follows CONVENTIONAL signal detection / multi-target tracking:

1. **Temporal Association Window**:
   - Track persists for MAX_MISS = FRAME_WINDOW + BUFFER frames without observation
   - No complex positional constraints
   - Standard in: radar tracking, video surveillance, sports analytics

2. **Motion Model** (Current: Linear Constant Velocity):
   - Prediction: x(t+Δt) = x(t) + v·Δt
   - Appropriate for: straight-line motion, short gaps (<5 frames)
   - Upgrade to Kalman if: curved trajectories, longer gaps, uncertainty quantification needed

3. **Association Strategy** (Current: Greedy Nearest Neighbor):
   - Simple distance-based assignment
   - Appropriate for: low object density, well-separated tracks
   - Upgrade to Hungarian if: crossing trajectories, >5 objects per frame

### <font color = lime> Advanced Description: What this pipeline actually is ...

### <font color = yellow> 🧠 SO WHAT’S THE RIGHT LABEL?

- A: 🏆 Temporal Motion Validation Pipeline (Most accurate description)
  - A pipeline whose job is to confirm whether a moving pixel cluster maintains enough temporal coherence to qualify as a real trackable entity.
  - 

- B: 🏆 A Motion-First Tracker (Also true description)
  - You’re not tracking objects - you’re tracking motion signatures.
  - 

- C: 🏆 Lightweight Greedy Tracker + Heavy Temporal Gating Engine
- 
  - 

  

#### <font color = yellow> 🔥 FINAL CHISELED ANSWER (use this in your own documentation):

  - The MD pipeline is primarily a temporal-motion validation system with a lightweight tracker used only to structure the motion evidence.

  - It tracks motion signatures rather than objects, and it validates them based on temporal coherence, density, heading stability, and gap structure.

  - In other words: It’s a temporal validator first, and an object tracker second.




## <font color = yellow> 7.1: Import and Fix Description (Fully Self-Contained)

#### <font color = lime> Step 7 - Assumptions (see 7.8)


    Self-contained by design (good software engineering):

    ✅ Step 7 is a complete, standalone tracking system
    ✅ Can be used independently for production deployment
    ✅ No need to run Step 4 first (Step 7 includes detection)
    
    Evidence:
    
    All detection logic re-implemented in Cell 7.8 (run_detection())
    No imports from previous steps (no from step4 import detections)
    No pickled/cached data (reads video from disk)
    All variables explicitly defined in Cell 7.2
    Deterministic output (same config → same results)
    
    

In [3]:
# 7.1 Imports

#  Req'd for using rpy2.  Done once/ Done
# pip install --upgrade --force-reinstall numpy
# pip install --upgrade --force-reinstall numpy pandas
# pip install --upgrade rpy2

"""
================================================================================
📌 CELL PURPOSE
--------------------------------------------------------------------------------
This cell establishes the foundational environment for Step 7 by:
- Importing all required libraries and setting visualization styles
- Documenting architectural corrections from the previous version
- Surfacing design choices for motion models and association strategies

Key considerations carried forward:
- Standard temporal window with buffer
- Simple miss counter for track persistence
- Motion model: Linear constant velocity (upgrade path to Kalman if needed)
- Association strategy: Greedy nearest neighbor (upgrade path to Hungarian if needed)

No pipeline execution occurs here; it is strictly setup and configuration.

📌 ROLE IN PIPELINE
--------------------------------------------------------------------------------
[Reserved for later refinement once all Step 7.n cells are evaluated]
================================================================================
"""


import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import math
import csv
import os
from typing import List, Tuple, Set, Dict, Any, Optional


from dataclasses import dataclass, field
from collections import deque

import copy

import logging
from pathlib import Path
from datetime import datetime
import json
from pprint import pformat


import ast
import re


import rpy2


import os
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
sns.set_style("whitegrid")

print("✓ Imports loaded")
print("\n" + "=" * 70)
print("STEP 7: STANDARD TEMPORAL ASSOCIATION TRACKING")
print("=" * 70)

print("\n🔧 Architectural:")
print("   1. Removed custom gap position logic → Standard temporal window")
print("   2. Removed post-hoc stitching → Proper temporal persistence")
print("   3. Simplified data structure → No redundant fields")
print("   4. Surfaced design choices → Linear model, greedy association")

print("\n📐 Design Choices (Surfaced  / Noted w/ Msg):")
print("   • Motion Model: Linear Constant Velocity")
print("     - Upgrade to Kalman if: curved paths, uncertainty needed")
print("   • Association: Greedy Nearest Neighbor")
print("     - Upgrade to Hungarian if: crossing tracks, high density")

print("\n" + "=" * 70)


# Define once / to use: print this block anywhere in notebook
info_block =  """  
\nStep 7.1: Standard Temporal Association (print block)

     📐 DESIGN CHOICES SURFACED:  
   \n   1. MOTION MODEL: Linear Constant Velocity  
         Current: x t+Δt  = x t  + v·Δt  
         ✅ Appropriate for:  
            - Straight-line flight  gliding   
            - Short gaps  < 5 frames   
            - Low computational overhead  
         ⚠️  Upgrade to Kalman Filter if:  
            - Curved/maneuvering flight  
            - Need uncertainty quantification  
            - Gaps > 5 frames  
            - Multi-sensor fusion needed  

   \n   2. ASSOCIATION STRATEGY: Greedy Nearest Neighbor  
         Current: Assign detection to closest track Centroid prediction  
         ✅ Appropriate for:  
            - Low object density  1-3 per frame   
            - Well-separated objects  >100px   
            - Real-time requirements  
         ⚠️  Upgrade to Hungarian Algorithm if:  
            - Crossing trajectories  
            - Object density > 5 per frame  
            - Identity switches problematic  


   \n🔜 FUTURE ENHANCEMENTS  If Needed :  
      • Kalman Filter: Better motion prediction for curved paths  
      • Hungarian Assignment: Optimal matching for high-density scenarios  
      • Appearance Model: Color/texture consistency for re-identification  
      • Multi-Hypothesis Tracking: Maintain multiple candidates for ambiguous cases  
   """

# ============= call the print block ======================================
print(info_block)

✓ Imports loaded

STEP 7: STANDARD TEMPORAL ASSOCIATION TRACKING

🔧 Architectural:
   1. Removed custom gap position logic → Standard temporal window
   2. Removed post-hoc stitching → Proper temporal persistence
   3. Simplified data structure → No redundant fields
   4. Surfaced design choices → Linear model, greedy association

📐 Design Choices (Surfaced  / Noted w/ Msg):
   • Motion Model: Linear Constant Velocity
     - Upgrade to Kalman if: curved paths, uncertainty needed
   • Association: Greedy Nearest Neighbor
     - Upgrade to Hungarian if: crossing tracks, high density

  

Step 7.1: Standard Temporal Association (print block)

     📐 DESIGN CHOICES SURFACED:  
   
   1. MOTION MODEL: Linear Constant Velocity  
         Current: x t+Δt  = x t  + v·Δt  
         ✅ Appropriate for:  
            - Straight-line flight  gliding   
            - Short gaps  < 5 frames   
            - Low computational overhead  
         ⚠️  Upgrade to Kalman Filter if:  
            - Curved/

In [4]:
# ======================== logger ========================================================

def init_notebook_logger(
    name: str = "nb_logger",
    subfolder: str = "notebook_logs",
    level: int = logging.INFO,
):
    """
    Create a notebook-friendly logger:

    - Logs to <CWD>/<subfolder>/<name>_<timestamp>.log
    - Still echoes to notebook (StreamHandler)
    - Adds log.silent(msg, note="...") for file-only logging.

     
    FMI:
        CRITICAL = 50
        ERROR    = 40
        WARNING  = 30
        INFO     = 20
        DEBUG    = 10
        NOTSET   = 0    

    """
    
    # Where the notebook is running
    base_dir = Path.cwd()
    log_dir = base_dir / subfolder
    log_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    logfile = log_dir / f"{name}_{ts}.log"

    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(level)
    logger.propagate = False

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                            "%H:%M:%S")

    # File handler
    fh = logging.FileHandler(logfile, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Stream handler (to notebook)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    # ---- Attach a .silent() helper to this logger ----
    def silent(msg, level=logging.INFO, note="(data logged to file)"):
        """
        Write full 'msg' ONLY to the log file.
        Show only a short note in the notebook.
        """
        # Write to file handler(s) only
        for handler in logger.handlers:
            if isinstance(handler, logging.FileHandler):
                record = logger.makeRecord(
                    logger.name, level, fn="", lno=0,
                    msg=msg, args=None, exc_info=None
                )
                handler.emit(record)

        # Minimal notebook feedback
        print(note)

    # Attach as a method: log.silent(...)
    logger.silent = silent  # type: ignore[attr-defined]

    logger.info(f"Notebook logger started → {logfile}")
    return logger, logfile

    

In [5]:
# Start log session for this notebook run 
log, logfile = init_notebook_logger("Step7_notebook_logdata")


18:37:30 | INFO | Notebook logger started → C:\Axis_code_projects\OF_vs_Step7\Code\notebook_logs\Step7_notebook_logdata_20260101_183730.log


In [6]:
log.info("Step 7: starting analysis")

18:37:30 | INFO | Step 7: starting analysis


In [7]:
# log.info("This is logged")
# log.warning("Motion threshold unusually low")
# log.error("Track length mismatch detected")


## <font color = yellow> 7.2: Step 7 - Configurations (all) for Step 7 MD pipeline


#### <font color = lime>  Live vs Recorded Video: The Foundational Gate

    Central config block: MD thresholds, gating toggles, ROI params, speed bands, etc., for Step-7. One source of truth for all knobs used below.

In [8]:
# =================================================================================================
# STEP 7.2: CONFIGURATION (STANDARD SIGNAL DETECTION APPROACH)
# =================================================================================================

"""
================================================================================
📌 CELL PURPOSE
--------------------------------------------------------------------------------
This cell centralizes all configuration for Step 7. It defines user-adjustable
default settings that govern preprocessing, temporal gating, motion prediction,
association strategy, quality scoring, ROI filtering, and output/export controls.

Key considerations carried forward:
- Single source of truth for runtime parameters
- Explicit separation of active, disabled, and legacy (non-usable) settings
- Human-readable configuration summary for reproducibility

Additional design features:
- Strict validation checks to enforce safe parameter ranges
- Printed summary for transparency and auditability
- Default configuration dictionary to enable change tracking
- Diagnostic toggles for retro stitching and interpolation analysis

No pipeline execution occurs here; it is strictly setup and configuration.

📌 ROLE IN PIPELINE
--------------------------------------------------------------------------------
[Reserved for later refinement once all Step 7.n cells are evaluated]
================================================================================
"""


VIDEO_PATH = r"C:\AxisRecordings\Optical_Flow\videos\big_bird_R2L.mkv"


# "C:\AxisRecordings\Optical_Flow\videos\1010-1651-Two-small-birds-complex_flight.mkv"

# "C:\AxisRecordings\keep_videos\Unseen_video\reqs-cleaning\092-1917.mkv"

# "C:\AxisRecordings\Optical_Flow\videos\1010-1641-2-small.mkv"

# "C:\AxisRecordings\Optical_Flow\videos\1010-1641-2-115-small.mkv"

# =============================================================================
# "C:\AxisRecordings\Optical_Flow\videos\1010-1641-1_clean-small.mkv"
# renamed: 
#  "C:\AxisRecordings\Optical_Flow\videos\1-smallObject-R2L-bankingTurn_L2R.mkv"
# =============================================================================

# "C:\AxisRecordings\Optical_Flow\videos\1010-1641-1_clean-small.mkv"

#  r "C:\AxisRecordings\Optical_Flow\videos\1010-1651-small-birds-short333.mkv"

#  "C:\AxisRecordings\Optical_Flow\videos\big_bird_R2L.mkv"


# =================================================================================================
# === Steps 1-6 Configuration (inherited) ===
# =================================================================================================

# --- Frame Preprocessing Parameters (ACTIVELY USED in run_detection()) ---
BLUR_KERNEL = (5, 5)          # Default = (5,5)
USE_MORPH_OPEN = True          # Default = True
USE_MORPH_CLOSE = True         # Default = True
MORPH_KERNEL = None           # Default = None

# --- Detection Threshold Sweeps (USABLE BUT NOT CURRENTLY USED) ---
diff_threshold_values = [2.0, 2.30]      # Default = [2.0, 2.30]
min_area_values = [51, 65]               # Default = [51, 65] 
centroid_thresholds = [35, 50]          # Default = [35, 50] 

DIFF_THRESHOLD = 2
# diff_threshold = 2 
MIN_AREA = 51
# min_area = 51
CENTROID_THRESHOLD = 35
# centroid_threshold = 35



# --- Step 4 Legacy Parameters (NON-USABLE RELICS) ---
TAU = 1.4           # ❌ NON-USABLE RELIC
L1 = 0.7            # ❌ NON-USABLE RELIC
L2 = 0.3            # ❌ NON-USABLE RELIC
C_MAX = 300         # ❌ NON-USABLE RELIC
MIN_COVERAGE = 0.6  # ❌ NON-USABLE RELIC

# --- ROI Filtering (ACTIVELY USED in run_detection()) ---
USE_ROI_REMOVE = True                                    # Default = True
ROI_REMOVE_RECT = (0, 300, 1920, 1080)                   # (0, 300, 1920, 1080) 

# =================================================================================================
# Step 5 Temporal Gates (ACTIVELY USED in apply_temporal_gates())
# =================================================================================================

USE_DISPLACEMENT_GATE = False          # Default = TRUE
USE_VARIANCE_GATE = False                  # Default = TRUE
USE_DIRECTIONAL_GATE = False              # Default = TRUE
USE_HYSTERESIS_GATE = False              # Default  = False

USE_LENGTH_GATE    = True        # almost always True
USE_DENSITY_GATE   = True       # turn OFF for now while we debug
USE_VARIANCE_GATE  = False       # turn OFF for now

# --- Displacement Gate Parameters ---
MIN_TRACK_LENGTH_FRAMES = 8            # Default = 8
MIN_DISP_PX = 5                        # Default = 5
MAX_DISP_PX = 9999.0                   # Default = 9999.0

# --- Variance Gate Parameters ---
MAX_HEADING_STD_DEG = 180.0           # Default = 180
MIN_LINEARITY_RATIO = 0.45            # Default = 0.45

# --- Directional Gate Parameters ---
ALLOWED_HEADING_BANDS = [(-45, 45), (-135, 135)]  # Default = [(-45, 45), (-135, 135)]

# =================================================================================================
# Step 6 Quality Scoring (ACTIVELY USED in score_track_quality())
# =================================================================================================

USE_TRACK_QUALITY_SCORING = True           # Default True
EXPECTED_HEADING_BAND = (-45.0, 45.0)      # Default (-45.0, 45.0)
EAGLE_Y_MIN = 0                            # Default = 0
EAGLE_Y_MAX = 400                          # Default = 400
MIN_LEN_FOR_SCORING = 8                     # Default = 8

# --- Quality Score Weights ---
TRACK_SCORE_W_LEN = 0.5                    # Default = 0.5
TRACK_SCORE_W_HEADING = 0.3                  # Default = 0.3
TRACK_SCORE_W_Y_BAND = 0.2                   # Default = 0.2

# =================================================================================================
# === OUTPUT ===
# =================================================================================================

import os

# Set the working directory to the folder you want
# os.chdir("C:/Axis_code_projects/OF_vs_Step7")

# Confirm
# print("Current working directory:", os.getcwd())


EXPORT_CSV = True
ENABLE_VIDEO_OUTPUT = True
DISPLAY_WINDOWS = False
VIDEO_OUTPUT_PATH = "step7_standard_tracking.mp4"
OUTPUT_FPS_OVERRIDE = None


# Project root for this notebook
PROJECT_ROOT = Path(r"C:\Axis_code_projects\OF_vs_Step7")
os.chdir(PROJECT_ROOT)
print("Current working directory:", os.getcwd())

# ---- Run identity ----
MODULE_NAME = "Step7_full"
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# VIDEO_PATH must be defined somewhere above this cell
video_stem = Path(VIDEO_PATH).stem if "VIDEO_PATH" in globals() else "unknown_clip"

RUN_NAME = f"step7_{video_stem}_{RUN_TS}"
RUN_DIR = PROJECT_ROOT / "outputs" / MODULE_NAME / RUN_NAME

LOG_DIR = RUN_DIR / "logs"
IMG_DIR = RUN_DIR / "images"
VID_DIR = RUN_DIR / "video"


# GS





for d in (LOG_DIR, IMG_DIR, VID_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

# ---- Your existing toggles, now routed to RUN_DIR ----
EXPORT_CSV = True
ENABLE_VIDEO_OUTPUT = True
DISPLAY_WINDOWS = False
OUTPUT_FPS_OVERRIDE = None

VIDEO_OUTPUT_PATH = str(VID_DIR / "step7_standard_tracking.mp4")

# Optional: write a lightweight run_config snapshot
run_config = {
    "module": MODULE_NAME,
    "run_name": RUN_NAME,
    "timestamp": RUN_TS,
    "video_path": str(VIDEO_PATH) if "VIDEO_PATH" in globals() else None,
    "export_csv": EXPORT_CSV,
    "enable_video_output": ENABLE_VIDEO_OUTPUT,
    "display_windows": DISPLAY_WINDOWS,
    "video_output_path": VIDEO_OUTPUT_PATH,
    "output_fps_override": OUTPUT_FPS_OVERRIDE,
}
with open(LOG_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)



# =================================================================================================
# DETECTION METADATA EXPORT SETTINGS 7.10
# =================================================================================================

EXPORT_DETECTION_METADATA = True   # Export confidence scores and bbox dimensions
INCLUDE_CONFIDENCE_SCORES = True   # Include detection confidence in output
INCLUDE_BBOX_DIMENSIONS = True     # Include bounding box width/height/aspect ratio



# =================================================================================================
# ✅ STEP 7: STANDARD TEMPORAL ASSOCIATION PARAMETERS
# =================================================================================================

print("\n" + "=" * 70)
print("STEP 7: STANDARD TEMPORAL ASSOCIATION CONFIGURATION")
print("=" * 70)

# --- Core Temporal Window Parameters ---
FRAME_WINDOW = 4                            # Default = 4
BUFFER = 2                               # Default = 1 { buffer > 2 fragments ^, misses the banking turn.
MAX_MISS = FRAME_WINDOW + BUFFER

# --- Motion Prediction Parameters ---
USE_MOTION_PREDICTION = True      # Default = True
VELOCITY_SMOOTH_WINDOW = 3        # Default = 3
SEARCH_RADIUS_BASE = 1.0          # Default = 1.0
SEARCH_RADIUS_GROWTH = 1.3        # Default = 1.3

# --- Association Strategy ---
USE_GREEDY_ASSOCIATION = True      # Default = True

# --- Quality Assessment Parameters ---
MIN_OBSERVATION_DENSITY = 0.5   # Default = 0.65

# =================================================================================================
# 🆕 STEP 7.10: Retro Best Track  DIAGNOSTIC PARAMETERS
# =================================================================================================



# --- Master Toggle ---
ENABLE_STITCHING_DIAGNOSTIC = True   # Defaut = True

# --- Stitching Algorithm ---
ENABLE_RETRO_INTERPOLATION = True  # Default = True

# --- Compatibility Thresholds ---
# S_T is the maximum allowed spatial gap (in pixels) between the predicted position of a track's endpoint and 
# the actual starting position of a candidate stitching track.
SPATIAL_THRESHOLD = 100    # Default = 100  
TEMPORAL_THRESHOLD = 15     # Default = 15
HEADING_TOLERANCE = 30      # Default = 30
Y_POSITION_TOLERANCE = 50    # Default = 50



# =============================================================================================
# STEP 7.15 D?
# =============================================================================================

TEP7_DIAG = dict(
    enabled=True,  # turn this off if you want to disable diagnostics

    # RAW pre-gates best track video (from all tracks before gates)
    make_raw_best_pre_gates_video=True,

    # Simulated gating (diagnostic only – does not change real results)
    simulate_min_track_length=None,        # e.g. 3 or 1 to test "too-short"
    simulate_disable_direction_gate=False, # True = direction/bearing off

    # Make a video for simulated best track (if any)
    make_simulated_best_video=True,
)


# =================================================================================================
# Print Configuration Summary
# =================================================================================================
print("\nSettings and Setup Only. Default Configuration Settings. No Real Ouput From This Cell.")

print(f"\n⏱️ Temporal Window:")
print(f"   FRAME_WINDOW: {FRAME_WINDOW} frames (core association window)")
print(f"   BUFFER: {BUFFER} frames (grace period, configurable 0-10)")
print(f"   MAX_MISS: {MAX_MISS} frames (total allowed consecutive misses)")
print(f"   → Track terminates if not observed for {MAX_MISS} consecutive frames")

print(f"\n🎯 Motion Prediction:")
print(f"   Model: Linear Constant Velocity")
print(f"   Velocity smoothing: {VELOCITY_SMOOTH_WINDOW} frames")
print(f"   Search radius scaling: {SEARCH_RADIUS_BASE}× base, {SEARCH_RADIUS_GROWTH}× per miss")
print(f"   → Upgrade to Kalman if: curved paths, long gaps, uncertainty needed")

print(f"\n🔗 Association Strategy:")
print(f"   Algorithm: {'Greedy Nearest Neighbor' if USE_GREEDY_ASSOCIATION else 'Hungarian (optimal)'}")
print(f"   → Upgrade to Hungarian if: crossing tracks, >5 objects/frame")

print(f"\n📊 Quality Criteria:")
print(f"   Min observation density: {MIN_OBSERVATION_DENSITY:.0%}")
print(f"   Min track length: {MIN_TRACK_LENGTH_FRAMES} frames")

print(f"\n🔗 Stitching Diagnostic:")
print(f"   Enabled: {ENABLE_STITCHING_DIAGNOSTIC}")
print(f"   Spatial threshold: {SPATIAL_THRESHOLD}px")
print(f"   Temporal threshold: {TEMPORAL_THRESHOLD} frames")
print(f"   Heading tolerance: {HEADING_TOLERANCE}°")

print(f"\n⚠️  Unused Parameters (Legacy Dead Code - Safe to Delete):")
print(f"   TAU={TAU} (Step 4 adaptive threshold - no functional logic)")
print(f"   L1={L1}, L2={L2} (Step 4 composite scoring - no functional logic)")
print(f"   C_MAX={C_MAX}, MIN_COVERAGE={MIN_COVERAGE} (Step 4 filtering - no functional logic)")

print(f"\n⚠️  Usable But Not Currently Used:")
print(f"   diff_threshold_values, min_area_values, centroid_thresholds (parameter sweep arrays)")
print(f"   USE_HYSTERESIS_GATE={USE_HYSTERESIS_GATE} (implemented but disabled)")
print(f"   MAX_DISP_PX={MAX_DISP_PX}, MAX_HEADING_STD_DEG={MAX_HEADING_STD_DEG} (effectively disabled)")
print(f"   ENABLE_VIDEO_OUTPUT, DISPLAY_WINDOWS, VIDEO_OUTPUT_PATH (not implemented in Step 7)")

print("\n" + "=" * 70)

# Validation
assert 0 <= BUFFER <= 10, "BUFFER must be in range [0, 10]"
assert FRAME_WINDOW > 0, "FRAME_WINDOW must be positive"
assert 0.0 < MIN_OBSERVATION_DENSITY <= 1.0, "MIN_OBSERVATION_DENSITY must be in (0, 1]"

print("✓ Configuration validated")

# =================================================================================================
# 🆕 DEFAULT CONFIGURATION DICTIONARY (FOR CHANGE TRACKING)
# =================================================================================================

DEFAULT_CONFIG = {
    # --- Frame Preprocessing ---
    "BLUR_KERNEL": (5, 5),
    "USE_MORPH_OPEN": True,
    "USE_MORPH_CLOSE": True,
    "MORPH_KERNEL": None,
    
    # --- Detection Threshold Sweeps ---
    "diff_threshold_values": [2.0, 2.30],
    "min_area_values": [51, 65],
    "centroid_thresholds": [35, 50],


    
    # --- ROI Filtering ---
    "USE_ROI_REMOVE": True,
    "ROI_REMOVE_RECT": (0, 300, 1920, 1080),
    
    # --- Temporal Gates ---
    "USE_DISPLACEMENT_GATE": True,
    "USE_VARIANCE_GATE": True,
    "USE_DIRECTIONAL_GATE": True,
    "USE_HYSTERESIS_GATE": False,
    
    # --- Displacement Gate Parameters ---
    "MIN_TRACK_LENGTH_FRAMES": 8,
    "MIN_DISP_PX": 5,
    "MAX_DISP_PX": 9999.0,
    
    # --- Variance Gate Parameters ---
    "MAX_HEADING_STD_DEG": 180.0,
    "MIN_LINEARITY_RATIO": 0.45,
    
    # --- Directional Gate Parameters ---
    "ALLOWED_HEADING_BANDS": [(-45, 45)],
    
    # --- Quality Scoring ---
    "USE_TRACK_QUALITY_SCORING": True,
    "EXPECTED_HEADING_BAND": (-45.0, 45.0),
    "EAGLE_Y_MIN": 0,
    "EAGLE_Y_MAX": 400,
    "MIN_LEN_FOR_SCORING": 8,
    
    # --- Quality Score Weights ---
    "TRACK_SCORE_W_LEN": 0.5,
    "TRACK_SCORE_W_HEADING": 0.3,
    "TRACK_SCORE_W_Y_BAND": 0.2,
    
    # --- Output ---
    "EXPORT_CSV": True,
    "ENABLE_VIDEO_OUTPUT": True,
    "DISPLAY_WINDOWS": False,
    "VIDEO_OUTPUT_PATH": "step7_standard_tracking.mp4",
    "OUTPUT_FPS_OVERRIDE": None,
    
    # --- Temporal Window ---
    "FRAME_WINDOW": 4,
    "BUFFER": 1,  # ← DEFAULT is 1, not 5
    
    # --- Motion Prediction ---
    "USE_MOTION_PREDICTION": True,
    "VELOCITY_SMOOTH_WINDOW": 3,
    "SEARCH_RADIUS_BASE": 1.0,
    "SEARCH_RADIUS_GROWTH": 1.3,
    
    # --- Association Strategy ---
    "USE_GREEDY_ASSOCIATION": True,
    
    # --- Quality Assessment ---
    "MIN_OBSERVATION_DENSITY": 0.65,
    
    # --- 🆕 Stitching Diagnostic (Cell 7.10) ---
    "ENABLE_STITCHING_DIAGNOSTIC": True,
    "ENABLE_RETRO_INTERPOLATION": True,
    "SPATIAL_THRESHOLD": 100,
    "TEMPORAL_THRESHOLD": 15,
    "HEADING_TOLERANCE": 30,
    "Y_POSITION_TOLERANCE": 50,
}


print("✓ Default configuration dictionary created (for change tracking)")

# =================================================================================================
# 🆕 CONFIGURATION CHANGE DETECTION FUNCTION
# =================================================================================================

def detect_config_changes() -> Dict[str, Any]:
    """
    Detect which configuration settings have been changed from defaults.
    
    Compares current settings (from Cell 7.2 globals) against DEFAULT_CONFIG dictionary.
    
    Returns:
        Dict with keys:
          - "has_changes": bool (True if any settings changed)
          - "changes": List[Dict] with details of each change
          - "summary": str (formatted summary for printing)
    
    Example Output:
    ---------------
    {
        "has_changes": True,
        "changes": [
            {"setting": "BUFFER", "default": 1, "current": 5},
            {"setting": "USE_ROI_REMOVE", "default": False, "current": True}
        ],
        "summary": "This Run: One or more default settings changed:\n   • BUFFER: default=1, current=5\n..."
    }
    """
    
    changes = []
    
    # Get current config values from globals
    current_config = {
        # --- Frame Preprocessing ---
        "BLUR_KERNEL": BLUR_KERNEL,
        "USE_MORPH_OPEN": USE_MORPH_OPEN,
        "USE_MORPH_CLOSE": USE_MORPH_CLOSE,
        "MORPH_KERNEL": MORPH_KERNEL,
        
        # --- Detection Threshold Sweeps ---
        "diff_threshold_values": diff_threshold_values,
        "min_area_values": min_area_values,
        "centroid_thresholds": centroid_thresholds,
        
        # --- ROI Filtering ---
        "USE_ROI_REMOVE": USE_ROI_REMOVE,
        "ROI_REMOVE_RECT": ROI_REMOVE_RECT,
        
        # --- Temporal Gates ---
        "USE_DISPLACEMENT_GATE": USE_DISPLACEMENT_GATE,
        "USE_VARIANCE_GATE": USE_VARIANCE_GATE,
        "USE_DIRECTIONAL_GATE": USE_DIRECTIONAL_GATE,
        "USE_HYSTERESIS_GATE": USE_HYSTERESIS_GATE,
        
        # --- Displacement Gate Parameters ---
        "MIN_TRACK_LENGTH_FRAMES": MIN_TRACK_LENGTH_FRAMES,
        "MIN_DISP_PX": MIN_DISP_PX,
        "MAX_DISP_PX": MAX_DISP_PX,
        
        # --- Variance Gate Parameters ---
        "MAX_HEADING_STD_DEG": MAX_HEADING_STD_DEG,
        "MIN_LINEARITY_RATIO": MIN_LINEARITY_RATIO,
        
        # --- Directional Gate Parameters ---
        "ALLOWED_HEADING_BANDS": ALLOWED_HEADING_BANDS,
        
        # --- Quality Scoring ---
        "USE_TRACK_QUALITY_SCORING": USE_TRACK_QUALITY_SCORING,
        "EXPECTED_HEADING_BAND": EXPECTED_HEADING_BAND,
        "EAGLE_Y_MIN": EAGLE_Y_MIN,
        "EAGLE_Y_MAX": EAGLE_Y_MAX,
        "MIN_LEN_FOR_SCORING": MIN_LEN_FOR_SCORING,
        
        # --- Quality Score Weights ---
        "TRACK_SCORE_W_LEN": TRACK_SCORE_W_LEN,
        "TRACK_SCORE_W_HEADING": TRACK_SCORE_W_HEADING,
        "TRACK_SCORE_W_Y_BAND": TRACK_SCORE_W_Y_BAND,
        
        # --- Output ---
        "EXPORT_CSV": EXPORT_CSV,
        "ENABLE_VIDEO_OUTPUT": ENABLE_VIDEO_OUTPUT,
        "DISPLAY_WINDOWS": DISPLAY_WINDOWS,
        "VIDEO_OUTPUT_PATH": VIDEO_OUTPUT_PATH,
        "OUTPUT_FPS_OVERRIDE": OUTPUT_FPS_OVERRIDE,
        
        # --- Temporal Window ---
        "FRAME_WINDOW": FRAME_WINDOW,
        "BUFFER": BUFFER,
        
        # --- Motion Prediction ---
        "USE_MOTION_PREDICTION": USE_MOTION_PREDICTION,
        "VELOCITY_SMOOTH_WINDOW": VELOCITY_SMOOTH_WINDOW,
        "SEARCH_RADIUS_BASE": SEARCH_RADIUS_BASE,
        "SEARCH_RADIUS_GROWTH": SEARCH_RADIUS_GROWTH,
        
        # --- Association Strategy ---
        "USE_GREEDY_ASSOCIATION": USE_GREEDY_ASSOCIATION,
        
        # --- Quality Assessment ---
        "MIN_OBSERVATION_DENSITY": MIN_OBSERVATION_DENSITY,
        
        # --- Stitching Diagnostic (Cell 7.10) ---
        "ENABLE_STITCHING_DIAGNOSTIC": ENABLE_STITCHING_DIAGNOSTIC,
        "ENABLE_RETRO_INTERPOLATION": ENABLE_RETRO_INTERPOLATION,
        "SPATIAL_THRESHOLD": SPATIAL_THRESHOLD,
        "TEMPORAL_THRESHOLD": TEMPORAL_THRESHOLD,
        "HEADING_TOLERANCE": HEADING_TOLERANCE,
        "Y_POSITION_TOLERANCE": Y_POSITION_TOLERANCE,
    }
    
    # Compare current vs default
    for setting_name, default_value in DEFAULT_CONFIG.items():
        current_value = current_config.get(setting_name)
        
        # Check if value changed
        if current_value != default_value:
            changes.append({
                "setting": setting_name,
                "default": default_value,
                "current": current_value
            })
    
    # Build summary string
    if not changes:
        summary = "This Run: All default settings used"
    else:
        summary_lines = ["This Run: One or more default settings changed:"]
        
        for change in changes:
            setting = change["setting"]
            default = change["default"]
            current = change["current"]
            
            # Format value display (handle complex types)
            if isinstance(default, tuple) and setting == "ROI_REMOVE_RECT":
                # ROI rect is too verbose, just say "modified"
                summary_lines.append(f"   • {setting}: modified")
            elif isinstance(default, list) and len(str(default)) > 50:
                # Long lists, show abbreviated
                summary_lines.append(f"   • {setting}: modified")
            else:
                # Show full values
                summary_lines.append(f"   • {setting}: default={default}, current={current}")
        
        summary = "\n".join(summary_lines)
    
    return {
        "has_changes": len(changes) > 0,
        "changes": changes,
        "summary": summary,
    }

    

print("✓ Configuration change detection function defined")


# Test print the detect_config_changes() 
# Check if function exists, define inline if needed
if 'detect_config_changes' not in dir():
    def detect_config_changes() -> Dict[str, Any]:
        """Fallback inline definition"""
        # ... (same code as above)
        pass

# ==========================================================
# Works.  This outputs the non-default - aka user modified settings.  S/be aligned w/ log entry.
config_changes = detect_config_changes()

print("\n\n" + "=" * 70)
print("⏱️ CONFIGURATION Change SUMMARY")
print("=" * 70)

config_changes = detect_config_changes()
print(f"\n{config_changes['summary']}")

Current working directory: C:\Axis_code_projects\OF_vs_Step7
RUN_DIR: C:\Axis_code_projects\OF_vs_Step7\outputs\Step7_full\step7_big_bird_R2L_20260101_183730

STEP 7: STANDARD TEMPORAL ASSOCIATION CONFIGURATION

Settings and Setup Only. Default Configuration Settings. No Real Ouput From This Cell.

⏱️ Temporal Window:
   FRAME_WINDOW: 4 frames (core association window)
   BUFFER: 2 frames (grace period, configurable 0-10)
   MAX_MISS: 6 frames (total allowed consecutive misses)
   → Track terminates if not observed for 6 consecutive frames

🎯 Motion Prediction:
   Model: Linear Constant Velocity
   Velocity smoothing: 3 frames
   Search radius scaling: 1.0× base, 1.3× per miss
   → Upgrade to Kalman if: curved paths, long gaps, uncertainty needed

🔗 Association Strategy:
   Algorithm: Greedy Nearest Neighbor
   → Upgrade to Hungarian if: crossing tracks, >5 objects/frame

📊 Quality Criteria:
   Min observation density: 50%
   Min track length: 8 frames

🔗 Stitching Diagnostic:
   Enabl

## <font color = yellow> 7.2B: export function (adapter)

In [9]:
# Use in the new comparison folder: C:\Axis_code_projects\OF_vs_Step7\Code



def export_comparison_packet_step7(
    *,
    run_dir: Path,
    video_path: str,
    fps: float,
    total_frames: int,
    frame_w: int,
    frame_h: int,
    primary_span: Tuple[int, int],
    md_frames: Set[int],
    active_pixels_by_frame: Optional[Dict[int, float]] = None,
    x_by_frame: Optional[Dict[int, float]] = None,
    y_by_frame: Optional[Dict[int, float]] = None,
    extra_manifest: Optional[Dict] = None,
) -> None:

    """
    Standard comparison packet for Step7:
      packet/run_manifest.json
      packet/frame_signal.csv
      packet/segment_summary.csv
    """
    # GS
    run_dir = Path(run_dir)
   #  packet_dir = run_dir / "packet"
     # CHANGE THIS LINE:
    packet_dir = run_dir / packet_name   # <-- was run_dir / "packet"
    packet_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # 1) run_manifest.json
    # -------------------------
    manifest = {
        "pipeline_name": "Step7_full",
        "pipeline_version": "notebook",
        "video_path": str(video_path),
        "video_name": Path(video_path).name,
        "clip_info": {"fps": float(fps), "total_frames": int(total_frames), "w": int(frame_w), "h": int(frame_h)},
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    if extra_manifest:
        manifest.update(extra_manifest)
    with (packet_dir / "run_manifest.json").open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    s, e = primary_span

    # -------------------------
    # 2) frame_signal.csv
    # -------------------------
    rows = []
    for fidx in range(s, e+1):
        md_flag = 1 if fidx in md_frames else 0
        active = ""
        if active_pixels_by_frame is not None:
            active = active_pixels_by_frame.get(fidx, "")
        x = ""
        y = ""
        if x_by_frame is not None: x = x_by_frame.get(fidx, "")
        if y_by_frame is not None: y = y_by_frame.get(fidx, "")

        rows.append([fidx, md_flag, 1, active, "", "", x, y, "Step7"])

    with (packet_dir / "frame_signal.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([
            "frame_idx",
            "md_flag",
            "in_primary_span",
            "active_pixels",
            "energy_raw",
            "energy_norm",
            "x",
            "y",
            "source",
        ])
        w.writerows(rows)

    # -------------------------
    # 3) segment_summary.csv
    # -------------------------
    span_len = int(e - s + 1)
    md_total = int(sum(1 for fidx in range(s, e+1) if fidx in md_frames))
    density = float(md_total / span_len) if span_len else 0.0

    # gap stats
    md_flags = [1 if fidx in md_frames else 0 for fidx in range(s, e+1)]
    gap_count = 0
    max_gap = 0
    cur = 0
    for v in md_flags:
        if v == 0:
            cur += 1
        else:
            if cur > 0:
                gap_count += 1
                max_gap = max(max_gap, cur)
                cur = 0
    if cur > 0:
        gap_count += 1
        max_gap = max(max_gap, cur)

    with (packet_dir / "segment_summary.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["segment_id","span_start","span_end","span_len","md_total","density","gap_count","max_gap","is_primary"])
        w.writerow([1, s, e, span_len, md_total, density, gap_count, max_gap, 1])

    print(f"📦 Step7 packet export → {packet_dir}")


### <font color = lime> Create the Silent Logger / me_note (for reference)

In [10]:
cfg_text = "\n".join(f"  {k}: {v}" for k, v in config_changes.items())
# log.silent(f"\n=== Default Config Changes ===\n{cfg_text}\n======================", note = "Config Changes logged")
log.silent(f"\n=== Default Config Changes ===\n{cfg_text}\n==========GS============", note = "Config Changes logged")

# create reusable template
me_note = ' log.silent(f"\\n=== Default Config Changes ===\\n{cfg_text}\\n==========GS============",  note=" template ") '

# test print me_note
me_note

Config Changes logged


' log.silent(f"\\n=== Default Config Changes ===\\n{cfg_text}\\n==========GS============",  note=" template ") '

In [11]:
# test settings


print("DIFF_THRESHOLD" in globals())
print("MIN_AREA" in globals())
print("CENTROID_THRESHOLD" in globals())

try:
    print("DIFF =", DIFF_THRESHOLD)
except:
    print("DIFF undefined")

try:
    print("AREA =", MIN_AREA)
except:
    print("MIN_AREA undefined")

try:
    print("CENTROID =", CENTROID_THRESHOLD)
except:
    print("CENTROID_THRESHOLD undefined")


True
True
True
DIFF = 2
AREA = 51
CENTROID = 35


## <font color = yellow> 7.3: Create Track - defines the blueprint (Track class)

#### <font color = lime> Track is designed to represent the current state of a track (not keep a log history)


        Define Track class / data structure (fields like track_id, observations, gap counters, status). No pipeline work here; purely the object blueprint.



##### Note: w/a class using @dataclass, Python automatically generates an __init__ constructor. A normal class does not include the constructor — it’s just the container that holds methods and attributes (vars)


In [12]:
# =================================================================================================
# ✅ STEP 7.3: TRACK DATACLASS & EXPORT HELPER
# =================================================================================================

from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional, Any, Set

@dataclass
class Track:
    """
    Core temporal track representation for Step 7.

    Design:
    -------
    - Only *raw* per-frame data is stored directly:
        • observations: {frame_idx -> (cx, cy)}
        • bboxes:       {frame_idx -> (x, y, w, h)}
        • gap_frames:   {frame_idx, ...} frames where the track was alive but no match

    - Everything else (span, density, gap sequences, etc.) is computed
      on-demand from these raw structures.

    - Later steps (7.7, 7.8, 7.10) hang extra stats on these objects but
      do not redefine the core membership.
    """

    track_id: int

    # Temporal anchors
    created_frame: int
    last_observed_frame: int
    last_updated_frame: int
    miss_count: int = 0

    status: str = "candidate"   # "candidate", "confirmed", "rejected", "terminated", etc.

    # --- Raw per-frame data ---
    observations: Dict[int, Tuple[float, float]] = field(default_factory=dict)
    bboxes: Dict[int, Tuple[int, int, int, int]] = field(default_factory=dict)

    # Frames where the track was evaluated but no detection matched
    gap_frames: Set[int] = field(default_factory=set)

    # --- Motion stats (Step 7.7) ---
    step_displacements: List[float] = field(default_factory=list)
    step_headings: List[float] = field(default_factory=list)

    disp_min: Optional[float] = None
    disp_max: Optional[float] = None
    disp_mean: Optional[float] = None
    disp_median: Optional[float] = None
    heading_mean: Optional[float] = None
    heading_std: Optional[float] = None
    linearity_ratio: Optional[float] = None

    # --- Scoring / vertical stats (Step 7.8) ---
    mean_heading_deg: Optional[float] = None
    std_heading_deg: Optional[float] = None

    mean_y: Optional[float] = None
    min_y: Optional[float] = None
    max_y: Optional[float] = None
    y_band_fraction: float = 0.0

    quality_score: float = 0.0
    confidence_score: float = 0.0

    # --- Interpolation metadata (Step 7.10) ---
    interpolated_frames: Set[int] = field(default_factory=set)   # frames added synthetically
    original_observation_density: Optional[float] = None         # BEFORE interpolation
    was_interpolated: bool = False

    # ------------------------------------------------------------------
    # Derived properties
    # ------------------------------------------------------------------
    @property
    def frames(self) -> List[int]:
        """Sorted list of frames with actual observations."""
        return sorted(self.observations.keys())

    @property
    def length(self) -> int:
        """Number of observed frames."""
        return len(self.observations)

    @property
    def span(self) -> int:
        """Total span in frames from first to last observation (includes gaps)."""
        f = self.frames
        return 0 if not f else (f[-1] - f[0] + 1)

    @property
    def observation_density(self) -> float:
        """Observed frames / total span."""
        s = self.span
        return self.length / s if s > 0 else 0.0


    @property
    def gap_frames_sorted(self) -> List[int]:
        """
        Derived list of gap frame indices based on observations.

        A "gap" is any frame between the first and last observed frame
        that does NOT have an observation.
        """
        if not self.observations:
            return []

        frames = self.frames
        first_f = frames[0]
        last_f = frames[-1]

        all_frames = set(range(first_f, last_f + 1))
        observed = set(frames)

        gap_frames = sorted(all_frames - observed)
        return gap_frames

    @property
    def gap_count(self) -> int:
        """Number of missing frames within track span (derived)."""
        return len(self.gap_frames_sorted)

    @property
    def gap_sequences(self) -> List[Tuple[int, int, int]]:
        """
        Identify contiguous gap sequences.

        Returns:
            List of (start_frame, end_frame, gap_length)
        """
        gaps = self.gap_frames_sorted
        if not gaps:
            return []

        sequences = []
        start = gaps[0]
        prev = gaps[0]

        for f in gaps[1:]:
            if f == prev + 1:
                prev = f
            else:
                sequences.append((start, prev, prev - start + 1))
                start = f
                prev = f

        # add final sequence
        sequences.append((start, prev, prev - start + 1))
        return sequences

    @property
    def longest_gap(self) -> int:
        """Length of longest contiguous gap."""
        seqs = self.gap_sequences
        if not seqs:
            return 0
        return max(length for (_, _, length) in seqs)




    
    @property
    def longest_gap(self) -> int:
        """Length of the longest contiguous gap sequence."""
        if not self.gap_sequences:
            return 0
        return max(length for (_, _, length) in self.gap_sequences)

    # ------------------------------------------------------------------
    # Helpers used by 7.4 / 7.7 / 7.8 / 7.10
    # ------------------------------------------------------------------
    def add_observation(
        self,
        frame: int,
        cx: float,
        cy: float,
        bbox: Optional[Tuple[int, int, int, int]] = None,
    ) -> None:
        """Add an observed centroid (and optional bbox) for a frame."""
        self.observations[frame] = (cx, cy)
        if bbox is not None:
            self.bboxes[frame] = bbox
        self.last_observed_frame = frame
        self.last_updated_frame = frame
        self.miss_count = 0  # reset miss counter on hit

        # If this frame was previously marked as a gap, clear it
        if frame in self.gap_frames:
            self.gap_frames.discard(frame)

    def add_gap(self, frame: int) -> None:
        """
        Record a gap frame (called when track is alive but no detection matched).
        """
        self.gap_frames.add(frame)
        self.last_updated_frame = frame

    def is_observed(self, frame: int) -> bool:
        """True if we have an actual centroid for this frame."""
        return frame in self.observations

    def get_centroids_list(self) -> List[Tuple[float, float]]:
        """Centroids in temporal order (used for headings, vertical stats, etc.)."""
        return [self.observations[f] for f in self.frames]


def track_to_dict(track: Track) -> Dict[str, Any]:
    """
    Convert a Track into a flat dict suitable for logging/export (CSV/JSON).
    """
    return {
        "track_id": track.track_id,
        "status": track.status,
        "created_frame": track.created_frame,
        "last_observed_frame": track.last_observed_frame,
        "last_updated_frame": track.last_updated_frame,
        "length": track.length,
        "span": track.span,
        "observation_density": track.observation_density,
        "observed_frames": track.frames,
        "gap_frames": track.gap_frames_sorted,
        "gap_sequences": track.gap_sequences,
        "longest_gap": track.longest_gap,
        "gap_count": track.gap_count,
        "miss_count": track.miss_count,
        "quality_score": track.quality_score,
        "confidence_score": track.confidence_score,
    }


print("\n✅ Step 7.3: Track dataclass + export helper defined.")
print("   - No velocity state, no Hungarian-specific fields.")
print("   - gap_frames is the single source of truth for gaps.")



✅ Step 7.3: Track dataclass + export helper defined.
   - No velocity state, no Hungarian-specific fields.
   - gap_frames is the single source of truth for gaps.


### <font color = lime> Review the dataclass object (FMI)

In [13]:
# # From Track: prints every attribute name, its type annotation, and default value.
# from dataclasses import fields

# for f in fields(Track):
#     print(f.name, f.type, f.default)



## <font color = yellow> 7.4: Track Building / Defines the logic functions (prediction, greedy association).

#### <font color = lime> Gap Tolerance - Centroid Predictions

    Define all functions that take per-frame detections and build tracks: prediction, greedy association, updating state, handling gaps, finalizing tracks. This is the algorithm brain, but still function definitions only.

In [14]:
# =================================================================================================
# ✅ STEP 7.4: TRACK BUILDING (STANDARD TEMPORAL ASSOCIATION, GREEDY ONLY)
# =================================================================================================

from typing import List, Dict, Any, Tuple
import math

def predict_position(track: Track, target_frame: int) -> Tuple[float, float]:
    """
    Stateless, simple prediction for an active track.

    Strategy:
    ---------
    - If there are at least 2 observations and target_frame > last_observed_frame:
        • Use a 2-point linear extrapolation.
    - Otherwise:
        • Use the last observed position.

    No persistent velocity state is stored on the Track; everything is derived
    from its observations.
    """
    frames = track.frames
    if not frames:
        return (0.0, 0.0)

    last_f = frames[-1]
    last_pos = track.observations[last_f]

    if len(frames) >= 2 and target_frame > last_f:
        prev_f = frames[-2]
        prev_pos = track.observations[prev_f]
        dt = last_f - prev_f
        if dt > 0:
            vx = (last_pos[0] - prev_pos[0]) / dt
            vy = (last_pos[1] - prev_pos[1]) / dt
            dt_future = target_frame - last_f
            return (
                last_pos[0] + vx * dt_future,
                last_pos[1] + vy * dt_future,
            )

    # Fallback: nearest neighbor in time (last seen)
    return last_pos


def compute_adaptive_search_radius(
    base_distance_threshold: float,
    miss_count: int,
) -> float:
    """
    Very simple adaptive radius:
      - base radius for fresh tracks
      - grows linearly with miss_count

    You can tune the 0.5 factor if needed.
    """
    return base_distance_threshold * (1.0 + 0.5 * miss_count)


def greedy_association(
    tracks: List[Track],
    centroids: List[Tuple[float, float]],
    current_frame: int,
    base_distance_threshold: float,
) -> Tuple[List[Tuple[int, int]], List[int]]:
    """
    Greedy nearest-neighbor association between active tracks and detections.

    Returns:
        assignments: [(track_index, detection_index), ...]
        unassigned_detections: [detection_index, ...]
    """
    if not tracks or not centroids:
        return [], list(range(len(centroids)))

    unused_dets = set(range(len(centroids)))
    assignments: List[Tuple[int, int]] = []

    for ti, track in enumerate(tracks):
        if not unused_dets:
            break

        predicted = predict_position(track, current_frame)
        radius = compute_adaptive_search_radius(
            base_distance_threshold, track.miss_count
        )

        best_idx = None
        best_dist = None

        for di in unused_dets:
            cx, cy = centroids[di]
            d = math.hypot(cx - predicted[0], cy - predicted[1])
            if best_dist is None or d < best_dist:
                best_dist = d
                best_idx = di

        if best_idx is not None and best_dist is not None and best_dist <= radius:
            assignments.append((ti, best_idx))
            unused_dets.remove(best_idx)

    return assignments, sorted(unused_dets)


def build_tracks_standard(
    detections: List[Dict[str, Any]],
    max_link_distance: float,
) -> List[Track]:
    """
    Build tracks using a standard temporal window + greedy association.

    Inputs:
        detections: list of detection dicts with keys:
            - frame_index
            - centroids: [(cx, cy), ...]
            - bboxes:    [(x, y, w, h), ...]

        max_link_distance: base centroid distance threshold for linking.

    Uses:
        - MAX_MISS from Step 7.2 to decide when to terminate tracks
        - Track.gap_frames via track.add_gap(current_frame) when no match
    """
    if not detections:
        print("   ⚠ No detections passed into build_tracks_standard.")
        return []

    active_tracks: List[Track] = []
    finished_tracks: List[Track] = []
    next_track_id = 0

    print("\n🔄 Building tracks with temporal window (MAX_MISS={})...".format(MAX_MISS))

    for det in detections:
        frame_idx = det["frame_index"]
        centroids = det.get("centroids", [])
        bboxes = det.get("bboxes", [])

        # ------------------------------------------------------------------
        # 1) Associate existing tracks to new detections
        # ------------------------------------------------------------------
        if active_tracks and centroids:
            assignments, unassigned_dets = greedy_association(
                active_tracks,
                centroids,
                frame_idx,
                max_link_distance,
            )
        else:
            assignments = []
            unassigned_dets = list(range(len(centroids)))

        # Tracks that got at least one detection this frame
        assigned_track_indices = {ti for (ti, _) in assignments}

        # Update matched tracks
        for ti, di in assignments:
            track = active_tracks[ti]
            cx, cy = centroids[di]
            bbox = bboxes[di] if di < len(bboxes) else None
            track.add_observation(frame_idx, cx, cy, bbox)
            # status upgrade is handled later by gating/scoring; we just keep them alive

        # Handle unmatched tracks (possible gaps)
        new_active_tracks: List[Track] = []

        for ti, track in enumerate(active_tracks):
            if ti in assigned_track_indices:
                # Already updated above
                new_active_tracks.append(track)
                continue

            # No detection matched this frame → mark gap
            track.miss_count += 1
            track.add_gap(frame_idx)

            if track.miss_count > MAX_MISS:
                track.status = "terminated"
                finished_tracks.append(track)
            else:
                new_active_tracks.append(track)

        active_tracks = new_active_tracks

        # ------------------------------------------------------------------
        # 2) Spawn new tracks for unassigned detections
        # ------------------------------------------------------------------
        for di in unassigned_dets:
            cx, cy = centroids[di]
            bbox = bboxes[di] if di < len(bboxes) else None
            t = Track(
                track_id=next_track_id,
                created_frame=frame_idx,
                last_observed_frame=frame_idx,
                last_updated_frame=frame_idx,
            )
            t.add_observation(frame_idx, cx, cy, bbox)
            active_tracks.append(t)
            next_track_id += 1

    # Move any remaining active tracks to finished
    finished_tracks.extend(active_tracks)

    print("\n================================================================================")
    print("📊 TRACK BUILDING COMPLETED")
    print("================================================================================")
    print(f"   ✓ Built {len(finished_tracks)} tracks total")
    return finished_tracks


print("\n" + "=" * 70)
print("STEP 7.4: TRACK BUILDING (STANDARD TEMPORAL ASSOCIATION, GREEDY ONLY)")
print("=" * 70)

print("✓ Functions defined:")
print("   - predict_position(track, target_frame)")
print("   - compute_adaptive_search_radius(base_distance_threshold, miss_count)")
print("   - greedy_association(tracks, centroids, current_frame, base_distance_threshold)")
print("   - build_tracks_standard(detections, max_link_distance)")

# Optional: tiny sanity hint if detections already exist in the notebook
if "detections" in globals():
    print("\n🔎 Hint: 'detections' is already defined in this notebook.")
    print("   You can do a quick dry run like:")
    print("   tracks_test = build_tracks_standard(detections, max_link_distance=CENTROID_THRESHOLD)")
    print("   len(tracks_test)  # to inspect track count")
else:
    print("\nℹ No 'detections' variable in scope yet.")
    print("   Tracks will be built later by Step 7.9 (run_step7_pipeline).")




STEP 7.4: TRACK BUILDING (STANDARD TEMPORAL ASSOCIATION, GREEDY ONLY)
✓ Functions defined:
   - predict_position(track, target_frame)
   - compute_adaptive_search_radius(base_distance_threshold, miss_count)
   - greedy_association(tracks, centroids, current_frame, base_distance_threshold)
   - build_tracks_standard(detections, max_link_distance)

ℹ No 'detections' variable in scope yet.
   Tracks will be built later by Step 7.9 (run_step7_pipeline).


## <font color = yellow> 7.5a: Diagnostic toggles

In [15]:
# ======================================================================
# STEP 7.D CONFIG – DIAGNOSTICS
# ======================================================================

from typing import Dict, Any

# Shared artifact bucket for 7.6, 7.7, 7.9, 7.13, 7.D
STEP7_ARTIFACTS: Dict[str, Any] = {}

# Diagnostic options (TURN ON/OFF HERE)
STEP7_DIAG: Dict[str, Any] = dict(
    enabled=True,   # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< TURN THIS ON

    # Raw pre-gates diagnostic video
    make_raw_best_pre_gates_video=True,

    # Simulated gating (optional, keep off for now)
    simulate_min_track_length=None,
    simulate_disable_direction_gate=False,
    make_simulated_best_video=False,
)

print("\n[Step 7.D CONFIG] Diagnostics ENABLED =", STEP7_DIAG["enabled"])
print("[Step 7.D CONFIG] STEP7_DIAG + STEP7_ARTIFACTS defined successfully.\n")



[Step 7.D CONFIG] Diagnostics ENABLED = True
[Step 7.D CONFIG] STEP7_DIAG + STEP7_ARTIFACTS defined successfully.



In [16]:
# ======================================================================
# 7.D – Diagnostic best-track lens (function definition)
# ======================================================================

from typing import Dict, Any
from pathlib import Path

def run_step7_diag(
    artifacts: Dict[str, Any],
    diag_config: Dict[str, Any],
    video_path: str,
    output_dir: Path,
    logger,
    config: Dict[str, Any],
) -> None:
    """
    Minimal diagnostic lens:
      - Checks that artifacts are present.
      - Picks a 'raw best' pre-gates track from gap_stats.
      - Writes a debug video for that track with label 'raw_pre_gates'.
    """
    print("\n" + "=" * 70)
    print("STEP 7.D – DIAGNOSTIC BEST-TRACK LENS")
    print("=" * 70)

    all_tracks = artifacts.get("all_tracks")
    gap_stats = artifacts.get("gap_stats")

    print(f"[Step 7.D] all_tracks present:    {all_tracks is not None}")
    print(f"[Step 7.D] gap_stats present:    {gap_stats is not None}")

    # If we don't have tracks or gap_stats, bail with a clear message
    if not all_tracks or not gap_stats:
        print("[Step 7.D] Missing tracks or gap_stats – nothing to do.")
        return

    # ------------------------------------------------------------------
    # RAW PRE-GATES BEST TRACK VIDEO
    # ------------------------------------------------------------------
    if diag_config.get("make_raw_best_pre_gates_video", False):
        scored = []
        for row in gap_stats:
            try:
                tid = int(row["track_id"])
                length = float(row["length"])
                density = float(row.get("density", 0.0) or 0.0)
                longest_gap = float(row["longest_gap"])
            except Exception:
                continue

            # Simple heuristic:
            #   score = length + 50*density - 0.1*longest_gap
            score = length + 50.0 * density - 0.1 * longest_gap
            scored.append((score, tid))

        if not scored:
            print("[Step 7.D] No valid entries in gap_stats for RAW best.")
        else:
            scored.sort(reverse=True, key=lambda x: x[0])
            best_score, raw_best_id = scored[0]
            print(f"[Step 7.D] RAW pre-gates best track → ID={raw_best_id}, score={best_score:.3f}")

            track_obj = all_tracks.get(raw_best_id)
            if track_obj is None:
                print("[Step 7.D] RAW best track not found in all_tracks dict.")
            else:
                try:
                    raw_label = "raw_pre_gates"
                    raw_video_path = build_best_track_video(
                        video_path=video_path,
                        track=track_obj,
                        track_id=raw_best_id,
                        output_dir=output_dir,
                        label=raw_label,
                        logger=logger,
                        config=config,
                    )
                    print(f"✓ STEP 7.D: Wrote RAW pre-gates best-track video → {raw_video_path}")
                except Exception as e:
                    print(f"⚠ STEP 7.D: Failed to write RAW pre-gates video: {e}")

    print("\n[Step 7.D] Diagnostics complete.")


## <font color = yellow> 7.6:A Gap Metrics / Analysis (READ-ONLY SUMMARY)

#### <font color = lime> New fields added to the existing Track instances (built in 7.5)

#####  7.6 is the bridge: it takes the raw tracks from 7.5 and equips them with the temporal continuity features that gating (7.7) will consume.

In [17]:
# =================================================================================================
# ✅ STEP 7.6A: GAP METRICS / ANALYSIS (READ-ONLY SUMMARY, NO LOGGER)
# =================================================================================================

from typing import List, Dict, Any
import math

def step76a_gap_metrics(tracks: List["Track"]) -> List[Dict[str, Any]]:
    """
    Read-only gap/density analysis for all tracks.
    Returns a list of dict rows (gap_stats) suitable for artifacts/CSV/plots.
    """

    if not tracks:
        print("\n📊 Step 7.6A: No tracks to analyze.")
        return []

    # Per-track rows (this is the useful artifact)
    gap_stats: List[Dict[str, Any]] = []
    for t in tracks:
        gap_stats.append({
            "track_id": t.track_id,
            "length": t.length,
            "span": t.span,
            "density": float(t.observation_density) if t.span else 0.0,
            "gap_count": t.gap_count,
            "longest_gap": t.longest_gap,
            "start_frame": t.frames[0] if t.frames else None,
            "end_frame": t.frames[-1] if t.frames else None,
        })

    # Aggregate summary
    densities = [r["density"] for r in gap_stats]
    gaps = [r["gap_count"] for r in gap_stats]
    lengths = [r["length"] for r in gap_stats]

    avg_density = sum(densities) / len(densities)
    avg_gaps = sum(gaps) / len(gaps)

    lengths_sorted = sorted(lengths)
    n = len(lengths_sorted)
    median_len = lengths_sorted[n // 2] if n else 0

    print("\n📊 STEP 7.6A: GAP / DENSITY SUMMARY")
    print("   --------------------------------")
    print(f"   Tracks analyzed     : {len(tracks)}")
    print(f"   Mean density        : {avg_density:.3f}")
    print(f"   Mean gap_count      : {avg_gaps:.2f}")
    print(f"   Tracks w/ gaps      : {sum(1 for r in gap_stats if r['gap_count'] > 0)}")
    print(f"   Tracks w/ no gaps   : {sum(1 for r in gap_stats if r['gap_count'] == 0)}")

    print("\n   Length statistics (observed frames):")
    print(f"      min / median / max : {min(lengths):d} / {median_len:d} / {max(lengths):d}")

    bucket_1   = sum(1 for L in lengths if L == 1)
    bucket_2_4 = sum(1 for L in lengths if 2 <= L <= 4)
    bucket_5_9 = sum(1 for L in lengths if 5 <= L <= 9)
    bucket_10p = sum(1 for L in lengths if L >= 10)

    print("      bucket counts      : "
          f"len=1: {bucket_1}, len 2–4: {bucket_2_4}, "
          f"len 5–9: {bucket_5_9}, len ≥10: {bucket_10p}")

    print("\n   Sample tracks (up to 5):")
    for t in tracks[:5]:
        print(
            f"      Track {t.track_id}: "
            f"span={t.span}, length={t.length}, "
            f"density={t.observation_density:.3f}, "
            f"gaps={t.gap_count}, longest_gap={t.longest_gap}"
        )

    return gap_stats


print("\n" + "=" * 70)
print("STEP 7.6A: GAP METRICS / ANALYSIS")
print("=" * 70)
print("✓ Function defined: step76a_gap_metrics(tracks) -> gap_stats (list[dict])")

# ------------------------------------------------------------------
# STEP 7.6A: EXECUTION (creates gap_stats in globals)
# ------------------------------------------------------------------
gap_stats = None

if "all_tracks" not in globals() or not all_tracks:
    print("\n⚠ Step 7.6A: `all_tracks` not found / empty yet. Run Step 7.9 first.")
else:
    gap_stats = step76a_gap_metrics(all_tracks)   # <-- THIS is what was missing
    print(f"\n✓ Step 7.6A: gap_stats created (rows={len(gap_stats)})")



STEP 7.6A: GAP METRICS / ANALYSIS
✓ Function defined: step76a_gap_metrics(tracks) -> gap_stats (list[dict])

⚠ Step 7.6A: `all_tracks` not found / empty yet. Run Step 7.9 first.


## <font color = yellow> 7.6B: Optional logger ( only)

In [18]:
# =================================================================================================
# ✅ STEP 7.6B: OPTIONAL LOGGER SNAPSHOT (OFF BY DEFAULT)
# =================================================================================================

from typing import List, Dict, Any

ENABLE_STEP76_LOG_SNAPSHOT = True  # <-- default OFF

def step76b_log_snapshot(
    gap_stats: List[Dict[str, Any]],
    stage: str = "7.6_gap_density_summary",
    log=None,
) -> None:
    """
    Optional: emit a compact snapshot to notebook logger.
    Does NOTHING unless:
      - ENABLE_STEP76_LOG_SNAPSHOT is True
      - `log` exists in globals()
    """
    if not ENABLE_STEP76_LOG_SNAPSHOT:
        print("\n⏭️ Step 7.6B: logging skipped (ENABLE_STEP76_LOG_SNAPSHOT=False)")
        return

    if "log" not in globals():
        print("\n⚠ Step 7.6B: no `log` found; cannot log snapshot.")
        return

    if not gap_stats:
        print("\n⚠ Step 7.6B: gap_stats empty; nothing to log.")
        return

    # Sort: worst density first, then shorter
    rows = sorted(gap_stats, key=lambda r: (r["density"], r["length"]))

    def slim(r):
        return {
            "id": r["track_id"],
            "len": r["length"],
            "span": r["span"],
            "dens": round(r["density"], 3),
            "gaps": r["gap_count"],
            "long": r["longest_gap"],
        }

    payload = {
        "stage": stage,
        "track_count": len(rows),
        "with_gaps": sum(1 for r in rows if r["gap_count"] > 0),
        "no_gaps": sum(1 for r in rows if r["gap_count"] == 0),
        "worst_10_tracks": [slim(r) for r in rows[:10]],
        "best_10_tracks": [slim(r) for r in rows[-10:]],
    }

    try:
        log.silent(payload, note="Step 7.6B compact snapshot")
        print("\n📁 Step 7.6B: snapshot logged (compact dict).")
    except Exception as e:
        print(f"\n⚠ Step 7.6B logging failed: {e}")


print("\n" + "=" * 70)
print("STEP 7.6B: OPTIONAL LOGGER SNAPSHOT")
print("=" * 70)
print("✓ Function defined: step76b_log_snapshot(gap_stats)")
print("   Toggle: ENABLE_STEP76_LOG_SNAPSHOT (default False)")

# -----------------------------
# AUTO-RUN (so it actually logs)
# -----------------------------
try:
    if "gap_stats" in globals():
        step76b_log_snapshot(gap_stats)
    else:
        print("\n⚠ Step 7.6B: `gap_stats` not found in globals() — run Step 7.6A first.")
except Exception as e:
    print(f"\n⚠ Step 7.6B: auto-run failed: {e}")




STEP 7.6B: OPTIONAL LOGGER SNAPSHOT
✓ Function defined: step76b_log_snapshot(gap_stats)
   Toggle: ENABLE_STEP76_LOG_SNAPSHOT (default False)

⚠ Step 7.6B: gap_stats empty; nothing to log.


## <font color = yellow> 7.7: Temporal Gates - Filter by mean_speed and ROI

In [19]:
# =================================================================================================
# ✅ STEP 7.7: TEMPORAL GATES (SIMPLE GATE - filters tracks based on min_mean_speed & ROI )
# =================================================================================================


"""
================================================================================
📌 CELL PURPOSE -   STEP 7.7: MOTION / ROI GATING
--------------------------------------------------------------------------------

Purpose:
This cell applies motion and ROI gating to enriched tracks. It filters out
trajectories that fail to meet motion thresholds or fall outside the defined
region of interest, ensuring only plausible eagle flight paths are retained.



📌 INPUT DATA / DATA CONSUMED  
--------------------------------------------------------------------------------
Uses the following Step 7.2 configurations:
- MIN_MEAN_SPEED (minimum average speed threshold)
- ROI_X_MIN, ROI_X_MAX, ROI_Y_MIN, ROI_Y_MAX (region of interest bounds)

Inputs consumed:
- tracks_enriched (output of Step 7.6)

📌 NEW DATA MADE AVAILABLE (persistent/returned)
--------------------------------------------------------------------------------

- `gated_tracks`: list of Track objects that passed all gates
- Updated track fields:
  • `status` (confirmed, rejected, reason codes)
  • Motion statistics (`disp_min`, `disp_max`, `disp_mean`, `disp_median`)
  • Heading statistics (`heading_mean`, `heading_std`)
  • Linearity ratio (`linearity_ratio`)
  • Optional hysteresis state (`promoted`, `demoted`, counters)

huh .. ..???? 
- Track fields updated:
  • `mean_speed`  
  • ROI membership flags (inside/outside corridor)


📌 Descriptive Outputs produced:
---------------------------------------------------------------------------------
- tracks_gated (subset of tracks_enriched that pass motion and ROI gating)


📌 Key considerations carried forward / Limitations
---------------------------------------------------------------------------------
- Explicit separation of ROI gating from temporal gating
- Deterministic enforcement of motion thresholds and spatial boundaries
- Visibility into rejected tracks and reasons

Limitations:
- Tracks with mean speed below MIN_MEAN_SPEED are discarded
- Tracks outside ROI bounds are discarded



📌 PROCESS FLOW (mirroring code execution)
--------------------------------------------------------------------------------
1. Iterate over tracks_enriched
2. Compute `mean_speed` for each track
3. Check ROI bounds (ROI_X_MIN, ROI_X_MAX, ROI_Y_MIN, ROI_Y_MAX)
4. Retain tracks that meet both motion and ROI criteria
5. Flag or discard tracks that fail thresholds


huh ... ???

1. Capture incoming track gap summary (for audit visibility)
2. Apply sequential gates:
   - Minimum observed length (>= MIN_TRACK_LENGTH_FRAMES)
   - Observation density (>= MIN_OBSERVATION_DENSITY)
   - Displacement range (median displacement within MIN_DISP_PX–MAX_DISP_PX)
   - Variance/linearity (linearity ratio >= MIN_LINEARITY_RATIO, heading_std <= MAX_HEADING_STD_DEG)
   - Directional constraint (heading_mean within ALLOWED_HEADING_BANDS)
   - Hysteresis (optional promote/demote logic)
3. Update track status fields (e.g., "confirmed", "too_short", "low_density")
4. Collect rejection reasons for reporting
5. Return filtered list of tracks that passed all gates



📌 PERSISTENCE NOTE
--------------------------------------------------------------------------------
Step 7.6 preserved `tracks_pre_gating` as the BEFORE snapshot. Step 7.7 consumes 
that enriched state and surfaces `gated_tracks` as the AFTER state. This enables 
benchmarking of gating outcomes, but benchmarking is a **secondary benefit** — the 
primary purpose is gating.



📌 ROLE IN PIPELINE
--------------------------------------------------------------------------------
[Reserved for later refinement once all Step 7.n cells are evaluated]
================================================================================
"""


def summarize_track_stats(track: Track) -> None:
    """
    Compute track statistics from observations.
    
    NOTE: This operates only on OBSERVED frames (not predicted/gap frames).
    Statistics computed:
      - Displacement: min, max, mean, median
      - Heading: mean, std deviation
      - Linearity: net displacement / path length
    
    Args:
        track: Track with populated observations
    
    Side Effects:
        Updates track statistics fields (disp_mean, heading_std, etc.)
    """
    
    # Displacement statistics (from step_displacements computed in build_tracks_standard)
    disps = track.step_displacements
    if disps:
        track.disp_min = min(disps)
        track.disp_max = max(disps)
        track.disp_mean = sum(disps) / len(disps)
        track.disp_median = sorted(disps)[len(disps) // 2]
    else:
        track.disp_min = None
        track.disp_max = None
        track.disp_mean = None
        track.disp_median = None
    
    # Heading statistics
    headings = track.step_headings
    if headings:
        mean = sum(headings) / len(headings)
        track.heading_mean = mean
        var = sum((h - mean) ** 2 for h in headings) / len(headings)
        track.heading_std = math.sqrt(var) if var > 0 else 0.0
    else:
        track.heading_mean = None
        track.heading_std = None
    
    # Linearity ratio (net displacement / total path length)
    centroids = track.get_centroids_list()
    if len(centroids) >= 2 and disps:
        # Net displacement (straight-line distance from start to end)
        net_dx = centroids[-1][0] - centroids[0][0]
        net_dy = centroids[-1][1] - centroids[0][1]
        net_disp = math.hypot(net_dx, net_dy)
        
        # Path length (sum of step displacements)
        path_len = sum(disps)
        
        track.linearity_ratio = (net_disp / path_len) if path_len > 0 else 0.0
    else:
        track.linearity_ratio = None


def gate_track_observation_density(track: Track) -> bool:
    """
    Gate: Reject tracks with too many gaps (low observation density).
    
    RATIONALE:
    ----------
    observation_density = observed_frames / total_span
    
    Example:
      Track spans frames 10-20 (11 frames total)
      Observed at frames: 10, 11, 13, 15, 16, 18, 20 (7 observations)
      Density = 7/11 = 0.636 (63.6%)
    
    Threshold:
      MIN_OBSERVATION_DENSITY = 0.65
      → This track FAILS (0.636 < 0.65)
    
    Adjust threshold based on expected occlusion frequency:
      - Low occlusion (open sky): 0.80-0.90
      - Moderate occlusion (trees): 0.65-0.75
      - Heavy occlusion (urban): 0.50-0.60
    
    Args:
        track: Track to evaluate
    
    Returns:
        True if density >= threshold, False otherwise
    """
    return track.observation_density >= MIN_OBSERVATION_DENSITY


def apply_temporal_gates(tracks: List[Track]) -> List[Track]:
    """
    Apply all temporal gates to filter tracks.

    GATES APPLIED (in order, each controlled by a USE_* flag):
    ----------------------------------------------------------
    1. Minimum observed length        (USE_LENGTH_GATE)
    2. Observation density            (USE_DENSITY_GATE)
    3. Displacement range             (USE_DISPLACEMENT_GATE)
    4. Variance / linearity           (USE_VARIANCE_GATE)
    5. Directional constraint         (USE_DIRECTIONAL_GATE)
    6. Hysteresis (promotion/demotion (USE_HYSTERESIS_GATE)

    Args:
        tracks: All tracks from build_tracks_standard

    Returns:
        Filtered list of tracks that passed all enabled gates.
    """

    # =====================================================================
    # 📊 CAPTURE INCOMING TRACK GAP DATA BEFORE FILTERING
    # =====================================================================
    print(f"\n📊 INCOMING TRACK GAP SUMMARY (before temporal gates):")
    print(f"   Total tracks: {len(tracks)}")

    tracks_with_gaps = [t for t in tracks if t.gap_count > 0]
    print(f"   Tracks with gaps: {len(tracks_with_gaps)}")

    if tracks_with_gaps:
        print(f"\n   Sample gap details (first 5 tracks with gaps):")
        for i, track in enumerate(tracks_with_gaps[:5]):
            print(f"\n      Track {track.track_id}:")
            print(f"         gap_frames: {track.gap_frames_sorted}")
            print(f"         gap_sequences: {track.gap_sequences}")
            print(f"         longest_gap: {track.longest_gap}")
            print(f"         observation_density: {track.observation_density:.3f}")


    # =====================================================================
    # 📊 CAPTURE INCOMING TRACK GAP DATA BEFORE FILTERING
    # =====================================================================
    print(f"\n📊 INCOMING TRACK GAP SUMMARY (before temporal gates):")
    print(f"   Total tracks: {len(tracks)}")
    # ...

   
    # =====================================================================
    # NEW: CAPTURE REAL GATING CONFIG FOR STEP 7.D DIAGNOSTICS
    # =====================================================================
    # Only do this if the shared artifacts dict exists (defined in Step 7 config).
    if "STEP7_ARTIFACTS" in globals():
        try:
            real_gating_config = dict(
                min_track_length=MIN_TRACK_LENGTH_FRAMES,
                min_observation_density=MIN_OBSERVATION_DENSITY,
                use_length_gate=USE_LENGTH_GATE,
                use_density_gate=USE_DENSITY_GATE,
                use_displacement_gate=USE_DISPLACEMENT_GATE,
                use_variance_gate=USE_VARIANCE_GATE,
                use_directional_gate=USE_DIRECTIONAL_GATE,
                use_hysteresis_gate=USE_HYSTERESIS_GATE,
                allowed_heading_bands=ALLOWED_HEADING_BANDS,
            )
            STEP7_ARTIFACTS["real_gating_config"] = real_gating_config
        except Exception as e:
            # Don’t let diagnostics break the main pipeline
            print(f"⚠ STEP7_DIAG: Failed to capture gating config: {e}")

    # =====================================================================
    # NOW PROCEED WITH GATING AS NORMAL
    # =====================================================================
    kept: List[Track] = []

    
    print(f"\n🚦 Applying temporal gates to {len(tracks)} tracks...")

    # Counters for reporting
    reject_reasons = {
        "too_short": 0,
        "low_density": 0,
        "displacement": 0,
        "variance": 0,
        "direction": 0,
        "hysteresis": 0,
    }

    for track in tracks:
        # -----------------------------------------------------------------
        # GATE 1: Minimum Observed Length
        # -----------------------------------------------------------------
        if USE_LENGTH_GATE and track.length < MIN_TRACK_LENGTH_FRAMES:
            track.status = "too_short"
            reject_reasons["too_short"] += 1
            continue

        # -----------------------------------------------------------------
        # GATE 2: Observation Density (uses 7.6’s observation_density)
        # -----------------------------------------------------------------
        if USE_DENSITY_GATE:
            if track.observation_density < MIN_OBSERVATION_DENSITY:
                track.status = "low_density"
                reject_reasons["low_density"] += 1
                continue

        # -----------------------------------------------------------------
        # GATE 3–5: Motion-based gates need stats
        # -----------------------------------------------------------------
        summarize_track_stats(track)

        # GATE 3: Displacement range
        if USE_DISPLACEMENT_GATE:
            if track.disp_median is None:
                ok_disp = False
            else:
                ok_disp = (MIN_DISP_PX <= track.disp_median <= MAX_DISP_PX)

            if not ok_disp:
                track.status = "displacement_gate"
                reject_reasons["displacement"] += 1
                continue

        # GATE 4: Variance / linearity
        if USE_VARIANCE_GATE:
            ok_var = True

            if track.linearity_ratio is None or track.linearity_ratio < MIN_LINEARITY_RATIO:
                ok_var = False
            if track.heading_std is None or track.heading_std > MAX_HEADING_STD_DEG:
                ok_var = False

            if not ok_var:
                track.status = "variance_gate"
                reject_reasons["variance"] += 1
                continue

        # GATE 5: Directional constraint
        if USE_DIRECTIONAL_GATE:
            ok_dir = False
            if track.heading_mean is not None and ALLOWED_HEADING_BANDS:
                for (mn, mx) in ALLOWED_HEADING_BANDS:
                    if mn <= track.heading_mean <= mx:
                        ok_dir = True
                        break
            if not ok_dir:
                track.status = "direction_gate"
                reject_reasons["direction"] += 1
                continue

        # -----------------------------------------------------------------
        # GATE 6: Hysteresis (optional promotion/demotion layer)
        # -----------------------------------------------------------------
        if USE_HYSTERESIS_GATE:
            # For now we treat “made it through all gates above” as a “good” track
            is_good = True
            track_dict = track_to_dict(track)
            update_track_hysteresis(track_dict, is_good)

            track.status = track_dict["status"]
            track.promoted = track_dict.get("promoted", False)
            track.demoted = track_dict.get("demoted", False)

            if track.promoted and not track.demoted:
                kept.append(track)
            else:
                reject_reasons["hysteresis"] += 1
        else:
            # No hysteresis: everything that survived prior gates is kept
            track.status = "confirmed"
            kept.append(track)

    # === Compact summary block ===
    total_tracks = len(tracks)
    total_rejected = sum(reject_reasons.values())

    print("\n📊 GATING SUMMARY (compact):")
    print(f"   Total tracks: {total_tracks}")
    print(f"   Passed: {len(kept)}")
    print(f"   Rejected: {total_rejected}")
    if total_rejected > 0:
        print("   Breakdown:")
        for reason, count in reject_reasons.items():
            if count > 0:
                print(f"      - {reason}: {count}")

    # -----------------------------------------------------------------
    # NEW: Persist gating results for Step 7.D diagnostics
    # -----------------------------------------------------------------
    if "STEP7_ARTIFACTS" in globals():
        try:
            STEP7_ARTIFACTS["gating_reject_reasons"] = reject_reasons
            STEP7_ARTIFACTS["gated_track_ids"] = [t.track_id for t in kept]
        except Exception as e:
            print(f"⚠ STEP7_DIAG: Failed to capture gating results: {e}")

    return kept


def update_track_hysteresis(track_dict: Dict[str, Any], is_good_now: bool) -> None:
    """
    Update hysteresis state for track (Step 5 compatibility).
    
    HYSTERESIS LOGIC:
    -----------------
    - Track must be "good" for PROMOTE_MIN_CONSEC_GOOD consecutive frames to promote
    - Promoted track must be "bad" for DEMOTE_MAX_CONSEC_BAD consecutive frames to demote
    
    This prevents jittery gate decisions from fragmenting tracks.
    
    Args:
        track_dict: Track dictionary (modified in-place)
        is_good_now: Whether track passes gates in current evaluation
    
    Side Effects:
        Updates track_dict fields: consecutive_good_frames, promoted, status, etc.
    """
    
    if not USE_HYSTERESIS_GATE:
        track_dict["status"] = "confirmed" if is_good_now else "rejected"
        track_dict["promoted"] = is_good_now
        return
    
    # Update counters
    if is_good_now:
        track_dict["good_frame_count"] = track_dict.get("good_frame_count", 0) + 1
        track_dict["consecutive_good_frames"] = track_dict.get("consecutive_good_frames", 0) + 1
        track_dict["consecutive_bad_frames"] = 0
    else:
        track_dict["bad_frame_count"] = track_dict.get("bad_frame_count", 0) + 1
        track_dict["consecutive_bad_frames"] = track_dict.get("consecutive_bad_frames", 0) + 1
        track_dict["consecutive_good_frames"] = 0
    
    # Promotion logic (candidate → confirmed)
    # Requires PROMOTE_MIN_CONSEC_GOOD consecutive good frames (if defined)
    PROMOTE_MIN_CONSEC_GOOD = 3  # Default value (define in config if needed)
    
    if (not track_dict.get("promoted", False)) and \
       track_dict.get("consecutive_good_frames", 0) >= PROMOTE_MIN_CONSEC_GOOD:
        track_dict["promoted"] = True
        track_dict["status"] = "confirmed"
    
    # Demotion logic (confirmed → demoted)
    # Requires DEMOTE_MAX_CONSEC_BAD consecutive bad frames (if defined)
    DEMOTE_MAX_CONSEC_BAD = 5  # Default value (define in config if needed)
    
    if track_dict.get("promoted", False) and \
       track_dict.get("consecutive_bad_frames", 0) >= DEMOTE_MAX_CONSEC_BAD:
        track_dict["demoted"] = True
        track_dict["status"] = "demoted"



print("\n🚦 Gates Applied:")
print("\n======================================================================")
print("✅ STEP 7.7: TEMPORAL / MOTION GATES READY")
print("======================================================================")
print("This cell defines the complete gating workflow used in Step 7.9.")
print("Gates are modular and controlled entirely by the USE_* flags set in Step 7.2.\n")

print("🚦 Available Gates (toggle via Step 7.2):")
print("   • USE_LENGTH_GATE        → minimum observed length")
print("   • USE_DENSITY_GATE       → observation-density requirement (7.6 output)")
print("   • USE_DISPLACEMENT_GATE  → median displacement inside allowed range")
print("   • USE_VARIANCE_GATE      → linearity + heading variance constraints")
print("   • USE_DIRECTIONAL_GATE   → heading must fall inside allowed bands")
print("   • USE_HYSTERESIS_GATE    → optional promotion/demotion layer\n")

print("📌 Notes:")
print("   • No stitching occurs here (no merging, no extrapolation).")
print("   • Gating is purely a filter over the tracks produced in 7.5.")
print("   • All gap/density data used here comes from Step 7.6.")
print("   • Final kept tracks flow into scoring (7.8) and interpolation (7.10).\n")

print("✓ apply_temporal_gates() is ready for full-pipeline use.")




🚦 Gates Applied:

✅ STEP 7.7: TEMPORAL / MOTION GATES READY
This cell defines the complete gating workflow used in Step 7.9.
Gates are modular and controlled entirely by the USE_* flags set in Step 7.2.

🚦 Available Gates (toggle via Step 7.2):
   • USE_LENGTH_GATE        → minimum observed length
   • USE_DENSITY_GATE       → observation-density requirement (7.6 output)
   • USE_DISPLACEMENT_GATE  → median displacement inside allowed range
   • USE_VARIANCE_GATE      → linearity + heading variance constraints
   • USE_DIRECTIONAL_GATE   → heading must fall inside allowed bands
   • USE_HYSTERESIS_GATE    → optional promotion/demotion layer

📌 Notes:
   • No stitching occurs here (no merging, no extrapolation).
   • Gating is purely a filter over the tracks produced in 7.5.
   • All gap/density data used here comes from Step 7.6.
   • Final kept tracks flow into scoring (7.8) and interpolation (7.10).

✓ apply_temporal_gates() is ready for full-pipeline use.


## <font color = yellow>  7.8: Quality Scoring (Gap-aware) plus

    The Two Components: (Balanced Approach)
    
    Rewards tracks that are both long AND complete
    Penalizes tracks that have good motion but sparse data
    Penalizes tracks that have complete data but poor motion
    
    Quality Score (0.0 to 1.0)
    Evaluates the observed motion characteristics:
    
    ✅ Length: Longer tracks score higher (saturates at ~40 frames)
    ✅ Heading alignment: Tracks moving in expected direction (-45° to +45° for rightward flight)
    ✅ Vertical position: Tracks staying in expected Y corridor (0-400px for eagles)
    
    Weighted formula:
    
    50% length
    30% heading alignment
    20% vertical corridor occupancy

    Not Just Longest:
    
        A 100-frame track with 80 gaps (20% density) is not reliable even if long
    
    Not Just Fewest Gaps:
    
        A 3-frame track with zero gaps is too short to be meaningful
    
    Balanced Approach:
    
    Rewards tracks that are both long AND complete
    Penalizes tracks that have good motion but sparse data
    Penalizes tracks that have complete data but poor motion



In [20]:
# =================================================================================================
# ✅ STEP 7.8: QUALITY SCORING (GAP-AWARE)
# =================================================================================================


"""

docstring
================================================================================
📌 CELL PURPOSE -   STEP 7.8: QUALITY SCORING (GAP‑AWARE)
--------------------------------------------------------------------------------

Purpose:
This cell implements gap‑aware quality scoring for tracks in Step 7. It evaluates
trajectory quality based on both motion characteristics and completeness of
observations. Each track receives:
- quality_score: a measure of trajectory goodness (smoothness, alignment, corridor fit)
- confidence_score: a measure of trajectory completeness, adjusted for temporal gaps

The scoring process computes heading and vertical statistics, then applies
weighted components to derive both scores. "Gap‑aware" means confidence_score
explicitly incorporates observation_density, penalizing tracks with missing
frames or sparse coverage.
Robust handling of short or stationary tracks (minimum length, zero-displacement skips)


📌 INPUT DATA / DATA CONSUMED  
--------------------------------------------------------------------------------
Uses the following Step 7.2 configurations:
- MIN_LEN_FOR_SCORING (minimum track length threshold)
- EXPECTED_HEADING_BAND (heading alignment tolerance)
- EAGLE_Y_MIN, EAGLE_Y_MAX (vertical corridor bounds)
- TRACK_SCORE_W_LEN, TRACK_SCORE_W_HEADING, TRACK_SCORE_W_Y_BAND (scoring weights)

Inputs consumed:
- tracks_gated (the surviving set from Step 7.7)
  • `track.length` (from Step 7.6 enrichment)
  • `track.observation_density` (from Step 7.6 enrichment)

📌 NEW DATA MADE AVAILABLE (persistent/returned)
--------------------------------------------------------------------------------
- Track fields enriched:
  • `track.mean_heading_deg`, `track.std_heading_deg`  
  • `track.mean_y`, `track.min_y`, `track.max_y`, `track.y_band_fraction`  
  • `track.quality_score`, `track.confidence_score`

📌 Descriptive Outputs produced:
---------------------------------------------------------------------------------
- tracks_gated (same set, enriched in-place with scoring attributes)

Outputs produced:
- tracks_gated (same set, no further reduction)
- Enriched with:
  - track.mean_heading_deg, track.std_heading_deg
  - track.mean_y, track.min_y, track.max_y, track.y_band_fraction
  - track.quality_score, track.confidence_score


📌 Key considerations carried forward / Limitations
---------------------------------------------------------------------------------
- Separation of trajectory quality (quality_score) from completeness (confidence_score)
- Explicit use of precomputed attributes and tunable constants
- Deterministic scoring formula with transparent component contributions

Limitations:
- Requires minimum length (MIN_LEN_FOR_SCORING) to produce meaningful scores
- Heading alignment uses exponential decay with fixed 30° half‑life (tunable)
- Confidence_score sensitivity depends on observation_density; sparse tracks may be penalized heavily

📌 PROCESS FLOW (mirroring code execution)
--------------------------------------------------------------------------------
1. Iterate over tracks_gated
2. Compute heading statistics (mean, std)
3. Compute vertical statistics (mean_y, min_y, max_y, y_band_fraction)
4. Apply scoring weights (TRACK_SCORE_W_LEN, TRACK_SCORE_W_HEADING, TRACK_SCORE_W_Y_BAND)
5. Assign `quality_score` and `confidence_score` to each track
6. Summarize results (count, mean quality, mean confidence)

📌 ROLE IN PIPELINE
--------------------------------------------------------------------------------
[Reserved for later refinement once all Step 7.n cells are evaluated]
================================================================================
"""


def compute_track_heading_stats(track: Track) -> None:
    """
    Compute heading statistics from observed centroids.
    
    Computes:
      - Heading for each step (angle between consecutive observations)
      - Mean heading
      - Heading standard deviation
    
    Args:
        track: Track with observations
    
    Side Effects:
        Updates track.mean_heading_deg, track.std_heading_deg
    """

    
    centroids = track.get_centroids_list()
    
    if len(centroids) < 2:
        track.mean_heading_deg = None
        track.std_heading_deg = None
        return
    
    headings: List[float] = []
    
    # Loop over consecutive centroid pairs
    for (x0, y0), (x1, y1) in zip(centroids[:-1], centroids[1:]):
        dx = x1 - x0
        dy = y1 - y0
        
        # Skip zero displacement (stationary)
        if dx == 0 and dy == 0:
            continue
        
        # Compute heading angle (standard image coords: +x right, +y down)
        # atan2(-dy, dx) gives heading in standard mathematical convention
        angle_rad = math.atan2(-dy, dx)
        angle_deg = math.degrees(angle_rad)
        headings.append(angle_deg)
    
    if not headings:
        track.mean_heading_deg = None
        track.std_heading_deg = None
        return
    
    # Compute statistics
    mean_h = sum(headings) / len(headings)
    var_h = sum((h - mean_h) ** 2 for h in headings) / len(headings)
    std_h = math.sqrt(var_h)
    
    track.mean_heading_deg = mean_h
    track.std_heading_deg = std_h


def compute_track_vertical_stats(track: Track) -> None:
    """
    Compute vertical position statistics.
    
    Computes:
      - Mean Y position
      - Min/Max Y
      - Fraction of observations in expected Y corridor (EAGLE_Y_MIN to EAGLE_Y_MAX)
    
    Args:
        track: Track with observations
    
    Side Effects:
        Updates track.mean_y, track.min_y, track.max_y, track.y_band_fraction
    """
    
    centroids = track.get_centroids_list()
    
    if not centroids:
        track.mean_y = None
        track.min_y = None
        track.max_y = None
        track.y_band_fraction = 0.0
        return
    
    # Extract Y coordinates
    ys = [cy for (_, cy) in centroids]
    
    # Statistics
    mean_y = sum(ys) / len(ys)
    min_y = min(ys)
    max_y = max(ys)
    
    # Fraction in expected corridor
    in_band = sum(1 for y in ys if EAGLE_Y_MIN <= y <= EAGLE_Y_MAX)
    frac_band = in_band / len(ys)
    
    track.mean_y = mean_y
    track.min_y = min_y
    track.max_y = max_y
    track.y_band_fraction = frac_band


def score_track_quality(track: Track) -> float:
    """
    Compute gap-aware quality score for track.
    
    SCORING COMPONENTS:
    -------------------
    1. Length term: Reward longer tracks
       - Uses OBSERVED frames (track.length)
       - Normalized by expected length (e.g., 40 frames)
    
    2. Heading alignment: Reward tracks aligned with expected direction
       - Expected band: EXPECTED_HEADING_BAND (e.g., -45° to +45° for rightward)
       - Exponential decay for deviation from band
    
    3. Vertical corridor: Reward tracks in expected Y range
       - Expected range: EAGLE_Y_MIN to EAGLE_Y_MAX
       - Fraction of observations in range
    
    4. Gap penalty (NEW): Penalize tracks with many gaps
       - observation_density = observed / span
       - Lower density → lower confidence
    
    FORMULA:
    --------
    base_score = w_len * len_norm + w_heading * heading_term + w_y * y_term
    quality_score = base_score (no gap penalty in base score)
    confidence_score = quality_score * observation_density
    
    INTERPRETATION:
    ---------------
    - quality_score: How good is the observed trajectory?
    - confidence_score: How complete is the track? (accounts for gaps)
    
    Example:
      Track A: 40 frames observed, 40 frame span → density = 1.0
      Track B: 40 frames observed, 60 frame span → density = 0.67
      
      If both have same base_score = 0.8:
        Track A confidence = 0.8 * 1.0 = 0.80
        Track B confidence = 0.8 * 0.67 = 0.53
      
      → Track A is "more confident" (fewer gaps)
    
    Args:
        track: Track to score
    
    Returns:
        confidence_score (quality adjusted by observation density)
    
    Side Effects:
        Updates track.quality_score, track.confidence_score
    """
    
    
    # -------------------------------------------------------------
    # BASIC GUARDS
    # -------------------------------------------------------------
    n = track.length  # number of OBSERVED frames

    if n < MIN_LEN_FOR_SCORING:
        track.quality_score = 0.0
        track.confidence_score = 0.0
        track.score_meta = {
            "n_obs": n,
            "reason": "below_min_length",
        }
        return 0.0

    # -------------------------------------------------------------
    # TERM 1: LENGTH
    # -------------------------------------------------------------
    len_norm = min(n / 40.0, 1.0)

    # -------------------------------------------------------------
    # TERM 2: HEADING
    # -------------------------------------------------------------
    mean_h = track.mean_heading_deg

    if mean_h is None:
        heading_term = 0.0
        dist_to_band = None
    else:
        lo, hi = EXPECTED_HEADING_BAND
        if lo <= mean_h <= hi:
            heading_term = 1.0
            dist_to_band = 0.0
        else:
            dist_to_band = min(abs(mean_h - lo), abs(mean_h - hi))
            heading_term = math.exp(-dist_to_band / 30.0)

    # -------------------------------------------------------------
    # TERM 3: VERTICAL CORRIDOR
    # -------------------------------------------------------------
    y_band_frac = track.y_band_fraction

    # -------------------------------------------------------------
    # COMBINE
    # -------------------------------------------------------------
    base_score = (
        TRACK_SCORE_W_LEN * len_norm +
        TRACK_SCORE_W_HEADING * heading_term +
        TRACK_SCORE_W_Y_BAND * y_band_frac
    )

    confidence_score = base_score * track.observation_density

    track.quality_score = base_score
    track.confidence_score = confidence_score

    # -------------------------------------------------------------
    # 🔑 SCORE METADATA (THIS WAS YOUR ERROR)
    # -------------------------------------------------------------
    track.score_meta = dict(
        n_obs=n,
        len_norm=len_norm,
        mean_heading_deg=mean_h,
        expected_heading_band=EXPECTED_HEADING_BAND,
        heading_term=heading_term,
        dist_to_band=dist_to_band,
        y_band_fraction=y_band_frac,
        observation_density=track.observation_density,
        base_score=base_score,
        quality_score=base_score,
        confidence_score=confidence_score,
        weights=dict(
            w_len=TRACK_SCORE_W_LEN,
            w_heading=TRACK_SCORE_W_HEADING,
            w_y=TRACK_SCORE_W_Y_BAND,
        ),
    )

    return confidence_score


# gs
# --- Runtime summary block ---
def summarize_quality_scoring(tracks: List[Track]) -> None:
    """
    Print compact summary of quality/confidence across tracks.
    """
    scored = [t for t in tracks if hasattr(t, "confidence_score")]
    print("\n📊 QUALITY SCORING SUMMARY (Step 7.8):")
    print(f"   Tracks scored: {len(scored)}")
    if scored:
        avg_quality = sum(t.quality_score for t in scored) / len(scored)
        avg_conf = sum(t.confidence_score for t in scored) / len(scored)
        print(f"   Mean quality_score: {avg_quality:.3f}")
        print(f"   Mean confidence_score: {avg_conf:.3f}")


print("✓ Settings and Setup Only. Quality scoring helpers defined. No real output from this cell.\n")
print("\n📊 Scoring Components:")
print("   1. Length term: Reward longer tracks (observed frames)")
print("   2. Heading alignment: Reward expected direction")
print("   3. Vertical corridor: Reward expected Y range")
print("   4. Gap penalty: observation_density multiplier")
print("\n✅ confidence_score = quality_score × observation_density")

# =================================================================================================
# STEP 7.8 (ADD-ON): GATE STATUS DASHBOARD (PRINTS ACTIVE/INACTIVE + THRESHOLDS)
# Put this near the top of 7.8, before scoring functions run.
# =================================================================================================

def _get(name, default=None):
    return globals().get(name, default)

def print_gate_status_dashboard():
    # Gate toggles (default to True if you historically treat them as “on unless disabled”)
    use_disp = bool(_get("USE_DISPLACEMENT_GATE", True))
    use_var  = bool(_get("USE_VARIANCE_GATE", True))
    use_dir  = bool(_get("USE_DIRECTIONAL_GATE", True))
    use_hyst = bool(_get("USE_HYST_GATE", False))  # only if you actually have this gate
    
    # Core thresholds / settings used by gates
    min_len   = _get("MIN_TRACK_LENGTH", _get("MIN_LEN_FOR_SCORING", None))
    min_dens  = _get("MIN_OBSERVATION_DENSITY", None)
    bands     = _get("ALLOWED_HEADING_BANDS", None)

    # Displacement gate thresholds (names vary between your versions)
    disp_min_total = _get("MIN_TOTAL_DISPLACEMENT", None)
    disp_min_step  = _get("MIN_STEP_DISPLACEMENT", None)

    # Variance gate thresholds (if applicable)
    var_max = _get("MAX_POSITION_VARIANCE", _get("VAR_GATE_MAX", None))

    print("\n" + "=" * 80)
    print("STEP 7.8 — CURRENT GATING STATUS (for clarity; gates run in 7.7, not 7.8)")
    print("=" * 80)

    def onoff(v): 
        return "ON ✅" if v else "OFF ⛔"

    print("\nActive / inactive toggles:")
    print(f"  • USE_DISPLACEMENT_GATE : {onoff(use_disp)}")
    print(f"  • USE_VARIANCE_GATE     : {onoff(use_var)}")
    print(f"  • USE_DIRECTIONAL_GATE  : {onoff(use_dir)}")
    if "USE_HYST_GATE" in globals():
        print(f"  • USE_HYST_GATE         : {onoff(use_hyst)}")

    print("\nThresholds / parameters:")
    print(f"  • MIN_TRACK_LENGTH (length gate)        : {min_len}")
    print(f"  • MIN_OBSERVATION_DENSITY (density gate): {min_dens}")
    print(f"  • ALLOWED_HEADING_BANDS (dir gate)      : {bands}")

    if use_disp:
        print("\nDisplacement gate params:")
        print(f"  • MIN_TOTAL_DISPLACEMENT : {disp_min_total}")
        print(f"  • MIN_STEP_DISPLACEMENT  : {disp_min_step}")

    if use_var:
        print("\nVariance gate params:")
        print(f"  • MAX_POSITION_VARIANCE / VAR_GATE_MAX : {var_max}")

    # Optional: show temporal window settings if they exist (not a gate, but affects fragmentation)
    fw  = _get("FRAME_WINDOW", None)
    buf = _get("BUFFER", None)
    mm  = _get("MAX_MISS", None)
    if fw is not None or buf is not None or mm is not None:
        print("\nTemporal association window (affects fragmentation):")
        print(f"  • FRAME_WINDOW : {fw}")
        print(f"  • BUFFER       : {buf}")
        print(f"  • MAX_MISS     : {mm}")

    print("=" * 80 + "\n")

# Call it once when 7.8 is run (safe even if some vars don’t exist)
print_gate_status_dashboard()





✓ Settings and Setup Only. Quality scoring helpers defined. No real output from this cell.


📊 Scoring Components:
   1. Length term: Reward longer tracks (observed frames)
   2. Heading alignment: Reward expected direction
   3. Vertical corridor: Reward expected Y range
   4. Gap penalty: observation_density multiplier

✅ confidence_score = quality_score × observation_density

STEP 7.8 — CURRENT GATING STATUS (for clarity; gates run in 7.7, not 7.8)

Active / inactive toggles:
  • USE_DISPLACEMENT_GATE : OFF ⛔
  • USE_VARIANCE_GATE     : OFF ⛔
  • USE_DIRECTIONAL_GATE  : OFF ⛔

Thresholds / parameters:
  • MIN_TRACK_LENGTH (length gate)        : 8
  • MIN_OBSERVATION_DENSITY (density gate): 0.5
  • ALLOWED_HEADING_BANDS (dir gate)      : [(-45, 45), (-135, 135)]

Temporal association window (affects fragmentation):
  • FRAME_WINDOW : 4
  • BUFFER       : 2
  • MAX_MISS     : 6



In [21]:
def print_step7_config_dashboard():
    print("\n" + "=" * 78)
    print("🔧 STEP 7 CONFIG DASHBOARD (READ-ONLY)")
    print("=" * 78)

    # ------------------------------------------------------------------
    # 1) TEMPORAL ASSOCIATION (Step 7.5)
    #    Affects fragmentation vs long tracks
    # ------------------------------------------------------------------
    print("\n🧩 Temporal Association Window (Step 7.5)")
    print("   (controls stitching / fragmentation — NOT rejection)")
    print(f"   • FRAME_WINDOW : {FRAME_WINDOW}")
    print(f"   • BUFFER       : {BUFFER}")
    print(f"   • MAX_MISS     : {MAX_MISS}")

    # ------------------------------------------------------------------
    # 2) TEMPORAL GATES – TOGGLES (Step 7.7)
    #    Whether a gate is active at all
    # ------------------------------------------------------------------
    print("\n🚧 Temporal Gates – Active Toggles (Step 7.7)")
    gate_flags = {
        "USE_DISPLACEMENT_GATE": USE_DISPLACEMENT_GATE,
        "USE_VARIANCE_GATE": USE_VARIANCE_GATE,
        "USE_DIRECTIONAL_GATE": USE_DIRECTIONAL_GATE,
    }

    for k, v in gate_flags.items():
        state = "ON  ✅" if v else "OFF ⛔"
        print(f"   • {k:24s}: {state}")

    # ------------------------------------------------------------------
    # 3) GATING THRESHOLDS (used only if gate is active)
    # ------------------------------------------------------------------
    print("\n📏 Gating Thresholds (7.7 consumes these)")
    print("   (values alone do NOTHING unless the gate is ON)")

    print(f"   • MIN_TRACK_LENGTH_FRAMES        : {MIN_TRACK_LENGTH_FRAMES}")
    print(f"   • MIN_OBSERVATION_DENSITY : {MIN_OBSERVATION_DENSITY}")
    print(f"   • ALLOWED_HEADING_BANDS   : {ALLOWED_HEADING_BANDS}")

    # ------------------------------------------------------------------
    # 4) QUALITY SCORING INPUTS (Step 7.8)
    # ------------------------------------------------------------------
    print("\n🏁 Quality Scoring Inputs (Step 7.8)")
    print("   (used for ranking — does NOT reject tracks)")

    print(f"   • MIN_LEN_FOR_SCORING     : {MIN_LEN_FOR_SCORING}")
    print(f"   • EXPECTED_HEADING_BAND   : {EXPECTED_HEADING_BAND}")
    print(f"   • EAGLE_Y_MIN / MAX       : {EAGLE_Y_MIN} → {EAGLE_Y_MAX}")

    print("\n   Weights:")
    print(f"     – TRACK_SCORE_W_LEN     : {TRACK_SCORE_W_LEN}")
    print(f"     – TRACK_SCORE_W_HEADING : {TRACK_SCORE_W_HEADING}")
    print(f"     – TRACK_SCORE_W_Y_BAND  : {TRACK_SCORE_W_Y_BAND}")

    print("\n" + "=" * 78)

# fct call
print_step7_config_dashboard()



🔧 STEP 7 CONFIG DASHBOARD (READ-ONLY)

🧩 Temporal Association Window (Step 7.5)
   (controls stitching / fragmentation — NOT rejection)
   • FRAME_WINDOW : 4
   • BUFFER       : 2
   • MAX_MISS     : 6

🚧 Temporal Gates – Active Toggles (Step 7.7)
   • USE_DISPLACEMENT_GATE   : OFF ⛔
   • USE_VARIANCE_GATE       : OFF ⛔
   • USE_DIRECTIONAL_GATE    : OFF ⛔

📏 Gating Thresholds (7.7 consumes these)
   (values alone do NOTHING unless the gate is ON)
   • MIN_TRACK_LENGTH_FRAMES        : 8
   • MIN_OBSERVATION_DENSITY : 0.5
   • ALLOWED_HEADING_BANDS   : [(-45, 45), (-135, 135)]

🏁 Quality Scoring Inputs (Step 7.8)
   (used for ranking — does NOT reject tracks)
   • MIN_LEN_FOR_SCORING     : 8
   • EXPECTED_HEADING_BAND   : (-45.0, 45.0)
   • EAGLE_Y_MIN / MAX       : 0 → 400

   Weights:
     – TRACK_SCORE_W_LEN     : 0.5
     – TRACK_SCORE_W_HEADING : 0.3
     – TRACK_SCORE_W_Y_BAND  : 0.2



## <font color = yellow> 7.9: Full-pipeline Execution





In [22]:
# =================================================================================================
# STEP 7.9: FULL PIPELINE EXECUTION (STANDARD APPROACH)
# =================================================================================================

"""
================================================================================
📌 CELL PURPOSE - STEP 7.9: FULL PIPELINE EXECUTION (STANDARD APPROACH)
--------------------------------------------------------------------------------

This cell wires together the Step 7 sub-stages into a single callable function:

  1. Detection          (Steps 1–4)      → run_detection()
  2. Track building     (Step 7.5)       → build_tracks_standard()
  3. Gap enrichment     (Step 7.6)       → analyze_track_gaps()
  4. Temporal gates     (Step 7.7)       → apply_temporal_gates()
  5. Quality scoring    (Step 7.8)       → compute_* + score_track_quality()

It also provides an optional “auto-run” block at the bottom so this cell can
either:
    • define the pipeline only, OR
    • define + immediately execute a full run on VIDEO_PATH.

No constructors are redefined here. This cell only:
    • uses the existing Track dataclass
    • calls previously defined functions from 7.4, 7.6, 7.7, 7.8.
================================================================================
"""

from typing import List, Dict, Any, Tuple


# ---------------------------------------------------------------------------------
# STAGE 1: DETECTION (Steps 1–4)  
# Does not create a preliminary track or segment.  Only creates a frame‑indexed list of blobs.

# ---------------------------------------------------------------------------------
def run_detection(
    video_path: str,
    diff_threshold: float,
    min_area: int,
    render_frames: bool = False,
) -> List[Dict[str, Any]]:
    """
    Frame-diff based motion detection (Steps 1–4).

    Performs:
      1. Frame differencing (prev vs current)
      2. Thresholding
      3. Optional ROI removal
      4. Optional morphological opening/closing
      5. Contour extraction → bboxes + centroids
    """

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    prev_gray = None
    prev_centroids: List[Tuple[int, int]] = []
    frame_index = 0
    detections: List[Dict[str, Any]] = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, BLUR_KERNEL, 0)

        # First frame only seeds prev_gray
        if prev_gray is None:
            prev_gray = gray
            frame_index += 1
            continue

        # Frame differencing + threshold
        diff = cv2.absdiff(gray, prev_gray)
        _, mask = cv2.threshold(diff, diff_threshold, 255, cv2.THRESH_BINARY)

        # ROI removal
        if USE_ROI_REMOVE:
            x1, y1, x2, y2 = ROI_REMOVE_RECT
            h, w = mask.shape[:2]
            x1 = max(0, min(x1, w))
            x2 = max(0, min(x2, w))
            y1 = max(0, min(y1, h))
            y2 = max(0, min(y2, h))
            if x2 > x1 and y2 > y1:
                mask[y1:y2, x1:x2] = 0

        # Morphology
        if USE_MORPH_OPEN:
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, MORPH_KERNEL)
        if USE_MORPH_CLOSE:
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, MORPH_KERNEL)

        # Contours → bboxes + centroids
        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        bboxes: List[Tuple[int, int, int, int]] = []
        centroids: List[Tuple[int, int]] = []
        dist_matches: List[float] = []

        for cnt in contours:
            if cv2.contourArea(cnt) >= min_area:
                x, y, w_box, h_box = cv2.boundingRect(cnt)
                bboxes.append((x, y, w_box, h_box))
                cx, cy = x + w_box // 2, y + h_box // 2
                centroids.append((cx, cy))

        # Distance to previous centroids (for diagnostics)
        for (cx, cy) in centroids:
            if not prev_centroids:
                best_dist = float("inf")
            else:
                best_dist = min(
                    math.hypot(cx - pcx, cy - pcy) for (pcx, pcy) in prev_centroids
                )
            dist_matches.append(best_dist)

        if bboxes:
            detections.append(
                {
                    "frame_index": frame_index,
                    "frame": frame if render_frames else None,
                    "bboxes": bboxes,
                    "centroids": centroids,
                    "dist_matches": dist_matches,
                }
            )

        prev_gray = gray
        prev_centroids = centroids
        frame_index += 1

    cap.release()
    return detections


# ---------------------------------------------------------------------------------
# STAGE 2–5: ORCHESTRATION - Track Building (build_tracks_standard)

# transforms raw frame‑level motion blobs into tracks, enriches them with 
# gap/density metadata, applies temporal gates, and optionally computes quality scores
# ---------------------------------------------------------------------------------

def run_step7_pipeline(
    video_path: str,
    diff_threshold: float,
    min_area: int,
    centroid_threshold: float,
) -> Tuple[List[Track], List[Track]]:
    
    """
    Execute the complete Step 7 pipeline with STANDARD temporal association.

    STAGES:
      1. Detection           → run_detection
      2. Track building      → build_tracks_standard
      3. Gap enrichment      → analyze_track_gaps
      4. Temporal gates      → apply_temporal_gates
      5. Quality scoring     → compute_* + score_track_quality
    """

    # Basic config sanity check for user visibility
    print("\n" + "=" * 80)
    print("▶️  RUNNING STEP 7 PIPELINE (STANDARD TEMPORAL ASSOCIATION)")
    print("=" * 80)
    print(f"Video path         : {video_path}")
    print(f"diff_threshold     : {diff_threshold}")
    print(f"min_area           : {min_area}")
    print(f"centroid_threshold : {centroid_threshold}")

    if "MAX_MISS" in globals():
        if "FRAME_WINDOW" in globals() and "BUFFER" in globals():
            print(
                f"Temporal window    : MAX_MISS = {MAX_MISS} "
                f"(FRAME_WINDOW={FRAME_WINDOW}, BUFFER={BUFFER})"
            )
        else:
            print(f"Temporal window    : MAX_MISS = {MAX_MISS}")
    else:
        print("Temporal window    : MAX_MISS not defined (check Step 7.2 config)")

    # ------------------------------------------------------------------
    # 1) DETECTION - # only raw detections.  (aka detection-stage filtering - min_area filters blobs, then centroid_threshold links blobs into tracks
    # ------------------------------------------------------------------
    print("\n[1/4] Detection (frame differencing + contours)...")
    detections = run_detection(video_path, diff_threshold, min_area, render_frames=False)
    print(f"      ✓ Motion frames with detections: {len(detections)}")

    if not detections:
        print("      ⚠ No detections found. Downstream stages will be empty.")
        return [], []

    # ------------------------------------------------------------------
    # 2) TRACK BUILDING
    # min_area filters blobs, then centroid_threshold links blobs into tracks
    # Output: A list (or dict) of Track objects with fields like frames, centroids, length, etc.
    # ------------------------------------------------------------------
    print("\n[2/4] Track building (build_tracks_standard)...")
    all_tracks = build_tracks_standard(
        detections, max_link_distance=centroid_threshold
    )
    print(f"      ✓ Tracks built: {len(all_tracks)}")

    if not all_tracks:
        print("      ⚠ No tracks built. Skipping further stages.")
        return [], []

    # ------------------------------------------------------------------
    # 3) GAP ENRICHMENT (Step 7.6)
    # Creates the following metadata:
    #  annotates each track with: gap_count, longest_gap, observation_density, span
    # all_tracks is not just raw MD frames — it’s raw tracks built from MD frames
    # ------------------------------------------------------------------
    if "analyze_track_gaps" in globals():
        print("\n[3/4] Enriching tracks with gap/density metadata (Step 7.6)...")
        all_tracks = analyze_track_gaps(all_tracks)
    else:
        print("\n[3/4] Gap enrichment skipped (analyze_track_gaps not defined).")

    # ------------------------------------------------------------------
    # 4) TEMPORAL GATES (Step 7.7)

#  filters tracks based on temporal and motion criteria.  Tracks that fail these gates are removed.
    # ------------------------------------------------------------------
    print("\n[4/4] Applying temporal/motion gates (Step 7.7)...")
    gated_tracks = apply_temporal_gates(all_tracks)
    print(f"      ✓ Tracks surviving gates: {len(gated_tracks)}")


    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # NEW
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # ------------------------------------------------------------------
    # STEP 7.9: PREP FOR DIAGNOSTICS – GAP STATS + ALL TRACKS SNAPSHOT
    # ------------------------------------------------------------------
    
    # 1) Build a list of Track objects for gap analysis.
    #    Adjust this depending on how you store tracks right now:
    #    - If you already have all_tracks as a list[Track], use it directly.
    #    - If you have a dict {track_id: Track}, convert values → list.
    try:
        if isinstance(all_tracks, dict):
            all_track_list = list(all_tracks.values())
        else:
            all_track_list = list(all_tracks)  # assume iterable of Track
    except NameError:
        # If your variable is named differently (e.g., tracks_by_id), fix here:
        all_track_list = list(tracks_by_id.values())
    
    # B) Run 7.6 gap/density analysis on this list.  (deprecated)
    #  all_track_list = analyze_track_gaps(all_track_list)
    
    # --- Step 7.6A: compute gap_stats (read-only, no logging) ---
    gap_stats = step76a_gap_metrics(all_track_list)   # returns list[dict]
    
    # --- Step 7.6B: optional logger snapshot (no globals dependency) ---
    if ENABLE_STEP76_LOG_SNAPSHOT:
        step76b_log_snapshot(gap_stats, log=log if "log" in globals() else None)

    
    # C) Build gap_stats for diagnostics (Step 7.D).
   
    # gap_stats = [
    # dict(
    #     track_id=t.track_id,
    #     created_frame=t.created_frame,
    #     last_observed_frame=t.last_observed_frame,
    #     span=t.span,
    #     length=t.length,
    #     gap_count=t.gap_count,
    #     longest_gap=t.longest_gap,
    #     observation_density=round(t.observation_density, 3),
    # )
    # for t in all_track_list
#  ]

    # D) Persist artifacts for Step 7.D (use dict keyed by track_id).
    #    Make sure STEP7_ARTIFACTS exists in the global scope.
    #  all_tracks -> This should be the single source of truth.
    
    g = globals()
    if "STEP7_ARTIFACTS" not in g:
        g["STEP7_ARTIFACTS"] = {}
    
    STEP7_ARTIFACTS = g["STEP7_ARTIFACTS"]
    STEP7_ARTIFACTS["all_tracks"] = {t.track_id: t for t in all_track_list}
    STEP7_ARTIFACTS["gap_stats"] = gap_stats
    
    print(f"\n[Step 7.9] Captured {len(all_track_list)} tracks for diagnostics.")
    print("[Step 7.9] Saved gap_stats and all_tracks into STEP7_ARTIFACTS.")
    


    # ------------------------------------------------------------------
    # 5) QUALITY SCORING (Step 7.8)
    # This stage computes quality metrics for each surviving track.
    # ranks tracks - identifies the best track, stores scores in STEP7_ARTIFACTS
    # ------------------------------------------------------------------

    if "USE_TRACK_QUALITY_SCORING" in globals() and USE_TRACK_QUALITY_SCORING:
        print("\n[+] Quality scoring enabled (Step 7.8)...")

        for t in gated_tracks:
            compute_track_heading_stats(t)
            compute_track_vertical_stats(t)
            score_track_quality(t)

        if gated_tracks:
            # Best by confidence_score if present, else by quality_score
            def _score_for_rank(t: Track) -> float:
                if hasattr(t, "confidence_score") and t.confidence_score is not None:
                    return t.confidence_score
                if hasattr(t, "quality_score") and t.quality_score is not None:
                    return t.quality_score
                return 0.0

            ranked = sorted(gated_tracks, key=_score_for_rank, reverse=True)
            best = ranked[0]
            print("\n🏆 Best gated track (by score):")
            print(f"   Track ID         : {best.track_id}")
            print(f"   Span frames      : {best.frames[0]} → {best.frames[-1]}")
            print(f"   Observed length  : {best.length}")
            print(f"   Gap count        : {getattr(best, 'gap_count', 'n/a')}")
            print(f"   Obs. density     : {getattr(best, 'observation_density', 'n/a')}")
            print(
                f"   quality_score    : {getattr(best, 'quality_score', 'n/a')}"
            )
            print(
                f"   confidence_score : {getattr(best, 'confidence_score', 'n/a')}"
            )
        else:
            print("      ⚠ No gated tracks to score.")
    else:
        print("\n[+] Quality scoring disabled (USE_TRACK_QUALITY_SCORING = False).")

    print("\n" + "=" * 80)
    print("✅ STEP 7.9 PIPELINE RUN COMPLETE")
    print("=" * 80)
    print(f"Total tracks built : {len(all_tracks)}")
    print(f"Tracks after gates : {len(gated_tracks)}")


    
    
    # ------------------------------------------------------------------
    # STEP 7.D – Capture artifacts for diagnostics (pre-gate snapshot)
    # captures diagnostic artifacts for debugging and analysis.
    # for later cells like 7.13 / 7.D / plot export)
    # ------------------------------------------------------------------
    try:
        # Ensure STEP7_ARTIFACTS exists in globals
        g = globals()
        if "STEP7_ARTIFACTS" not in g:
            g["STEP7_ARTIFACTS"] = {}
        artifacts = g["STEP7_ARTIFACTS"]

        # Make sure we have a list of Track objects
        try:
            all_track_list = list(all_tracks)
        except TypeError:
            # Fallback if all_tracks is a dict keyed by track_id
            all_track_list = list(all_tracks.values())

        # Build simple gap stats table for diagnostics
        gap_stats = []
        for t in all_track_list:
            gap_stats.append(
                dict(
                    track_id=t.track_id,
                    span=getattr(t, "span", None),
                    length=getattr(t, "length", None),
                    gap_count=getattr(t, "gap_count", None),
                    longest_gap=getattr(t, "longest_gap", None),
                    density=getattr(t, "observation_density", None),
                )
            )

        # Store into shared artifacts dict
        artifacts["all_tracks"] = {t.track_id: t for t in all_track_list}
        artifacts["gap_stats"] = gap_stats

        print(f"\n[Step 7.9] Captured {len(all_track_list)} tracks for diagnostics.")
        print("[Step 7.9] Saved gap_stats and all_tracks into STEP7_ARTIFACTS.")
    except Exception as e:
        print(f"\n⚠ STEP7_DIAG: Failed to capture diagnostic artifacts: {e}")

    return all_tracks, gated_tracks




# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# 
# Persist quality scores and best-track ID for diagnostics (if you have them)
    if "quality_scores" in locals():
        STEP7_ARTIFACTS["quality_scores"] = quality_scores
    
    STEP7_ARTIFACTS["best_track_id"] = best_track_id


# =================================================================================================
# OPTIONAL: AUTO-RUN STEP 7.9
# =================================================================================================

AUTO_RUN_STEP_7_9 = True  # ← flip to True when you want this cell to actually run

print("\n" + "=" * 70)
print("STEP 7.9: FULL PIPELINE EXECUTION (STANDARD APPROACH)")
print("=" * 70)

# Snapshot key config values for visibility
print("🔧 Current Step 7 Config Snapshot:")
if "VIDEO_PATH" in globals():
    print(f"   VIDEO_PATH          = {VIDEO_PATH}")
else:
    print("   VIDEO_PATH          = <not defined>")

for name in ["DIFF_THRESHOLD", "MIN_AREA", "CENTROID_THRESHOLD"]:
    if name in globals():
        print(f"   {name:18} = {globals()[name]}")
    else:
        print(f"   {name:18} = <not defined>")

if not AUTO_RUN_STEP_7_9:
    print(
        "\n✓ Step 7.9 pipeline functions defined (run_detection, run_step7_pipeline).\n"
        "  This cell is currently in DEFINE-ONLY mode.\n"
        "  To execute the full pipeline, either:\n"
        "    • Set AUTO_RUN_STEP_7_9 = True and re-run this cell, OR\n"
        "    • Call manually from another cell:\n"
        "        all_tracks, gated_tracks = run_step7_pipeline(\n"
        "            VIDEO_PATH, DIFF_THRESHOLD, MIN_AREA, CENTROID_THRESHOLD\n"
        "        )"
    )
else:
    # Make sure the required globals exist before calling
    missing = [
        name
        for name in ["VIDEO_PATH", "DIFF_THRESHOLD", "MIN_AREA", "CENTROID_THRESHOLD"]
        if name not in globals()
    ]
    if missing:
        print("\n❌ Cannot auto-run Step 7.9 – missing config variable(s):")
        for name in missing:
            print(f"   - {name}")
        print("   Define these in Step 7.2 and re-run.")
    else:
        all_tracks, gated_tracks = run_step7_pipeline(
            VIDEO_PATH, DIFF_THRESHOLD, MIN_AREA, CENTROID_THRESHOLD
        )



STEP 7.9: FULL PIPELINE EXECUTION (STANDARD APPROACH)
🔧 Current Step 7 Config Snapshot:
   VIDEO_PATH          = C:\AxisRecordings\Optical_Flow\videos\big_bird_R2L.mkv
   DIFF_THRESHOLD     = 2
   MIN_AREA           = 51
   CENTROID_THRESHOLD = 35

▶️  RUNNING STEP 7 PIPELINE (STANDARD TEMPORAL ASSOCIATION)
Video path         : C:\AxisRecordings\Optical_Flow\videos\big_bird_R2L.mkv
diff_threshold     : 2
min_area           : 51
centroid_threshold : 35
Temporal window    : MAX_MISS = 6 (FRAME_WINDOW=4, BUFFER=2)

[1/4] Detection (frame differencing + contours)...
      ✓ Motion frames with detections: 122

[2/4] Track building (build_tracks_standard)...

🔄 Building tracks with temporal window (MAX_MISS=6)...

📊 TRACK BUILDING COMPLETED
   ✓ Built 475 tracks total
      ✓ Tracks built: 475

[3/4] Gap enrichment skipped (analyze_track_gaps not defined).

[4/4] Applying temporal/motion gates (Step 7.7)...

📊 INCOMING TRACK GAP SUMMARY (before temporal gates):
   Total tracks: 475
   Track

In [23]:
# Is a track level score, NOT frame level data
t = gated_tracks[0]
t.score_meta



{'n_obs': 11,
 'len_norm': 0.275,
 'mean_heading_deg': 77.63074270833394,
 'expected_heading_band': (-45.0, 45.0),
 'heading_term': 0.3369935717925922,
 'dist_to_band': 32.63074270833394,
 'y_band_fraction': 1.0,
 'observation_density': 0.8461538461538461,
 'base_score': 0.4385980715377777,
 'quality_score': 0.4385980715377777,
 'confidence_score': 0.37112144514735035,
 'weights': {'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}}

In [24]:
# Sanity Check
print("have all_tracks?     ", "all_tracks" in globals())
print("have gated_tracks?   ", "gated_tracks" in globals())
print("have all_tracks_s?   ", "all_tracks_s" in globals())
print("have gated_tracks_s? ", "gated_tracks_s" in globals())
print("have STEP7_DF?       ", "STEP7_DF" in globals())


have all_tracks?      True
have gated_tracks?    True
have all_tracks_s?    False
have gated_tracks_s?  False
have STEP7_DF?        False


## <font color = yellow> 7.9S: FULL PIPELINE EXECUTION (SINGULAR-TRUTH + Create Single DF)


#### <font color = lime> The “earliest” data worth preserving (even earlier than all_tracks)

1. detections (output of run_detection)
  - Per motion-frame:
  - frame_index
  - bboxes
  - centroids
  - (optional) dist_matches

        This is the cleanest “what the detector saw” record

2. all_tracks (output of track builder)


          Tracks created from those detections using centroid-linking + temporal windowing

  

3. gated_tracks (output of Step 7.7)

          Tracks that survive your temporal density/length/etc gates.


In [25]:
# =================================================================================================
# STEP 7.9S: SINGULAR TRUTH + 3 DATAFRAMES (detections / pre-tracks / post-tracks)
# =================================================================================================
#
# WHY THIS CELL EXISTS
# --------------------
# Step 7.9 (full pipeline) is evolving and has multiple internal states.
# You asked for an alternate reality that DOES NOT butcher 7.9,
# but still produces a "singular truth" + a single, stable base dataframe
# (and in practice: 3 flavors of data that matter).
#
# This cell does NOT replace 7.9. It runs in parallel.
#
# It outputs three dataframes:
#
#   DF_DET   = raw detections (per-frame blobs: bbox + centroid)
#   DF_PRE   = all_tracks (pre-gate) as rows suitable for plotting / R / audits
#   DF_POST  = gated_tracks (post-gate) in the same row schema as DF_PRE
#
# It also returns:
#   all_tracks_list, gated_tracks
#
# IMPORTANT CONCEPTS
# ------------------
# 1) "Singular truth" = Track objects returned by build_tracks_standard() are the truth.
#    Dataframes are *views* for plotting + analysis + exporting.
#
# 2) Default is "dense-by-frame" meaning OBSERVED-only rows.
#    (You explicitly said: stop emphasizing gaps; emphasize observed MD frames.)
#
# 3) Optional: include full-span GAP rows if you are specifically trying to visualize continuity.
#    This does NOT mutate tracks. It only fills NaNs in the dataframe if you request it.
#
# 4) Optional: store DFs under STEP7_ARTIFACTS['df'] to avoid junk-drawer key sprawl.
#
# DEPENDENCIES
# ------------
# This assumes these already exist in your notebook (from earlier cells):
#   - run_detection(...)
#   - build_tracks_standard(...)
#   - apply_temporal_gates(...)
#   - Track dataclass with at least: track_id, frames, observations
# Optional if available:
#   - analyze_track_gaps(...)   (gap enrichment; safe if missing)
#   - scoring fns: compute_track_heading_stats, compute_track_vertical_stats, score_track_quality
#
# =================================================================================================

from typing import List, Dict, Any, Tuple, Optional
import os, math
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------------------------------
# OPTIONAL AUTO-RUN (mirrors the Step 7.9 pattern)
# -------------------------------------------------------------------------------------------------
# If True, this cell will immediately run the pipeline using VIDEO_PATH, DIFF_THRESHOLD, etc.
# If False, it only defines the functions and waits for you to call them manually.
AUTO_RUN_STEP_7_9S = True

# -------------------------------------------------------------------------------------------------
# USER TOGGLES
# -------------------------------------------------------------------------------------------------
# TRACK DF DEFAULT: "dense-by-frame" (OBSERVED ONLY).
# This matches your preference: "frames will only be the frames w/ MD".
DF_TRACK_INCLUDE_GAP_ROWS = False     # False = OBSERVED only (default). True = full span (OBSERVED + GAP)

# If you include GAP rows, you can optionally fill x/y for continuity plots:
#   "none"   = leave GAP rows as NaN
#   "ffill"  = forward-fill last observed x/y
#   "linear" = linear interpolation through gaps
# NOTE: Filling happens ONLY inside the dataframe (safe), not inside Track objects.
DF_TRACK_FILL_GAPS_MODE   = "none"    # "none" | "ffill" | "linear"

# DETECTIONS DF: include dist_match field or not (diagnostic only)
DF_DET_INCLUDE_DIST_MATCHES = True

# Optional capture to STEP7_ARTIFACTS["df"] (OFF by default to avoid junk drawer sprawl)
CAPTURE_TO_STEP7_ARTIFACTS_S = False


# -------------------------------------------------------------------------------------------------
# HELPER: normalize "all_tracks" into a real list[Track]
# -------------------------------------------------------------------------------------------------
def _as_track_list(all_tracks_any) -> List["Track"]:
    """
    Purpose:
      You have seen all_tracks show up as list-like or dict-like in different notebook eras.
      This function normalizes that into a stable list[Track], which becomes the singular truth.

    Inputs:
      all_tracks_any can be:
        - None
        - list[Track]
        - dict[track_id -> Track]
        - any iterable of Track

    Output:
      list[Track]
    """
    if all_tracks_any is None:
        return []
    if isinstance(all_tracks_any, dict):
        return list(all_tracks_any.values())
    try:
        return list(all_tracks_any)
    except TypeError:
        raise TypeError(f"Expected list/dict/iterable of Track, got: {type(all_tracks_any)}")


# -------------------------------------------------------------------------------------------------
# HELPER: safe preview display (won't crash outside Jupyter)
# -------------------------------------------------------------------------------------------------
def _safe_show_df(df: pd.DataFrame, n: int = 8, title: str = "") -> None:
    """
    Purpose:
      Show a quick peek in Jupyter using display(...).
      If running somewhere without display(), falls back to printing.

    Note:
      This is not required for logic; it’s pure ergonomics.
    """
    if title:
        print(title)
    if df is None or df.empty:
        print("  (empty)")
        return
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


# ================================================================================================
# DF #1: DETECTIONS → DF_DET
# ================================================================================================
# WHAT IT REPRESENTS:
#   One row per detected blob per frame.
#
# WHY IT MATTERS:
#   This is the earliest "real" data: contour/bbox/centroid after min_area and diff_threshold.
#   It lets you analyze:
#     - how many blobs per frame
#     - where centroids are
#     - bbox sizes and drift
#     - whether false positives cluster (waterline, noise zones, etc.)
#
# IMPORTANT:
#   DF_DET is PRE-tracking. It's just the raw MD detections.
# ================================================================================================
def build_df_detections(
    detections: List[Dict[str, Any]],
    include_dist_matches: bool = True
) -> pd.DataFrame:
    """
    Build canonical detections DF.

    Input detections list is produced by run_detection(...), where each element includes:
      - frame_index
      - bboxes:      list[(x,y,w,h)]
      - centroids:   list[(cx,cy)]
      - dist_matches list[float] (optional diagnostic distance to previous frame centroids)

    Output columns:
      frame, det_i, cx, cy, x, y, w, h, area, dist_match(optional)
    """
    rows: List[Dict[str, Any]] = []

    for det in detections or []:
        # Frame number where these detections occurred
        f = int(det.get("frame_index", -1))

        # Lists aligned by index:
        bboxes = det.get("bboxes", []) or []
        cents  = det.get("centroids", []) or []
        dists  = det.get("dist_matches", []) or []

        for i, bb in enumerate(bboxes):
            x, y, w, h = bb

            # Prefer centroid list if present; otherwise infer from bbox
            if i < len(cents):
                cx, cy = cents[i]
            else:
                cx, cy = (x + w / 2, y + h / 2)

            row = dict(
                frame=f,
                det_i=i,                      # which blob index inside this frame
                cx=float(cx),
                cy=float(cy),
                x=float(x),
                y=float(y),
                w=float(w),
                h=float(h),
                area=float(w * h),
            )

            # Optional diagnostic: nearest distance to previous frame's centroids
            if include_dist_matches:
                row["dist_match"] = float(dists[i]) if i < len(dists) else np.nan

            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Stable sort = makes downstream grouping deterministic
    return df.sort_values(["frame", "det_i"]).reset_index(drop=True)


# ================================================================================================
# DF #2 and DF #3: TRACKS → DF_PRE and DF_POST
# ================================================================================================
# WHAT IT REPRESENTS:
#   One row per track per frame (usually OBSERVED frames only, by default).
#
# WHY IT MATTERS:
#   Track objects are convenient for algorithms, but dataframes are easier for:
#     - plotting
#     - exporting
#     - R pipelines
#     - auditing
#
# IMPORTANT:
#   - Default is OBSERVED-only rows ("dense-by-frame"), matching your preference.
#   - Optional include_gap_rows=True yields full-span rows (OBSERVED + GAP).
#     In that mode, x/y are NaN for GAP rows unless you choose a fill.
# ================================================================================================
def build_df_tracks(
    tracks: List["Track"],
    gated_ids: Optional[set] = None,
    include_gap_rows: bool = False,
    fill_gaps_mode: str = "none",
) -> pd.DataFrame:
    """
    Build canonical tracks DF.

    Output columns:
      track_id, frame, status, x, y,
      is_gated, length, span, gap_count, density,
      confidence_score, quality_score,
      created_frame, last_observed_frame, last_updated_frame, status_track

    Notes:
      - Track-level fields are duplicated per-row. That is intentional.
        It makes grouping in R trivial (no joins required).
      - This function does not mutate Track objects.
    """
    gated_ids = gated_ids or set()
    rows: List[Dict[str, Any]] = []

    for t in tracks or []:
        frames_obs = sorted(getattr(t, "frames", []) or [])
        if not frames_obs:
            continue

        start_f = frames_obs[0]
        end_f   = frames_obs[-1]
        obs_set = set(frames_obs)

        # Track-level metadata is attached to every row for convenience
        meta = dict(
            track_id=int(getattr(t, "track_id", -1)),
            is_gated=(getattr(t, "track_id", -1) in gated_ids),
            length=int(getattr(t, "length", len(frames_obs))),
            span=int(getattr(t, "span", (end_f - start_f + 1))),
            gap_count=int(getattr(t, "gap_count", max(0, (end_f - start_f + 1) - len(frames_obs)))),
            density=float(getattr(t, "observation_density", (len(frames_obs) / (end_f - start_f + 1)))),
            confidence_score=float(getattr(t, "confidence_score", np.nan)) if getattr(t, "confidence_score", None) is not None else np.nan,
            quality_score=float(getattr(t, "quality_score", np.nan)) if getattr(t, "quality_score", None) is not None else np.nan,
            created_frame=int(getattr(t, "created_frame", start_f)),
            last_observed_frame=int(getattr(t, "last_observed_frame", end_f)),
            last_updated_frame=int(getattr(t, "last_updated_frame", end_f)),
            status_track=str(getattr(t, "status", "")),
        )

        # -----------------------------------------------------------------------------------------
        # MODE A: OBSERVED-only rows (DEFAULT; your "dense-by-frame" request)
        # -----------------------------------------------------------------------------------------
        if not include_gap_rows:
            for f in frames_obs:
                x, y = t.observations[f]   # observed centroid
                rows.append(dict(**meta, frame=int(f), status="OBSERVED", x=float(x), y=float(y)))
            continue

        # -----------------------------------------------------------------------------------------
        # MODE B: full span rows = OBSERVED + GAP
        # -----------------------------------------------------------------------------------------
        # GAP rows contain NaN x/y by default, and can be filled for continuity plots.
        for f in range(start_f, end_f + 1):
            if f in obs_set:
                x, y = t.observations[f]
                rows.append(dict(**meta, frame=int(f), status="OBSERVED", x=float(x), y=float(y)))
            else:
                rows.append(dict(**meta, frame=int(f), status="GAP", x=np.nan, y=np.nan))

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.sort_values(["track_id", "frame"]).reset_index(drop=True)

    # ---------------------------------------------------------------------------------------------
    # Optional gap filling (ONLY within the DF, only when include_gap_rows=True)
    # ---------------------------------------------------------------------------------------------
    fill_gaps_mode = (fill_gaps_mode or "none").lower().strip()
    if include_gap_rows and fill_gaps_mode in ("ffill", "linear"):

        def _fill_one(g: pd.DataFrame) -> pd.DataFrame:
            """
            Fill x/y for one track group.
            Used for continuity plots; does not change Track objects.
            """
            g = g.copy()
            if fill_gaps_mode == "ffill":
                g["x"] = g["x"].ffill()
                g["y"] = g["y"].ffill()
            else:  # "linear"
                g["x"] = g["x"].interpolate(limit_direction="both")
                g["y"] = g["y"].interpolate(limit_direction="both")
            return g

        df = df.groupby("track_id", group_keys=False).apply(_fill_one)

    return df


# ================================================================================================
# MAIN RUNNER: run_step7_pipeline_singular_dfs(...)
# ================================================================================================
# This mirrors the structure of Step 7.9 but:
#   - returns track list truth + 3 DFs
#   - keeps defaults aligned with your preferences
#   - does not touch STEP7_ARTIFACTS unless explicitly asked
# ================================================================================================
def run_step7_pipeline_singular_dfs(
    video_path: str,
    diff_threshold: float,
    min_area: int,
    centroid_threshold: float,
) -> Tuple[List["Track"], List["Track"], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Parallel pipeline runner:
      returns (all_tracks_list, gated_tracks, DF_DET, DF_PRE, DF_POST)
    """
    print("\n" + "=" * 80)
    print("▶️  RUNNING STEP 7 PIPELINE (7.9S — SINGULAR TRUTH + 3 DATAFRAMES)")
    print("=" * 80)
    print(f"Video path         : {video_path}")
    print(f"diff_threshold     : {diff_threshold}")
    print(f"min_area           : {min_area}")
    print(f"centroid_threshold : {centroid_threshold}")
    print(f"TRACK DF: include_gap_rows={DF_TRACK_INCLUDE_GAP_ROWS}, fill={DF_TRACK_FILL_GAPS_MODE!r}")

    # ---------------------------------------------------------------------------------------------
    # 1) DETECTION
    # ---------------------------------------------------------------------------------------------
    print("\n[1/4] Detection...")
    detections = run_detection(video_path, diff_threshold, min_area, render_frames=False)
    print(f"      ✓ Motion frames with detections: {len(detections)}")
    if not detections:
        # No detections means no tracks, no gates, no DFs
        return [], [], pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    # Build DF_DET immediately (raw MD data)
    DF_DET = build_df_detections(detections, include_dist_matches=DF_DET_INCLUDE_DIST_MATCHES)

    # ---------------------------------------------------------------------------------------------
    # 2) TRACK BUILDING
    # ---------------------------------------------------------------------------------------------
    print("\n[2/4] Track building...")
    all_tracks_any = build_tracks_standard(detections, max_link_distance=centroid_threshold)

    # Singular truth normalization
    all_tracks_list = _as_track_list(all_tracks_any)
    print(f"      ✓ Tracks built: {len(all_tracks_list)}")
    if not all_tracks_list:
        return [], [], DF_DET, pd.DataFrame(), pd.DataFrame()

    # ---------------------------------------------------------------------------------------------
    # 3) OPTIONAL GAP ENRICHMENT
    # ---------------------------------------------------------------------------------------------
    # This is safe. It should only derive gap metadata from frames/observations.
    # If analyze_track_gaps doesn't exist, we skip without failing the run.
    if "analyze_track_gaps" in globals():
        print("\n[3/4] Gap enrichment (analyze_track_gaps)...")
        try:
            all_tracks_list = analyze_track_gaps(all_tracks_list)
        except Exception as e:
            print(f"      ⚠ Gap enrichment failed (continuing): {e}")
    else:
        print("\n[3/4] Gap enrichment skipped (analyze_track_gaps not defined).")

    # ---------------------------------------------------------------------------------------------
    # 4) TEMPORAL GATES
    # ---------------------------------------------------------------------------------------------
    print("\n[4/4] Temporal gates...")
    gated_tracks = apply_temporal_gates(all_tracks_list)
    print(f"      ✓ Tracks surviving gates: {len(gated_tracks)}")

    # ---------------------------------------------------------------------------------------------
    # 5) QUALITY SCORING (optional)
    # ---------------------------------------------------------------------------------------------
    if "USE_TRACK_QUALITY_SCORING" in globals() and USE_TRACK_QUALITY_SCORING:
        print("\n[+] Quality scoring enabled...")
        for t in gated_tracks:
            compute_track_heading_stats(t)
            compute_track_vertical_stats(t)
            score_track_quality(t)

    # ---------------------------------------------------------------------------------------------
    # Build DF_PRE and DF_POST with identical schema
    # ---------------------------------------------------------------------------------------------
    gated_ids = set(t.track_id for t in (gated_tracks or []))

    DF_PRE = build_df_tracks(
        all_tracks_list,
        gated_ids=gated_ids,
        include_gap_rows=DF_TRACK_INCLUDE_GAP_ROWS,
        fill_gaps_mode=DF_TRACK_FILL_GAPS_MODE,
    )

    DF_POST = build_df_tracks(
        gated_tracks,
        gated_ids=gated_ids,
        include_gap_rows=DF_TRACK_INCLUDE_GAP_ROWS,
        fill_gaps_mode=DF_TRACK_FILL_GAPS_MODE,
    )

    # ---------------------------------------------------------------------------------------------
    # Summary + previews
    # ---------------------------------------------------------------------------------------------
    print("\n" + "-" * 80)
    print("📦 DataFrames created")
    print(f"   DF_DET  rows={len(DF_DET):,}   (detections/blobs)")
    print(f"   DF_PRE  rows={len(DF_PRE):,}   (all tracks, pre-gate)")
    print(f"   DF_POST rows={len(DF_POST):,}  (gated tracks, post-gate)")
    print("-" * 80)

    _safe_show_df(DF_DET,  6, "\nDF_DET preview:")
    _safe_show_df(DF_PRE,  6, "\nDF_PRE preview:")
    _safe_show_df(DF_POST, 6, "\nDF_POST preview:")

    # ---------------------------------------------------------------------------------------------
    # Optional artifact capture under a single scoped namespace
    # ---------------------------------------------------------------------------------------------
    # This avoids STEP7_ARTIFACTS becoming a junk drawer:
    #   STEP7_ARTIFACTS['df']['det'], ['pre'], ['post']
    if CAPTURE_TO_STEP7_ARTIFACTS_S:
        g = globals()
        if "STEP7_ARTIFACTS" not in g:
            g["STEP7_ARTIFACTS"] = {}
        if "df" not in g["STEP7_ARTIFACTS"]:
            g["STEP7_ARTIFACTS"]["df"] = {}
        g["STEP7_ARTIFACTS"]["df"]["det"]  = DF_DET
        g["STEP7_ARTIFACTS"]["df"]["pre"]  = DF_PRE
        g["STEP7_ARTIFACTS"]["df"]["post"] = DF_POST
        print("✅ (Optional) STEP7_ARTIFACTS['df'] updated: det / pre / post")

    return all_tracks_list, gated_tracks, DF_DET, DF_PRE, DF_POST


# -------------------------------------------------------------------------------------------------
# CELL HEADER OUTPUT (so you can confirm the cell is loaded and configured correctly)
# -------------------------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 7.9S: SINGULAR TRUTH + 3 DATAFRAMES")
print("=" * 70)
print("✓ Function defined: run_step7_pipeline_singular_dfs(...) -> (all_tracks_list, gated_tracks, DF_DET, DF_PRE, DF_POST)")
print(f"   TRACK_INCLUDE_GAP_ROWS = {DF_TRACK_INCLUDE_GAP_ROWS}")
print(f"   TRACK_FILL_GAPS_MODE   = {DF_TRACK_FILL_GAPS_MODE!r}")
print(f"   CAPTURE_TO_STEP7_ARTIFACTS_S = {CAPTURE_TO_STEP7_ARTIFACTS_S}")

# -------------------------------------------------------------------------------------------------
# OPTIONAL AUTO-RUN
# -------------------------------------------------------------------------------------------------
if AUTO_RUN_STEP_7_9S:
    missing = [
        name for name in ["VIDEO_PATH", "DIFF_THRESHOLD", "MIN_AREA", "CENTROID_THRESHOLD"]
        if name not in globals()
    ]
    if missing:
        print("\n❌ Cannot auto-run Step 7.9S – missing config variable(s):", missing)
    else:
        all_tracks_s, gated_tracks_s, DF_DET, DF_PRE, DF_POST = run_step7_pipeline_singular_dfs(
            VIDEO_PATH, DIFF_THRESHOLD, MIN_AREA, CENTROID_THRESHOLD
        )



STEP 7.9S: SINGULAR TRUTH + 3 DATAFRAMES
✓ Function defined: run_step7_pipeline_singular_dfs(...) -> (all_tracks_list, gated_tracks, DF_DET, DF_PRE, DF_POST)
   TRACK_INCLUDE_GAP_ROWS = False
   TRACK_FILL_GAPS_MODE   = 'none'
   CAPTURE_TO_STEP7_ARTIFACTS_S = False

▶️  RUNNING STEP 7 PIPELINE (7.9S — SINGULAR TRUTH + 3 DATAFRAMES)
Video path         : C:\AxisRecordings\Optical_Flow\videos\big_bird_R2L.mkv
diff_threshold     : 2
min_area           : 51
centroid_threshold : 35
TRACK DF: include_gap_rows=False, fill='none'

[1/4] Detection...
      ✓ Motion frames with detections: 122

[2/4] Track building...

🔄 Building tracks with temporal window (MAX_MISS=6)...

📊 TRACK BUILDING COMPLETED
   ✓ Built 475 tracks total
      ✓ Tracks built: 475

[3/4] Gap enrichment skipped (analyze_track_gaps not defined).

[4/4] Temporal gates...

📊 INCOMING TRACK GAP SUMMARY (before temporal gates):
   Total tracks: 475
   Tracks with gaps: 216

   Sample gap details (first 5 tracks with gaps):

   

,frame,det_i,cx,cy,x,y,w,h,area,dist_match
0,5,0,1799.0,232.0,1793.0,227.0,13.0,11.0,143.0,inf
1,11,0,1907.0,176.0,1895.0,167.0,25.0,19.0,475.0,inf
2,12,0,1904.0,178.0,1889.0,165.0,31.0,27.0,837.0,3.605551
3,13,0,1903.0,178.0,1887.0,166.0,33.0,24.0,792.0,1.000000
4,14,0,1903.0,177.0,1887.0,161.0,33.0,32.0,1056.0,1.000000
5,15,0,1903.0,180.0,1887.0,167.0,33.0,27.0,891.0,3.000000



DF_PRE preview:


,track_id,is_gated,length,span,gap_count,density,confidence_score,quality_score,created_frame,last_observed_frame,last_updated_frame,status_track,frame,status,x,y
0,0,False,3,19,16,0.157895,0.0,0.0,5,23,30,too_short,5,OBSERVED,1799.0,232.0
1,0,False,3,19,16,0.157895,0.0,0.0,5,23,30,too_short,16,OBSERVED,1894.0,178.0
2,0,False,3,19,16,0.157895,0.0,0.0,5,23,30,too_short,23,OBSERVED,1913.0,215.0
3,1,False,14,36,22,0.388889,0.0,0.0,11,46,53,low_density,11,OBSERVED,1907.0,176.0
4,1,False,14,36,22,0.388889,0.0,0.0,11,46,53,low_density,12,OBSERVED,1904.0,178.0
5,1,False,14,36,22,0.388889,0.0,0.0,11,46,53,low_density,13,OBSERVED,1903.0,178.0



DF_POST preview:


,track_id,is_gated,length,span,gap_count,density,confidence_score,quality_score,created_frame,last_observed_frame,last_updated_frame,status_track,frame,status,x,y
0,3,True,11,13,2,0.846154,0.371121,0.438598,19,31,38,confirmed,19,OBSERVED,1819.0,171.0
1,3,True,11,13,2,0.846154,0.371121,0.438598,19,31,38,confirmed,20,OBSERVED,1807.0,168.0
2,3,True,11,13,2,0.846154,0.371121,0.438598,19,31,38,confirmed,21,OBSERVED,1802.0,164.0
3,3,True,11,13,2,0.846154,0.371121,0.438598,19,31,38,confirmed,22,OBSERVED,1801.0,164.0
4,3,True,11,13,2,0.846154,0.371121,0.438598,19,31,38,confirmed,23,OBSERVED,1785.0,175.0
5,3,True,11,13,2,0.846154,0.371121,0.438598,19,31,38,confirmed,24,OBSERVED,1764.0,175.0


In [26]:
# Sanity Check
print("have all_tracks?     ", "all_tracks" in globals())
print("have gated_tracks?   ", "gated_tracks" in globals())
print("have all_tracks_s?   ", "all_tracks_s" in globals())
print("have gated_tracks_s? ", "gated_tracks_s" in globals())
print("have STEP7_DF?       ", "STEP7_DF" in globals())


have all_tracks?      True
have gated_tracks?    True
have all_tracks_s?    True
have gated_tracks_s?  True
have STEP7_DF?        False


## <font color = yellow> 7.9Review: Explore the df's before save to file -  (df.to_csv)

In [27]:
# explore python
DF_PRE.head(3)

,track_id,is_gated,length,span,gap_count,density,confidence_score,quality_score,created_frame,last_observed_frame,last_updated_frame,status_track,frame,status,x,y
0,0,False,3,19,16,0.157895,0.0,0.0,5,23,30,too_short,5,OBSERVED,1799.0,232.0
1,0,False,3,19,16,0.157895,0.0,0.0,5,23,30,too_short,16,OBSERVED,1894.0,178.0
2,0,False,3,19,16,0.157895,0.0,0.0,5,23,30,too_short,23,OBSERVED,1913.0,215.0


In [28]:
# explore python
DF_PRE.info

<bound method DataFrame.info of       track_id  is_gated  length  span  gap_count   density  confidence_score  \
0            0     False       3    19         16  0.157895               0.0   
1            0     False       3    19         16  0.157895               0.0   
2            0     False       3    19         16  0.157895               0.0   
3            1     False      14    36         22  0.388889               0.0   
4            1     False      14    36         22  0.388889               0.0   
...        ...       ...     ...   ...        ...       ...               ...   
1234       470     False       1     1          0  1.000000               0.0   
1235       471     False       1     1          0  1.000000               0.0   
1236       472     False       1     1          0  1.000000               0.0   
1237       473     False       1     1          0  1.000000               0.0   
1238       474     False       1     1          0  1.000000               0.0

In [29]:
# explore python
DF_POST.dtypes

track_id                 int64
is_gated                  bool
length                   int64
span                     int64
gap_count                int64
density                float64
confidence_score       float64
quality_score          float64
created_frame            int64
last_observed_frame      int64
last_updated_frame       int64
status_track            object
frame                    int64
status                  object
x                      float64
y                      float64
dtype: object

## <font color = yellow> 7.18_R: Create/save df files (3) to file (for use w/ R)

In [30]:
# 7.18R: Create/save d files

# DF_DET.to_csv("raw_temporal_data_noTracks.csv", index =False)
# DF_PRE.to_csv("basic_tracks_from_raw_temporal.csv", index=False)
# DF_POST.to_csv("tracks_gated_filtered.csv", index=False)  #  gated but unranked tracks


DF_DET.to_csv(os.path.join(str(LOG_DIR), "raw_temporal_data_noTracks.csv"), index=False)
DF_PRE.to_csv(os.path.join(str(LOG_DIR), "basic_tracks_from_raw_temporal.csv"), index=False)
DF_POST.to_csv(os.path.join(str(LOG_DIR), "tracks_gated_filtered.csv"), index=False)      #  gated but unranked tracks


In [31]:
DF_POST.info

<bound method DataFrame.info of      track_id  is_gated  length  span  gap_count   density  confidence_score  \
0           3      True      11    13          2  0.846154          0.371121   
1           3      True      11    13          2  0.846154          0.371121   
2           3      True      11    13          2  0.846154          0.371121   
3           3      True      11    13          2  0.846154          0.371121   
4           3      True      11    13          2  0.846154          0.371121   
..        ...       ...     ...   ...        ...       ...               ...   
119        48      True      10    16          6  0.625000          0.390625   
120        48      True      10    16          6  0.625000          0.390625   
121        48      True      10    16          6  0.625000          0.390625   
122        48      True      10    16          6  0.625000          0.390625   
123        48      True      10    16          6  0.625000          0.390625   

     qu

In [32]:
import pandas as pd

# def export_track_score_meta(tracks, path="track_score_meta.csv"):
#     rows = []
#     for t in tracks:
#         if not hasattr(t, "score_meta"):
#             continue
#         row = dict(track_id=t.track_id)
#         row.update(t.score_meta)
#         rows.append(row)

#     df = pd.DataFrame(rows)
#     df.to_csv(path, index=False)
#     print(f"✓ Track score meta saved → {path}")
#     return df


def export_track_score_meta(tracks, path=None):
    # Resolve output path
    if path is None:
        if "LOG_DIR" in globals():
            out_path = os.path.join(str(LOG_DIR), "track_score_meta.csv")
        else:
            out_path = "track_score_meta.csv"  # fallback to CWD
    else:
        out_path = path

    rows = []
    for t in tracks:
        if not hasattr(t, "score_meta"):
            continue
        row = dict(track_id=t.track_id)
        row.update(t.score_meta)
        rows.append(row)

    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"✓ Track score meta saved → {out_path}")
    return df

# fct call
export_track_score_meta(gated_tracks, path=None)





✓ Track score meta saved → C:\Axis_code_projects\OF_vs_Step7\outputs\Step7_full\step7_big_bird_R2L_20260101_183730\logs\track_score_meta.csv


,track_id,n_obs,len_norm,mean_heading_deg,expected_heading_band,heading_term,dist_to_band,y_band_fraction,observation_density,base_score,quality_score,confidence_score,weights
0,3,11,0.275,77.630743,"(-45.0, 45.0)",0.336994,32.630743,1.0,0.846154,0.438598,0.438598,0.371121,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"
1,48,10,0.250,44.266838,"(-45.0, 45.0)",1.000000,0.000000,1.0,0.625000,0.625000,0.625000,0.390625,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"
2,7,13,0.325,28.443420,"(-45.0, 45.0)",1.000000,0.000000,1.0,0.520000,0.662500,0.662500,0.344500,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"
3,9,90,1.000,45.383792,"(-45.0, 45.0)",0.987288,0.383792,1.0,0.857143,0.996187,0.996187,0.853874,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"


## <font color = yellow> 7.18_Rank: Surface the track rankings

In [33]:
# 7.18_Rank: Surface the track rankings

def rank_tracks(tracks, top=15, by="confidence"):
    def key(t):
        if by == "quality":
            return float(getattr(t, "quality_score", 0.0) or 0.0)
        return float(getattr(t, "confidence_score", 0.0) or 0.0)

    ranked = sorted(tracks, key=key, reverse=True)

    print(f"\n🏁 RANKED TRACKS (by {by}_score)  top={min(top, len(ranked))}/{len(ranked)}")
    print("rank | track_id | conf     | quality   | len | span | gaps | dens   | mean_heading | y_band")
    print("-----+----------+----------+----------+-----+------+-----+--------+-------------+-------")

    for i, t in enumerate(ranked[:top], start=1):
        conf = getattr(t, "confidence_score", None)
        qual = getattr(t, "quality_score", None)
        mh   = getattr(t, "mean_heading_deg", None)
        ybf  = getattr(t, "y_band_fraction", None)

        print(
            f"{i:4d} | {t.track_id:8d} | "
            f"{(conf if conf is not None else 0.0):8.3f} | "
            f"{(qual if qual is not None else 0.0):8.3f} | "
            f"{getattr(t,'length',0):3d} | {getattr(t,'span',0):4d} | "
            f"{getattr(t,'gap_count',0):3d} | "
            f"{(getattr(t,'observation_density',0.0) or 0.0):6.3f} | "
            f"{(mh if mh is not None else float('nan')):11.1f} | "
            f"{(ybf if ybf is not None else 0.0):5.2f}"
        )

    return ranked


In [34]:
# ranked = rank_tracks(gated_tracks, top=20, by="confidence")   # main ranking
# # or:
# ranked_q = rank_tracks(gated_tracks, top=20, by="quality")


# export_track_score_meta(ranked_tracks, path=None)


ranked = rank_tracks(gated_tracks, top=20, by="confidence")
export_track_score_meta(ranked, path=os.path.join(str(LOG_DIR), "track_score_meta_confidence.csv"))

ranked_q = rank_tracks(gated_tracks, top=20, by="quality")
export_track_score_meta(ranked_q, path=os.path.join(str(LOG_DIR), "track_score_meta_quality.csv"))



🏁 RANKED TRACKS (by confidence_score)  top=4/4
rank | track_id | conf     | quality   | len | span | gaps | dens   | mean_heading | y_band
-----+----------+----------+----------+-----+------+-----+--------+-------------+-------
   1 |        9 |    0.854 |    0.996 |  90 |  105 |  15 |  0.857 |        45.4 |  1.00
   2 |       48 |    0.391 |    0.625 |  10 |   16 |   6 |  0.625 |        44.3 |  1.00
   3 |        3 |    0.371 |    0.439 |  11 |   13 |   2 |  0.846 |        77.6 |  1.00
   4 |        7 |    0.345 |    0.663 |  13 |   25 |  12 |  0.520 |        28.4 |  1.00
✓ Track score meta saved → C:\Axis_code_projects\OF_vs_Step7\outputs\Step7_full\step7_big_bird_R2L_20260101_183730\logs\track_score_meta_confidence.csv

🏁 RANKED TRACKS (by quality_score)  top=4/4
rank | track_id | conf     | quality   | len | span | gaps | dens   | mean_heading | y_band
-----+----------+----------+----------+-----+------+-----+--------+-------------+-------
   1 |        9 |    0.854 |    0.996 |  

,track_id,n_obs,len_norm,mean_heading_deg,expected_heading_band,heading_term,dist_to_band,y_band_fraction,observation_density,base_score,quality_score,confidence_score,weights
0,9,90,1.000,45.383792,"(-45.0, 45.0)",0.987288,0.383792,1.0,0.857143,0.996187,0.996187,0.853874,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"
1,7,13,0.325,28.443420,"(-45.0, 45.0)",1.000000,0.000000,1.0,0.520000,0.662500,0.662500,0.344500,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"
2,48,10,0.250,44.266838,"(-45.0, 45.0)",1.000000,0.000000,1.0,0.625000,0.625000,0.625000,0.390625,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"
3,3,11,0.275,77.630743,"(-45.0, 45.0)",0.336994,32.630743,1.0,0.846154,0.438598,0.438598,0.371121,"{'w_len': 0.5, 'w_heading': 0.3, 'w_y': 0.2}"


## <font color = yellow> 7.18G: display globals

In [35]:
# test settings


print("DIFF_THRESHOLD" in globals())
print("MIN_AREA" in globals())
print("CENTROID_THRESHOLD" in globals())

try:
    print("DIFF =", DIFF_THRESHOLD)
except:
    print("DIFF undefined")

try:
    print("AREA =", MIN_AREA)
except:
    print("MIN_AREA undefined")

try:
    print("CENTROID =", CENTROID_THRESHOLD)
except:
    print("CENTROID_THRESHOLD undefined")


True
True
True
DIFF = 2
AREA = 51
CENTROID = 35


In [36]:
# import inspect
# print(inspect.getsource(Track.__init__))

In [37]:
me_note

' log.silent(f"\\n=== Default Config Changes ===\\n{cfg_text}\\n==========GS============",  note=" template ") '

### Save to log

In [38]:
# best_raw_record = {
#     "stage": "7.9_best_track_raw",
#     "video": VIDEO_PATH,
#     "config_changes_summary": config_changes["summary"],
#     "config_changes": config_changes["changes"],  # optional
#     "diff_threshold": diff_threshold,
#     "min_area": min_area,
#     "centroid_threshold": centroid_threshold,
#     "FRAME_WINDOW": FRAME_WINDOW,
#     "BUFFER": BUFFER,
#     "MAX_MISS": MAX_MISS,
#     #"track_id": best_track.track_id,
#     # "span": best_track.span,
#     "length": best_track.length,
#     "observation_density": best_track.observation_density,
#     "gap_count": best_track.gap_count,
#     "longest_gap": best_track.longest_gap,
#     "quality_score": best_track.quality_score,
#     "confidence_score": best_track.confidence_score,
# }


In [39]:
# __init__ is the constructor: it defines what data every Track object starts with when created.

### <font color = lime> The _ _init_ _ is the constructor:

Set to define the parameters for the frame data & for Data preservation.  Defines what data every Track object starts with when created.  Defines the identity and temporal state of the track.  Use: downstream code can safely loop over all_tracks and compute gaps, velocities, or scores without checking if fields exist.

The contructor s/align with the class attributes. That way, downstream pipeline stages don’t have to check if a field exists — they know it’s there.  No AttributeErrors -  when code tried to access Variables That Do Not Exist.

Also being used to preserve intermediate attributes as the pipeline advances.

Preserves intermediates such as: 
- gaps, 
- displacements, 
- velocities)

### <font color = lime> How to use the python inspect library

In [40]:
# Example 
import inspect

def my_function(x, y=10):
    return x + y

print(inspect.getsource(my_function))


def my_function(x, y=10):
    return x + y



### <font color = lime> How to inspect the fct: Track.__init__

In [41]:
# import inspect
# print(inspect.getsource(Track.__init__))


In [42]:
# Example in using inspect.signature()
import inspect

# Get the signature of the constructor
sig = inspect.signature(Track.__init__)

# Print it directly
print(sig)

# If you want to loop through parameters
for name, param in sig.parameters.items():
    print(f"{name}: default={param.default}")


(self, track_id: int, created_frame: int, last_observed_frame: int, last_updated_frame: int, miss_count: int = 0, status: str = 'candidate', observations: Dict[int, Tuple[float, float]] = <factory>, bboxes: Dict[int, Tuple[int, int, int, int]] = <factory>, gap_frames: Set[int] = <factory>, step_displacements: List[float] = <factory>, step_headings: List[float] = <factory>, disp_min: Optional[float] = None, disp_max: Optional[float] = None, disp_mean: Optional[float] = None, disp_median: Optional[float] = None, heading_mean: Optional[float] = None, heading_std: Optional[float] = None, linearity_ratio: Optional[float] = None, mean_heading_deg: Optional[float] = None, std_heading_deg: Optional[float] = None, mean_y: Optional[float] = None, min_y: Optional[float] = None, max_y: Optional[float] = None, y_band_fraction: float = 0.0, quality_score: float = 0.0, confidence_score: float = 0.0, interpolated_frames: Set[int] = <factory>, original_observation_density: Optional[float] = None, was_i

In [43]:
# pip install --upgrade rpy2 pandas

In [44]:
import pandas as pd

# First, let's explore what data we can extract from a track
sample = all_tracks[0]
print(f"Track ID: {sample.track_id}")
print(f"Frames: {sample.frames}")
print(f"Length: {sample.length}")
print(f"Quality Score: {sample.quality_score}")

# Convert all_tracks to a DataFrame
# Extract key attributes from each track

Track ID: 0
Frames: [5, 16, 23]
Length: 3
Quality Score: 0.0


In [45]:
# tracks_data = []
# for track in all_tracks:
#     track_dict = {
#         'track_id': track.track_id,
#         'length': track.length,
#         'span': track.span,
#         'quality_score': track.quality_score,
#         'confidence_score': track.confidence_score,
#         'created_frame': track.created_frame,
#         'last_updated_frame': track.last_updated_frame,
#         'good_frame_count': track.good_frame_count,
#         'bad_frame_count': track.bad_frame_count,
#         'gap_count': track.gap_count,
#         'observation_density': track.observation_density,
#         'linearity_ratio': track.linearity_ratio,
#         'mean_y': track.mean_y,
#         'min_y': track.min_y,
#         'max_y': track.max_y,
#         'y_band_fraction': track.y_band_fraction,
#         'status': track.status,
#         'promoted': track.promoted,
#         'demoted': track.demoted,
#         # Add more attributes as needed
#     }
#     tracks_data.append(track_dict)

# df_all = pd.DataFrame(tracks_data)
# # Save DataFrame to CSV
# df_all.to_csv('all_tracks.csv', index=False)

### <font color = lime> Explore all_tracks 

#### <font color = yellow> Noticing is a natural consequence of how the temporal association step builds tracks:

    Track 0 has a very long run of linked detections — it starts at frame 1 and accumulates observations all the way through frame 109. That’s why its observations and bboxes dictionaries are huge: every frame where motion was detected and successfully linked into the same track contributes another entry.
    
    Track 1 is shorter and only managed to link two detections (frames 7 and 17). Its observations and bboxes are correspondingly tiny.

#### Remember: this is preliminary (all_track) data before next (better) gap re-do w/

In [46]:
import pandas as pd

# Convert all_tracks into a list of dictionaries
track_dicts = [vars(track) for track in all_tracks]

# Create DataFrame
df_track_dicts = pd.DataFrame(track_dicts)

# Export to CSV
df_track_dicts.to_csv(os.path.join(str(LOG_DIR), "all_tracks_output.csv"), index=False)



In [47]:
# Look at the first track
first_track = all_tracks[0]

# See its attributes
print(vars(first_track))


{'track_id': 0, 'created_frame': 5, 'last_observed_frame': 23, 'last_updated_frame': 30, 'miss_count': 7, 'status': 'too_short', 'observations': {5: (1799, 232), 16: (1894, 178), 23: (1913, 215)}, 'bboxes': {5: (1793, 227, 13, 11), 16: (1887, 173, 14, 11), 23: (1906, 208, 14, 14)}, 'gap_frames': {11, 12, 13, 14, 15, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30}, 'step_displacements': [], 'step_headings': [], 'disp_min': None, 'disp_max': None, 'disp_mean': None, 'disp_median': None, 'heading_mean': None, 'heading_std': None, 'linearity_ratio': None, 'mean_heading_deg': None, 'std_heading_deg': None, 'mean_y': None, 'min_y': None, 'max_y': None, 'y_band_fraction': 0.0, 'quality_score': 0.0, 'confidence_score': 0.0, 'interpolated_frames': set(), 'original_observation_density': None, 'was_interpolated': False}


In [48]:
print(list(vars(first_track).keys()))

['track_id', 'created_frame', 'last_observed_frame', 'last_updated_frame', 'miss_count', 'status', 'observations', 'bboxes', 'gap_frames', 'step_displacements', 'step_headings', 'disp_min', 'disp_max', 'disp_mean', 'disp_median', 'heading_mean', 'heading_std', 'linearity_ratio', 'mean_heading_deg', 'std_heading_deg', 'mean_y', 'min_y', 'max_y', 'y_band_fraction', 'quality_score', 'confidence_score', 'interpolated_frames', 'original_observation_density', 'was_interpolated']


In [49]:
# Look at the second track
second_track = all_tracks[1]

# See its attributes
print(vars(second_track))

{'track_id': 10, 'created_frame': 23, 'last_observed_frame': 23, 'last_updated_frame': 30, 'miss_count': 7, 'status': 'too_short', 'observations': {23: (616, 136)}, 'bboxes': {23: (608, 130, 16, 13)}, 'gap_frames': {24, 25, 26, 27, 28, 29, 30}, 'step_displacements': [], 'step_headings': [], 'disp_min': None, 'disp_max': None, 'disp_mean': None, 'disp_median': None, 'heading_mean': None, 'heading_std': None, 'linearity_ratio': None, 'mean_heading_deg': None, 'std_heading_deg': None, 'mean_y': None, 'min_y': None, 'max_y': None, 'y_band_fraction': 0.0, 'quality_score': 0.0, 'confidence_score': 0.0, 'interpolated_frames': set(), 'original_observation_density': None, 'was_interpolated': False}


In [50]:
for i, track in enumerate(all_tracks[0:5]):
    print(f"Track {i} attributes:")
    for key, value in vars(track).items():
        print(f"  {key}: {value}")


Track 0 attributes:
  track_id: 0
  created_frame: 5
  last_observed_frame: 23
  last_updated_frame: 30
  miss_count: 7
  status: too_short
  observations: {5: (1799, 232), 16: (1894, 178), 23: (1913, 215)}
  bboxes: {5: (1793, 227, 13, 11), 16: (1887, 173, 14, 11), 23: (1906, 208, 14, 14)}
  gap_frames: {11, 12, 13, 14, 15, 19, 20, 21, 22, 24, 25, 26, 27, 28, 29, 30}
  step_displacements: []
  step_headings: []
  disp_min: None
  disp_max: None
  disp_mean: None
  disp_median: None
  heading_mean: None
  heading_std: None
  linearity_ratio: None
  mean_heading_deg: None
  std_heading_deg: None
  mean_y: None
  min_y: None
  max_y: None
  y_band_fraction: 0.0
  quality_score: 0.0
  confidence_score: 0.0
  interpolated_frames: set()
  original_observation_density: None
  was_interpolated: False
Track 1 attributes:
  track_id: 10
  created_frame: 23
  last_observed_frame: 23
  last_updated_frame: 30
  miss_count: 7
  status: too_short
  observations: {23: (616, 136)}
  bboxes: {23: (608,

In [51]:
# all_tracks

In [52]:
%load_ext rpy2.ipython


C:\Users\prior\AppData\Local\Programs\Python\Python39\lib\site-packages\rpy2\robjects\packages.py:366: UserWarning: The symbol 'quartz' is not in this R namespace/package.
  warnings.warn(


## <font color = yellow> 7.10: RETRO-INTERPOLATION DIAGNOSTIC (GAP FILL FOR BEST TRACK)

In [54]:
# ======================================================================
# STEP 7.10X – EXPORT STANDARD COMPARISON PACKETS (BASELINE + RETRO)
# Paste at END of Step 7.10 cell, after the BEFORE/AFTER/Δ/NOTE prints.
# Exports from the Step7 gap snapshot dict (no Track objects required).
# ======================================================================

from typing import Dict, Set, Tuple, Optional, Any
from pathlib import Path
import csv, json, os
from datetime import datetime

# ---------------------------
# 0) Required run context
# ---------------------------
if "RUN_DIR" not in globals():
    raise NameError("RUN_DIR is not defined. Run the cell that creates RUN_DIR/LOG_DIR/IMG_DIR/VID_DIR first.")
if "VIDEO_PATH" not in globals():
    raise NameError("VIDEO_PATH is not defined. Run the cell that defines VIDEO_PATH first.")

_run_dir = Path(RUN_DIR)
_video_path = str(VIDEO_PATH)

# ---------------------------
# 1) Find the gap snapshot dict from Step 7.10
#    Prefer ORIGINAL_BEST_TRACK_GAPS if present; otherwise auto-detect.
# ---------------------------
def _find_gap_snapshot() -> Tuple[str, Dict[str, Any]]:
    if "ORIGINAL_BEST_TRACK_GAPS" in globals() and isinstance(ORIGINAL_BEST_TRACK_GAPS, dict):
        return "ORIGINAL_BEST_TRACK_GAPS", ORIGINAL_BEST_TRACK_GAPS

    # Auto-detect: dict with required keys
    need = {"frames_span", "gap_frames", "length", "observation_density"}
    for k, v in globals().items():
        if isinstance(v, dict) and need.issubset(set(v.keys())):
            return k, v

    raise NameError(
        "Could not find a Step7 gap snapshot dict.\n"
        "Expected ORIGINAL_BEST_TRACK_GAPS (dict) OR any dict containing keys:\n"
        "  frames_span, gap_frames, length, observation_density\n"
        "Run Step 7.10 and ensure it saves that snapshot to a variable."
    )

_gap_name, _gap = _find_gap_snapshot()

# Pull baseline span + gaps from the snapshot
if not (isinstance(_gap.get("frames_span"), (tuple, list)) and len(_gap["frames_span"]) == 2):
    raise NameError(f"{_gap_name} missing/invalid frames_span: {_gap.get('frames_span')}")

_span_start, _span_end = int(_gap["frames_span"][0]), int(_gap["frames_span"][1])
_gap_frames = set(int(x) for x in (_gap.get("gap_frames") or []))

if _span_end < _span_start:
    raise NameError(f"Bad frames_span in {_gap_name}: {_gap['frames_span']}")

# Build md frame sets:
# Baseline (pre-retro): all frames in span minus gaps
baseline_md: Set[int] = set(range(_span_start, _span_end + 1)) - _gap_frames
baseline_span: Tuple[int, int] = (_span_start, _span_end)

# Retro (post-retro): gaps filled → full span observed
retro_md: Set[int] = set(range(_span_start, _span_end + 1))
retro_span: Tuple[int, int] = (_span_start, _span_end)

# ---------------------------
# 2) Resolve video metadata (no artifacts dependency)
#    Use clip_info/CLIP_INFO if present; else use common scalar globals.
# ---------------------------
def _get_meta() -> Dict[str, Any]:
    # Prefer dict-style clip info if present
    for name in ("clip_info", "CLIP_INFO", "VIDEO_META", "video_meta", "VIDEO_INFO", "video_info"):
        d = globals().get(name, None)
        if isinstance(d, dict):
            fps = d.get("fps", d.get("FPS"))
            n   = d.get("total_frames", d.get("n_frames", d.get("frame_count", d.get("frames"))))
            w   = d.get("w", d.get("width", d.get("W")))
            h   = d.get("h", d.get("height", d.get("H")))
            if fps is not None and w is not None and h is not None:
                return {"fps": float(fps), "total_frames": int(n or 0), "w": int(w), "h": int(h), "source": name}

    # Scalar fallbacks (common Step7 names)
    fps = globals().get("FPS", globals().get("CLIP_FPS", globals().get("VIDEO_FPS", None)))
    n   = globals().get("TOTAL_FRAMES", globals().get("N_FRAMES", globals().get("FRAME_COUNT", 0)))
    w   = globals().get("FRAME_W", globals().get("W", globals().get("WIDTH", None)))
    h   = globals().get("FRAME_H", globals().get("H", globals().get("HEIGHT", None)))

    if fps is None or w is None or h is None:
        raise NameError(
            "Missing video metadata for packet export.\n"
            "Provide one of: clip_info/CLIP_INFO dict with fps,total_frames,w,h OR globals FPS/TOTAL_FRAMES/FRAME_W/FRAME_H."
        )

    return {"fps": float(fps), "total_frames": int(n or 0), "w": int(w), "h": int(h), "source": "globals"}

_meta = _get_meta()
_fps = float(_meta["fps"])
_total_frames = int(_meta["total_frames"])
_frame_w = int(_meta["w"])
_frame_h = int(_meta["h"])

# ---------------------------
# 3) Ensure exporter exists (uses packet_name)
# ---------------------------
if "export_comparison_packet_step7" not in globals():
    raise NameError("export_comparison_packet_step7(...) is not defined. Run your adapter cell (7.2B) first.")

# ---------------------------
# 4) Export BOTH packets
# ---------------------------
export_comparison_packet_step7(
    run_dir=_run_dir,
    video_path=_video_path,
    fps=_fps,
    total_frames=_total_frames,
    frame_w=_frame_w,
    frame_h=_frame_h,
    primary_span=baseline_span,
    md_frames=baseline_md,
    active_pixels_by_frame=None,
    x_by_frame=None,
    y_by_frame=None,
    extra_manifest={
        "variant": "baseline_pre_retro",
        "gap_snapshot_var": _gap_name,
        "gap_frames_count": len(_gap_frames),
        "meta_source": _meta.get("source"),
    },
    packet_name="packet_step7_baseline",
)

export_comparison_packet_step7(
    run_dir=_run_dir,
    video_path=_video_path,
    fps=_fps,
    total_frames=_total_frames,
    frame_w=_frame_w,
    frame_h=_frame_h,
    primary_span=retro_span,
    md_frames=retro_md,
    active_pixels_by_frame=None,
    x_by_frame=None,
    y_by_frame=None,
    extra_manifest={
        "variant": "retro_interpolated",
        "interpolated_frames_count": len(_gap_frames),
        "meta_source": _meta.get("source"),
    },
    packet_name="packet_step7_retro",
)

print("📦 Step7 packets written:")
print(f"  • {_run_dir / 'packet_step7_baseline'}  (md_total={len(baseline_md)} gaps={len(_gap_frames)})")
print(f"  • {_run_dir / 'packet_step7_retro'}     (md_total={len(retro_md)} gaps=0)")
print(f"  span={baseline_span}  fps={_fps}  meta_source={_meta.get('source')}")


NameError: Could not find a Step7 gap snapshot dict.
Expected ORIGINAL_BEST_TRACK_GAPS (dict) OR any dict containing keys:
  frames_span, gap_frames, length, observation_density
Run Step 7.10 and ensure it saves that snapshot to a variable.

In [53]:
# ======================================================================
# STEP 7.10X – EXPORT STANDARD COMPARISON PACKETS (BASELINE + RETRO)
# Place at END of Step 7.10 cell, AFTER ORIGINAL_BEST_TRACK and INTERPOLATED_BEST_TRACK exist
# ======================================================================

from typing import Dict, Set, Tuple, Optional, Any
from pathlib import Path
import csv, json, os
from datetime import datetime

# ---------------------------
# 0) Required globals check
# ---------------------------
_required = ["RUN_DIR", "VIDEO_PATH", "ORIGINAL_BEST_TRACK", "INTERPOLATED_BEST_TRACK"]
_missing = [k for k in _required if k not in globals()]
if _missing:
    raise NameError(
        f"Step 7.10X export missing required globals: {_missing}\n"
        f"Expected: RUN_DIR, VIDEO_PATH, ORIGINAL_BEST_TRACK, INTERPOLATED_BEST_TRACK"
    )

_run_dir = Path(RUN_DIR)
_video_path = str(VIDEO_PATH)

# ---------------------------
# 1) Robust track → frame set
# ---------------------------
def _track_frame_set(t: Any) -> Set[int]:
    """
    Robustly extract frame indices from a Step7 track object.
    Supports common shapes:
      - t.observations: dict {frame_idx: (x,y) ...}
      - t.frames: list[int]
      - t.frame_indices: list[int]
    """
    if t is None:
        return set()

    obs = getattr(t, "observations", None)
    if isinstance(obs, dict) and len(obs) > 0:
        return set(int(k) for k in obs.keys())

    frames = getattr(t, "frames", None)
    if isinstance(frames, (list, tuple)) and len(frames) > 0:
        return set(int(x) for x in frames)

    fi = getattr(t, "frame_indices", None)
    if isinstance(fi, (list, tuple)) and len(fi) > 0:
        return set(int(x) for x in fi)

    raise ValueError("Cannot extract frames from track (no observations/frames/frame_indices).")

def _span_from_frames(frames_set: Set[int]) -> Tuple[int, int]:
    if not frames_set:
        return (0, 0)
    return (int(min(frames_set)), int(max(frames_set)))

# ---------------------------
# 2) Metadata resolver (NO artifacts, NO cv2 probe)
#    Tries common Step7 vars/dicts; fails loud with a useful message.
# ---------------------------
def _get_video_meta_from_globals() -> Dict[str, Any]:
    # Prefer dict-style metadata if present
    for name in ("clip_info", "CLIP_INFO", "VIDEO_INFO", "video_info", "VIDEO_META", "video_meta"):
        d = globals().get(name, None)
        if isinstance(d, dict):
            # tolerate multiple key names
            fps = d.get("fps", d.get("FPS"))
            n   = d.get("total_frames", d.get("n_frames", d.get("frame_count", d.get("frames"))))
            w   = d.get("w", d.get("width", d.get("W")))
            h   = d.get("h", d.get("height", d.get("H")))
            if fps is not None and w is not None and h is not None:
                return {"fps": float(fps), "total_frames": int(n or 0), "w": int(w), "h": int(h), "source": name}

    # Then scalar-style globals
    fps_candidates = ("FPS", "fps", "CLIP_FPS", "VIDEO_FPS", "INPUT_FPS")
    n_candidates   = ("TOTAL_FRAMES", "total_frames", "N_FRAMES", "FRAME_COUNT", "n_frames")
    w_candidates   = ("FRAME_W", "frame_w", "W", "WIDTH", "width")
    h_candidates   = ("FRAME_H", "frame_h", "H", "HEIGHT", "height")

    def pick(cands):
        for c in cands:
            if c in globals() and globals()[c] is not None:
                return c, globals()[c]
        return None, None

    fps_k, fps_v = pick(fps_candidates)
    n_k,   n_v   = pick(n_candidates)
    w_k,   w_v   = pick(w_candidates)
    h_k,   h_v   = pick(h_candidates)

    if fps_v is not None and w_v is not None and h_v is not None:
        return {
            "fps": float(fps_v),
            "total_frames": int(n_v or 0),
            "w": int(w_v),
            "h": int(h_v),
            "source": f"{fps_k},{n_k},{w_k},{h_k}",
        }

    # Fail loud with introspection
    present = [k for k in list(globals().keys()) if k in (
        "FPS","fps","CLIP_FPS","VIDEO_FPS","INPUT_FPS",
        "TOTAL_FRAMES","total_frames","N_FRAMES","FRAME_COUNT","n_frames",
        "FRAME_W","frame_w","W","WIDTH","width",
        "FRAME_H","frame_h","H","HEIGHT","height",
        "clip_info","CLIP_INFO","VIDEO_INFO","video_info","VIDEO_META","video_meta"
    )]
    raise NameError(
        "Missing video metadata for packet export (no artifacts used).\n"
        f"VIDEO_PATH={_video_path}\n"
        "Provide either:\n"
        "  • a dict named clip_info/CLIP_INFO/VIDEO_INFO/VIDEO_META with keys like fps,total_frames,w,h\n"
        "  OR\n"
        "  • scalar globals like FPS,TOTAL_FRAMES,FRAME_W,FRAME_H (or W/H).\n"
        f"Metadata-related globals currently present: {present}"
    )

_meta = _get_video_meta_from_globals()
_fps = float(_meta["fps"])
_total_frames = int(_meta["total_frames"])
_frame_w = int(_meta["w"])
_frame_h = int(_meta["h"])

if _fps <= 0 or _frame_w <= 0 or _frame_h <= 0:
    raise NameError(f"Bad metadata values: fps={_fps}, w={_frame_w}, h={_frame_h} (source={_meta.get('source')})")

# ---------------------------
# 3) Provide exporter if not already defined (safe)
# ---------------------------
if "export_comparison_packet_step7" not in globals():

    def export_comparison_packet_step7(
        *,
        run_dir: Path,
        video_path: str,
        fps: float,
        total_frames: int,
        frame_w: int,
        frame_h: int,
        primary_span: Tuple[int, int],
        md_frames: Set[int],
        active_pixels_by_frame: Optional[Dict[int, float]] = None,
        x_by_frame: Optional[Dict[int, float]] = None,
        y_by_frame: Optional[Dict[int, float]] = None,
        extra_manifest: Optional[Dict[str, Any]] = None,
        packet_name: str = "packet",
    ) -> None:
        run_dir = Path(run_dir)
        packet_dir = run_dir / packet_name
        packet_dir.mkdir(parents=True, exist_ok=True)

        manifest = {
            "pipeline_name": "Step7_full",
            "pipeline_version": "notebook",
            "video_path": str(video_path),
            "video_name": Path(video_path).name,
            "clip_info": {"fps": float(fps), "total_frames": int(total_frames), "w": int(frame_w), "h": int(frame_h)},
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
        if extra_manifest:
            manifest.update(extra_manifest)

        with (packet_dir / "run_manifest.json").open("w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)

        s, e = primary_span

        # frame_signal.csv
        rows = []
        for fidx in range(int(s), int(e) + 1):
            md_flag = 1 if fidx in md_frames else 0
            active = ""
            if active_pixels_by_frame is not None:
                active = active_pixels_by_frame.get(fidx, "")
            x = ""
            y = ""
            if x_by_frame is not None:
                x = x_by_frame.get(fidx, "")
            if y_by_frame is not None:
                y = y_by_frame.get(fidx, "")

            rows.append([fidx, md_flag, 1, active, "", "", x, y, "Step7"])

        with (packet_dir / "frame_signal.csv").open("w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["frame_idx","md_flag","in_primary_span","active_pixels","energy_raw","energy_norm","x","y","source"])
            w.writerows(rows)

        # segment_summary.csv
        span_len = int(e - s + 1)
        md_total = int(sum(1 for fidx in range(int(s), int(e) + 1) if fidx in md_frames))
        density = float(md_total / span_len) if span_len else 0.0

        md_flags = [1 if fidx in md_frames else 0 for fidx in range(int(s), int(e) + 1)]
        gap_count = 0
        max_gap = 0
        cur = 0
        for v in md_flags:
            if v == 0:
                cur += 1
            else:
                if cur > 0:
                    gap_count += 1
                    max_gap = max(max_gap, cur)
                    cur = 0
        if cur > 0:
            gap_count += 1
            max_gap = max(max_gap, cur)

        with (packet_dir / "segment_summary.csv").open("w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["segment_id","span_start","span_end","span_len","md_total","density","gap_count","max_gap","is_primary"])
            w.writerow([1, int(s), int(e), span_len, md_total, density, gap_count, max_gap, 1])

# ---------------------------
# 4) Export BOTH packets
# ---------------------------
baseline_track = ORIGINAL_BEST_TRACK
retro_track    = INTERPOLATED_BEST_TRACK

baseline_md = _track_frame_set(baseline_track)
baseline_span = _span_from_frames(baseline_md)
if baseline_span == (0, 0):
    raise NameError("Baseline track has no frames; cannot export packet_step7_baseline.")

retro_md = _track_frame_set(retro_track)
retro_span = _span_from_frames(retro_md)
if retro_span == (0, 0):
    raise NameError("Retro track has no frames; cannot export packet_step7_retro.")

export_comparison_packet_step7(
    run_dir=_run_dir,
    video_path=_video_path,
    fps=_fps,
    total_frames=_total_frames,
    frame_w=_frame_w,
    frame_h=_frame_h,
    primary_span=baseline_span,
    md_frames=baseline_md,
    active_pixels_by_frame=None,
    x_by_frame=None,
    y_by_frame=None,
    extra_manifest={"variant": "baseline_pre_retro", "meta_source": _meta.get("source")},
    packet_name="packet_step7_baseline",
)

export_comparison_packet_step7(
    run_dir=_run_dir,
    video_path=_video_path,
    fps=_fps,
    total_frames=_total_frames,
    frame_w=_frame_w,
    frame_h=_frame_h,
    primary_span=retro_span,
    md_frames=retro_md,
    active_pixels_by_frame=None,
    x_by_frame=None,
    y_by_frame=None,
    extra_manifest={"variant": "retro_interpolated", "meta_source": _meta.get("source")},
    packet_name="packet_step7_retro",
)

print("📦 Step7 packet export complete:")
print(f"  • {(_run_dir / 'packet_step7_baseline')}")
print(f"  • {(_run_dir / 'packet_step7_retro')}")
print(f"  meta_source={_meta.get('source')} fps={_fps} w={_frame_w} h={_frame_h} total_frames={_total_frames}")


NameError: Step 7.10X export missing required globals: ['ORIGINAL_BEST_TRACK', 'INTERPOLATED_BEST_TRACK']
Expected: RUN_DIR, VIDEO_PATH, ORIGINAL_BEST_TRACK, INTERPOLATED_BEST_TRACK

In [ ]:
stopper

In [ ]:
# =================================================================================================
# ✅ STEP 7.10: RETRO-INTERPOLATION DIAGNOSTIC (GAP FILL FOR BEST TRACK)
# =================================================================================================

"""
RETRO-INTERPOLATION: Fill short gaps in the best track using linear interpolation
=================================================================================

PURPOSE
-------
Step 7.10 is a *post-hoc diagnostic*, not a new tracking mode.

Given the best gated track from Step 7.9:

  • Identify internal gap frames within its span (frames with no observation)
  • For each gap sequence of length ≤ MAX_INTERPOLATION_GAP:
      - Interpolate centroid between surrounding observed frames
      - Optionally interpolate bbox
      - Mark those frames as interpolated (no longer true “gaps”)
  • Compare BEFORE vs AFTER:
      - gap_count
      - observation_density
      - number of interpolated frames

This cell must:
  • Preserve the original best track state (for audits / comparison)
  • Produce a separate interpolated copy for plotting / video work

It does NOT:
  • Re-run detection, track building, or gating
  • Change gating decisions (those are already done in 7.9)
"""


# -------------------------------------------------------------------------------------------------
# USER CONFIGURATION
# -------------------------------------------------------------------------------------------------

ENABLE_RETRO_INTERPOLATION = True        # Master toggle
APPLY_TO_BEST_TRACK_ONLY   = True       # Only operate on best gated track
MAX_INTERPOLATION_GAP      = 10         # Max consecutive missing frames to fill


print("\n" + "=" * 70)
print("STEP 7.10: RETRO-INTERPOLATION DIAGNOSTIC")
print("=" * 70)
print(f"\n✅ RETRO-INTERPOLATION ENABLED: {ENABLE_RETRO_INTERPOLATION}")
print(f"   Apply to best track only: {APPLY_TO_BEST_TRACK_ONLY}")
print(f"   Max interpolation gap:    {MAX_INTERPOLATION_GAP} frames")


# -------------------------------------------------------------------------------------------------
# CORE FUNCTION: RETRO INTERPOLATION
# -------------------------------------------------------------------------------------------------

def retro_interpolate_track(track: Track, max_gap_size: int = 10) -> Track:
    """
    Fill gaps in track with linearly interpolated positions.

    Assumes:
      • track.frames gives sorted observed frame indices
      • gap_frames is maintained as a set of missing frames inside the span

    Behavior:
      • For each gap frame:
          - Find nearest observed frame before and after
          - If (frame_after - frame_before - 1) <= max_gap_size → interpolate
          - Insert new centroid (and bbox if available)
          - Remove frame from track.gap_frames
      • Observation density rises as length increases (span unchanged)

    Returns:
      The same Track object (modified in-place).
    """

    # If track has fewer than 2 observations, no meaningful interpolation
    if not track.observations or len(track.frames) < 2:
        print(f"   ⚠ Track {track.track_id}: cannot interpolate (< 2 observations)")
        return track

    # Build full span and gap frames from existing data
    first_frame = track.frames[0]
    last_frame  = track.frames[-1]
    full_span   = set(range(first_frame, last_frame + 1))

    observed_set = set(track.frames)
    existing_gaps = sorted(full_span - observed_set)

    if not existing_gaps:
        print(f"   ✓ Track {track.track_id}: no gaps to fill")
        return track

    print(f"\n   📊 BEFORE INTERPOLATION (Track {track.track_id}):")
    print(f"      Span frames     : {first_frame} → {last_frame} ({track.span} frames)")
    print(f"      Observed frames : {track.length}")
    print(f"      Gap frames      : {len(existing_gaps)}")
    print(f"      Observation dens: {track.observation_density:.1%}")

    interpolated_frames = []
    skipped_frames      = []

    # Ensure gap_frames set exists / is in sync
    if not hasattr(track, "gap_frames") or track.gap_frames is None:
        track.gap_frames = set(existing_gaps)
    else:
        # Make sure it at least contains these gaps
        track.gap_frames.update(existing_gaps)

    for gap_f in existing_gaps:
        # Frames with observations before / after this gap
        prev_obs = [f for f in track.frames if f < gap_f]
        next_obs = [f for f in track.frames if f > gap_f]

        if not prev_obs or not next_obs:
            skipped_frames.append(gap_f)
            continue  # edge gaps; we skip

        f_before = max(prev_obs)
        f_after  = min(next_obs)

        gap_size = f_after - f_before - 1
        if gap_size > max_gap_size:
            skipped_frames.append(gap_f)
            continue  # too long, do not interpolate

        # Positions before/after
        cx0, cy0 = track.observations[f_before]
        cx1, cy1 = track.observations[f_after]

        dt_total = f_after - f_before
        if dt_total == 0:
            skipped_frames.append(gap_f)
            continue

        t = (gap_f - f_before) / dt_total

        # Interpolated centroid
        cx = int(round(cx0 + t * (cx1 - cx0)))
        cy = int(round(cy0 + t * (cy1 - cy0)))

        track.observations[gap_f] = (cx, cy)
        interpolated_frames.append(gap_f)

        # Remove from gap set (no longer a gap)
        track.gap_frames.discard(gap_f)

        # Interpolate bbox if available for both neighbors
        if hasattr(track, "bboxes"):
            if f_before in track.bboxes and f_after in track.bboxes:
                bx0, by0, bw0, bh0 = track.bboxes[f_before]
                bx1, by1, bw1, bh1 = track.bboxes[f_after]

                bx = int(round(bx0 + t * (bx1 - bx0)))
                by = int(round(by0 + t * (by1 - by0)))
                bw = int(round(bw0 + t * (bw1 - bw0)))
                bh = int(round(bh0 + t * (bh1 - bh0)))

                track.bboxes[gap_f] = (bx, by, bw, bh)

    # AFTER: recompute derived quantities (properties will reflect new observations)
    print(f"\n   📊 AFTER INTERPOLATION (Track {track.track_id}):")
    print(f"      Observed frames : {track.length}")
    print(f"      Remaining gaps  : {track.gap_count}")
    print(f"      Observation dens: {track.observation_density:.1%}")
    print(f"      Interpolated    : {len(interpolated_frames)} frame(s)")
    if interpolated_frames:
        # Show actual frame numbers, truncated if long
        if len(interpolated_frames) <= 20:
            print(f"      Interpolated at : {sorted(interpolated_frames)}")
        else:
            first_10 = sorted(interpolated_frames)[:10]
            last_5   = sorted(interpolated_frames)[-5:]
            print(f"      Interpolated at : {first_10} ... {last_5}")
    if skipped_frames:
        print(f"      Skipped         : {len(skipped_frames)} frame(s) "
              f"(edges or gaps > {max_gap_size})")

    # Attach metadata for downstream steps (optional)
    track.interpolated_frames = set(interpolated_frames)
    track.was_interpolated    = len(interpolated_frames) > 0

    return track

# -------------------------------------------------------------------------------------------------
# MAIN EXECUTION BLOCK
# -------------------------------------------------------------------------------------------------

if not ENABLE_RETRO_INTERPOLATION:
    print("\n⏭️  Retro-interpolation disabled (ENABLE_RETRO_INTERPOLATION = False)")
    ORIGINAL_BEST_TRACK       = None
    ORIGINAL_BEST_TRACK_GAPS  = None
    INTERPOLATED_BEST_TRACK   = None

else:
    # Sanity checks: we need gated_tracks from Step 7.9
    if "gated_tracks" not in globals():
        print("\n❌ No `gated_tracks` found. Run Step 7.9 pipeline first.")
        ORIGINAL_BEST_TRACK       = None
        ORIGINAL_BEST_TRACK_GAPS  = None
        INTERPOLATED_BEST_TRACK   = None

    elif not gated_tracks:
        print("\n❌ gated_tracks is empty. No tracks available for interpolation.")
        ORIGINAL_BEST_TRACK       = None
        ORIGINAL_BEST_TRACK_GAPS  = None
        INTERPOLATED_BEST_TRACK   = None

    else:
        # Select best track *by confidence* (post-gating, post-scoring)
        best_track = max(
            gated_tracks, key=lambda t: (getattr(t, "confidence_score", 0.0) or 0.0)
        )

        print("\n🏆 Best Track Selected for Retro-Interpolation:")
        print(f"   Track ID        : {best_track.track_id}")
        print(f"   Frames          : {best_track.frames[0]} → {best_track.frames[-1]}")
        print(f"   Observed length : {best_track.length}")
        print(f"   Gap count       : {best_track.gap_count}")
        print(f"   Observation dens: {best_track.observation_density:.1%}")
        print(f"   Confidence      : {getattr(best_track, 'confidence_score', 0.0):.3f}")

        # Preserve original state (deep copy) for later comparison / plotting
        import copy

        ORIGINAL_BEST_TRACK = copy.deepcopy(best_track)
        ORIGINAL_BEST_TRACK_GAPS = {
            "track_id": best_track.track_id,
            "frames_span": (best_track.frames[0], best_track.frames[-1]),
            "length": best_track.length,
            "span": best_track.span,
            "gap_frames": best_track.gap_frames_sorted,
            "gap_sequences": best_track.gap_sequences,
            "gap_count": best_track.gap_count,
            "observation_density": best_track.observation_density,
            "confidence_score": getattr(best_track, "confidence_score", 0.0),
        }

        # Optional: push to notebook logger if available
        try:
            if "log" in globals():
                from pprint import pformat
                log.silent(
                    "\n=== Step 7.10 ORIGINAL_BEST_TRACK_GAPS ===\n"
                    + pformat(ORIGINAL_BEST_TRACK_GAPS)
                    + "\n=========================================\n",
                    note="Step 7.10 original best-track gap snapshot",
                )
        except Exception as e:
            print(f"⚠ Step 7.10 logging failed: {e}")

        # Interpolate on a copy so 7.9 baseline is preserved
        best_track_interp = copy.deepcopy(best_track)
        INTERPOLATED_BEST_TRACK = retro_interpolate_track(
            best_track_interp, max_gap_size=MAX_INTERPOLATION_GAP
        )

        # Optional: provide a retro-interpolated gated set for later cells
        gated_tracks_retro = list(gated_tracks)
        # Replace the same track_id in the list if present
        for i, t in enumerate(gated_tracks_retro):
            if t.track_id == best_track.track_id:
                gated_tracks_retro[i] = INTERPOLATED_BEST_TRACK
                break

                print("\n" + "=" * 70)
                
        print("STEP 7.10 SUMMARY: BEST TRACK BEFORE vs AFTER RETRO-INTERPOLATION")
        print("=" * 70)

        before_gaps   = len(ORIGINAL_BEST_TRACK_GAPS["gap_frames"])
        before_density = ORIGINAL_BEST_TRACK_GAPS["observation_density"]
        after_gaps    = INTERPOLATED_BEST_TRACK.gap_count
        after_density = INTERPOLATED_BEST_TRACK.observation_density
        interp_frames = sorted(getattr(INTERPOLATED_BEST_TRACK,
                                       "interpolated_frames", []))

        print("\nBEFORE:")
        print(f"   Frames span      : {ORIGINAL_BEST_TRACK_GAPS['frames_span'][0]} → "
              f"{ORIGINAL_BEST_TRACK_GAPS['frames_span'][1]}")
        print(f"   Observed length  : {ORIGINAL_BEST_TRACK_GAPS['length']}")
        print(f"   Gap frames       : {before_gaps}")
        if before_gaps:
            print(f"   Gap frame list   : {ORIGINAL_BEST_TRACK_GAPS['gap_frames']}")
        print(f"   Observation dens : {before_density:.1%}")

        print("\nAFTER:")
        print(f"   Frames span      : {INTERPOLATED_BEST_TRACK.frames[0]} → "
              f"{INTERPOLATED_BEST_TRACK.frames[-1]}")
        print(f"   Observed length  : {INTERPOLATED_BEST_TRACK.length}")
        print(f"   Gap frames       : {after_gaps}")
        if interp_frames:
            print(f"   Interpolated at  : {interp_frames}")
        print(f"   Observation dens : {after_density:.1%}")

        # One-line delta summary
        print("\nΔ CHANGE:")
        print(f"   Gaps        : {before_gaps} → {after_gaps}")
        print(f"   Density     : {before_density:.3f} → {after_density:.3f}")
        print(f"   Interpolated: {len(interp_frames)} frame(s)")

        print("\nNOTE:")
        print("   • ORIGINAL_BEST_TRACK / ORIGINAL_BEST_TRACK_GAPS preserve the pre-7.10 state.")
        print("   • INTERPOLATED_BEST_TRACK is safe for plotting or video overlays.")
        print("   • gated_tracks_retro holds the same set with the best track retro-filled.")
        print("\n" + "=" * 70)

# GS
# ======================================================================
# STEP 7.10X – EXPORT STANDARD COMPARISON PACKETS (BASELINE + RETRO)
# Insert at END of Step 7.10 cell (after ORIGINAL_BEST_TRACK and INTERPOLATED_BEST_TRACK exist)
# ======================================================================

def _track_frame_set(t):
    """
    Robustly extract frame indices from Step7 track object.
    Works with:
      - t.observations: dict {frame_idx: (x,y) ...}
      - t.frames: list of frame_idx
      - t.frame_indices: list of frame_idx
    """
    if t is None:
        return set()
    if hasattr(t, "observations") and isinstance(getattr(t, "observations"), dict):
        return set(int(k) for k in t.observations.keys())
    if hasattr(t, "frames"):
        return set(int(x) for x in getattr(t, "frames"))
    if hasattr(t, "frame_indices"):
        return set(int(x) for x in getattr(t, "frame_indices"))
    raise ValueError("Cannot extract frames from track (no observations/frames/frame_indices)")

def _span_from_frames(frames_set):
    if not frames_set:
        return (0, 0)
    s = min(frames_set)
    e = max(frames_set)
    return (int(s), int(e))

# ---- REQUIRED: map these two variables to your Step 7.10 outputs ----
# These names MUST match what Step 7.10 currently uses.
# If your Step 7.10 uses different names, change ONLY these two lines.
baseline_track = ORIGINAL_BEST_TRACK
retro_track    = INTERPOLATED_BEST_TRACK

# ---- Video metadata (use your notebook vars if present; fallback if not) ----
_run_dir = RUN_DIR
_video_path = VIDEO_PATH

# _fps = FPS if "FPS" in globals() else float(artifacts.get("fps", 0))
# _total_frames = TOTAL_FRAMES if "TOTAL_FRAMES" in globals() else int(artifacts.get("total_frames", 0))
# _frame_w = FRAME_W if "FRAME_W" in globals() else int(artifacts.get("w", 0))
# _frame_h = FRAME_H if "FRAME_H" in globals() else int(artifacts.get("h", 0))


# ---- Video metadata: artifacts-free, uses Step7 globals; fallback to VIDEO_META if present ----
if "FPS" in globals() and "TOTAL_FRAMES" in globals() and "FRAME_W" in globals() and "FRAME_H" in globals():
    _fps = float(FPS)
    _total_frames = int(TOTAL_FRAMES)
    _frame_w = int(FRAME_W)
    _frame_h = int(FRAME_H)

elif "VIDEO_META" in globals() and isinstance(VIDEO_META, dict):
    _fps = float(VIDEO_META.get("fps", 0))
    _total_frames = int(VIDEO_META.get("total_frames", VIDEO_META.get("n_frames", 0)))
    _frame_w = int(VIDEO_META.get("w", VIDEO_META.get("width", 0)))
    _frame_h = int(VIDEO_META.get("h", VIDEO_META.get("height", 0)))

else:
    raise NameError(
        "Missing video metadata for packet export. Define FPS, TOTAL_FRAMES, FRAME_W, FRAME_H "
        "earlier (video load cell), or provide VIDEO_META={'fps':..,'total_frames':..,'w':..,'h':..}."
    )






# ---- Build baseline packet ----
baseline_md = _track_frame_set(baseline_track)
baseline_span = _span_from_frames(baseline_md)

export_comparison_packet_step7(
    run_dir=_run_dir,
    video_path=str(_video_path),
    fps=float(_fps),
    total_frames=int(_total_frames),
    frame_w=int(_frame_w),
    frame_h=int(_frame_h),
    primary_span=baseline_span,
    md_frames=baseline_md,
    active_pixels_by_frame=None,
    x_by_frame=None,
    y_by_frame=None,
    extra_manifest={"variant": "baseline_pre_retro"},
    packet_name="packet_step7_baseline",
)

# ---- Build retro packet ----
retro_md = _track_frame_set(retro_track)
retro_span = _span_from_frames(retro_md)

export_comparison_packet_step7(
    run_dir=_run_dir,
    video_path=str(_video_path),
    fps=float(_fps),
    total_frames=int(_total_frames),
    frame_w=int(_frame_w),
    frame_h=int(_frame_h),
    primary_span=retro_span,
    md_frames=retro_md,
    active_pixels_by_frame=None,
    x_by_frame=None,
    y_by_frame=None,
    extra_manifest={"variant": "retro_interpolated"},
    packet_name="packet_step7_retro",
)

print(f"📦 Step7 packets written:\n  {Path(_run_dir) / 'packet_step7_baseline'}\n  {Path(_run_dir) / 'packet_step7_retro'}")

        




## <font color = yellow> 7.10 B: Empty

In [ ]:
all_tracks[3]

## <font color = yellow> 7.11: Analysis and Comparison

In [ ]:
# =================================================================================================
# STEP 7.11: ANALYSIS, SUMMARY & OUTPUT (SIMPLIFIED)
# =================================================================================================

def summarize_step7_results(all_tracks: List[Track], 
                            gated_tracks: List[Track],
                            label: str = "") -> None:
    """
    Print comprehensive Step 7 summary with gap analysis.
    
    Reports:
      - Track counts (total vs gated)
      - Gap statistics (observation density)
      - Length distributions (observed vs span)
      - Status breakdown
      - Quality ranking
    
    Args:
        all_tracks: All tracks from build_tracks_standard
        gated_tracks: Tracks that passed temporal gates
        label: Optional label for output
    """
    
    total_tracks = len(all_tracks)
    kept_tracks = len(gated_tracks)
    
    print("\n" + "=" * 70)
    print(f"STEP 7.9 Full-pipeline execution: TRACK SUMMARY {f'({label})' if label else ''}")
    print("=" * 70)
    
    # -----------------------------------------------------------------
    # Basic Statistics
    # -----------------------------------------------------------------
    print(f"\n📊 Track Statistics:")
    print(f"   Total tracks built: {total_tracks}")
    print(f"   Tracks kept (gated): {kept_tracks} ({100*kept_tracks/total_tracks if total_tracks else 0:.1f}%)")
    
    # -----------------------------------------------------------------
    # Gap Statistics (ALL TRACKS)
    # -----------------------------------------------------------------
    if all_tracks:
        # Compute aggregates
        total_gaps = sum(t.gap_count for t in all_tracks)
        tracks_with_gaps = sum(1 for t in all_tracks if t.gap_count > 0)
        avg_obs_density = sum(t.observation_density for t in all_tracks) / len(all_tracks)
        
        total_observed = sum(t.length for t in all_tracks)
        total_span = sum(t.span for t in all_tracks)
        
        print(f"\n🔗 Gap Statistics (ALL TRACKS):")
        print(f"   Tracks with gaps: {tracks_with_gaps}/{total_tracks} ({100*tracks_with_gaps/total_tracks:.1f}%)")
        print(f"   Total gap frames: {total_gaps}")
        print(f"   Total observed frames: {total_observed}")
        print(f"   Total track span (obs+gaps): {total_span}")
        print(f"   Overall density: {total_observed/total_span:.1%}" if total_span > 0 else "")
        print(f"   Avg track density: {avg_obs_density:.1%}")
    
    # -----------------------------------------------------------------
    # Gap Statistics (KEPT TRACKS)
    # -----------------------------------------------------------------
    if gated_tracks:
        kept_gaps = sum(t.gap_count for t in gated_tracks)
        kept_observed = sum(t.length for t in gated_tracks)
        kept_span = sum(t.span for t in gated_tracks)
        kept_avg_density = sum(t.observation_density for t in gated_tracks) / len(gated_tracks)
        
        print(f"\n🔗 Gap Statistics (KEPT TRACKS):")
        print(f"   Total gap frames: {kept_gaps}")
        print(f"   Total observed frames: {kept_observed}")
        print(f"   Total track span: {kept_span}")
        print(f"   Overall density: {kept_observed/kept_span:.1%}" if kept_span > 0 else "")
        print(f"   Avg track density: {kept_avg_density:.1%}")
    
    # -----------------------------------------------------------------
    # Length Distribution
    # -----------------------------------------------------------------
    if all_tracks:
        lengths_all = [t.length for t in all_tracks]
        spans_all = [t.span for t in all_tracks]
        print(f"\n📏 Track Lengths (ALL):")
        print(f"   Observed: min={min(lengths_all)}, max={max(lengths_all)}, "
              f"mean={sum(lengths_all)/len(lengths_all):.1f}")
        print(f"   Span: min={min(spans_all)}, max={max(spans_all)}, "
              f"mean={sum(spans_all)/len(spans_all):.1f}")
    
    if gated_tracks:
        lengths_kept = [t.length for t in gated_tracks]
        spans_kept = [t.span for t in gated_tracks]
        print(f"\n📏 Track Lengths (KEPT):")
        print(f"   Observed: min={min(lengths_kept)}, max={max(lengths_kept)}, "
              f"mean={sum(lengths_kept)/len(lengths_kept):.1f}")
        print(f"   Span: min={min(spans_kept)}, max={max(spans_kept)}, "
              f"mean={sum(spans_kept)/len(spans_kept):.1f}")
    
    # -----------------------------------------------------------------
    # Status Breakdown
    # -----------------------------------------------------------------
    status_counts: Dict[str, int] = {}
    for t in all_tracks:
        st = t.status
        status_counts[st] = status_counts.get(st, 0) + 1
    
    print(f"\n🚦 Track Status Breakdown:")
    for st, cnt in sorted(status_counts.items()):
        pct = 100 * cnt / total_tracks if total_tracks else 0
        print(f"   {st:15s}: {cnt:3d} ({pct:5.1f}%)")
    
    # -----------------------------------------------------------------
    # Per-Track Gap Breakdown (Top 10 Kept Tracks)
    # -----------------------------------------------------------------
    if gated_tracks:
        print(f"\n🔍 Gap Breakdown (Top 10 Kept Tracks):")
        print(f"   {'Track':>5} | {'Obs':>4} | {'Gaps':>4} | {'Span':>4} | {'Density':>7} | {'Conf':>6}")
        print(f"   {'-'*5}-+-{'-'*4}-+-{'-'*4}-+-{'-'*4}-+-{'-'*7}-+-{'-'*6}")
        
        # Sort by confidence score
        ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
        
        # Loop over top 10 tracks
        for t in ranked[:10]:
            obs = t.length
            gaps = t.gap_count
            span = t.span
            density = t.observation_density
            score = t.confidence_score
            
            print(f"   {t.track_id:5d} | {obs:4d} | {gaps:4d} | {span:4d} | {density:6.1%} | {score:6.3f}")
    
    # -----------------------------------------------------------------
    # Quality Ranking (Best Track)
    # -----------------------------------------------------------------
    if USE_TRACK_QUALITY_SCORING and gated_tracks:
        print("\n" + "=" * 70)
        print("STEP 7: QUALITY RANKING (Gap-Aware)")
        print("=" * 70)
        
        ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
        
        if ranked:
            best = ranked[0]
            
            print(f"\n🏆 BEST TRACK:")
            print(f"   Track ID: {best.track_id}")
            print(f"   Observed frames: {best.length}")
            print(f"   Gap frames: {best.gap_count}")
            print(f"   Total span: {best.span} (frames {best.frames[0]}→{best.frames[-1]})")
            print(f"   Observation density: {best.observation_density:.1%}")
            print(f"   Quality score: {best.quality_score:.3f}")
            print(f"   Confidence score: {best.confidence_score:.3f}")
            
            if best.mean_heading_deg is not None:
                print(f"   Mean heading: {best.mean_heading_deg:.1f}°")
            if best.mean_y is not None:
                print(f"   Mean Y: {best.mean_y:.1f}px")
            print(f"   Corridor occupancy: {best.y_band_fraction:.1%}")
    
    print("\n" + "=" * 70)

# Now safe to call
config_changes = detect_config_changes()

print("\n\n" + "=" * 70)
print("⏱️ CONFIGURATION Change SUMMARY")
print("=" * 70)

config_changes = detect_config_changes()
print(f"\n{config_changes['summary']}")

# Run summary
summarize_step7_results(all_tracks, gated_tracks, label="STANDARD APPROACH")

## <font color = yellow> 7.12: Visualization and Comparison

#### <font color = lime> Create plots from data

In [ ]:
# =================================================================================================
# STEP 7.11: VISUALIZATION (SIMPLIFIED)
# =================================================================================================

def plot_step7_analysis(tracks: List[Track], title: str = "Step 7 Analysis"):
    """
    Comprehensive Step 7 visualization (4-panel).
    
    PANELS:
    -------
    1. Track trajectories (color by confidence)
    2. Frame timeline (observed vs gaps)
    3. Observation density histogram
    4. Quality vs density scatter
    
    Args:
        tracks: List of tracks to visualize
        title: Plot title
    """
    
    if not tracks:
        print("⚠️  No tracks to plot")
        return
    
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # -----------------------------------------------------------------
    # PANEL 1: Track Trajectories
    # -----------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Loop over tracks
    for track in tracks:
        centroids = track.get_centroids_list()
        xs = [c[0] for c in centroids]
        ys = [c[1] for c in centroids]
        
        conf = track.confidence_score
        color = plt.cm.RdYlGn(conf)
        
        # Plot trajectory
        ax1.plot(xs, ys, 'o-', color=color, linewidth=2, markersize=4, 
                alpha=0.7, label=f"T{track.track_id} (conf={conf:.2f})")
    
    ax1.set_xlim(0, 1920)
    ax1.set_ylim(1080, 0)
    ax1.set_xlabel('X (pixels)', fontsize=11)
    ax1.set_ylabel('Y (pixels)', fontsize=11)
    ax1.set_title(f'{title}\nTrajectories (color = confidence)', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=8, loc='upper right', ncol=2)
    ax1.grid(True, alpha=0.3)
    
    # -----------------------------------------------------------------
    # PANEL 2: Frame Timeline
    # -----------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 1])
    
    # Loop over tracks
    for i, track in enumerate(tracks):
        y_pos = i
        
        # Get all frames in span
        if not track.observations:
            continue
        
        frames_list = track.frames
        start_frame = frames_list[0]
        end_frame = frames_list[-1]
        
        # Plot observed frames as green bars
        for frame in frames_list:
            ax2.plot([frame, frame], [y_pos - 0.3, y_pos + 0.3], 
                    color='green', linewidth=3, alpha=0.7)
        
        # Plot gap frames as red bars
        all_frames_in_span = set(range(start_frame, end_frame + 1))
        gap_frames = all_frames_in_span - set(frames_list)
        
        for gap_frame in gap_frames:
            ax2.plot([gap_frame, gap_frame], [y_pos - 0.3, y_pos + 0.3], 
                    color='red', linewidth=3, alpha=0.5, linestyle='--')
        
        # Span line (gray background)
        ax2.plot([start_frame, end_frame], [y_pos, y_pos], 
                color='gray', linewidth=1, alpha=0.3, zorder=-1)
    
    ax2.set_xlabel('Frame Index', fontsize=11)
    ax2.set_ylabel('Track ID', fontsize=11)
    ax2.set_yticks(range(len(tracks)))
    ax2.set_yticklabels([f"T{t.track_id}" for t in tracks], fontsize=8)
    ax2.set_title('Frame Timeline\n(Green=observed, Red=gap)', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # -----------------------------------------------------------------
    # PANEL 3: Observation Density Histogram
    # -----------------------------------------------------------------
    ax3 = fig.add_subplot(gs[1, 0])
    
    densities = [t.observation_density for t in tracks]
    
    ax3.hist(densities, bins=20, alpha=0.7, color='blue', edgecolor='black')
    ax3.axvline(MIN_OBSERVATION_DENSITY, color='red', linestyle='--', linewidth=2, 
               label=f'Min threshold ({MIN_OBSERVATION_DENSITY:.1%})')
    
    ax3.set_xlabel('Observation Density', fontsize=11)
    ax3.set_ylabel('Frequency', fontsize=11)
    ax3.set_title('Observation Density Distribution', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # -----------------------------------------------------------------
    # PANEL 4: Quality vs Density Scatter
    # -----------------------------------------------------------------
    ax4 = fig.add_subplot(gs[1, 1])
    
    densities = [t.observation_density for t in tracks]
    quality_scores = [t.quality_score for t in tracks]
    confidence_scores = [t.confidence_score for t in tracks]
    lengths = [t.length for t in tracks]
    
    # Scatter: density vs quality, color by confidence, size by length
    scatter = ax4.scatter(densities, quality_scores, 
                         c=confidence_scores, s=[5*L for L in lengths],
                         cmap='RdYlGn', alpha=0.7, edgecolors='black', linewidths=1)
    
    # Annotate track IDs
    for t in tracks:
        ax4.annotate(f"T{t.track_id}", 
                    (t.observation_density, t.quality_score),
                    fontsize=8, alpha=0.6)
    
    ax4.set_xlabel('Observation Density', fontsize=11)
    ax4.set_ylabel('Quality Score', fontsize=11)
    ax4.set_title('Quality vs Density\n(color=confidence, size=length)', 
                 fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('Confidence Score', rotation=270, labelpad=15, fontsize=10)
    
    plt.suptitle(f'{title} - Comprehensive Analysis', 
                fontsize=14, fontweight='bold', y=0.98)
    
    plt.show()


# Execute visualization
if gated_tracks:
    plot_step7_analysis(gated_tracks, title="Step 7: Standard Temporal Association")
else:
    print("⚠️  No gated tracks to visualize")

print("✓ Visualization complete")

## <font color = yellow> 7.13: Best Video (optional)

##### <font color = lime> Create video from frame / Gap motion detections


In [ ]:
# =================================================================================================
# 🆕 STEP 7.13: BEST TRACK VISUALIZATION (VIDEO) - ENHANCED
# =================================================================================================

"""
BEST TRACK VIDEO GENERATION (ENHANCED)
========================================

PURPOSE:
--------
Visual confirmation that the algorithm correctly identified the eagle trajectory.
Enhanced with detailed motion metrics and gap visualization.

ENHANCEMENTS:
-------------
- Shows position (X, Y) and bearing for each frame
- Displays delta position (change from previous frame)
- Computes and displays acceleration
- RED text for predicted/gap frames
- Directional arrow showing bearing
- Frame-by-frame motion analysis

OUTPUT:
-------
Video with three-line caption:
  Line 1: Track metadata (ID, frame count, confidence)
  Line 2: Position (X, Y) and bearing (with gap indicator in RED)
  Line 3: Delta position and acceleration (in RED for gaps)
"""

# =================================================================================================
# USER CONFIGURATION (TOP OF CELL)
# =================================================================================================

# --- Master Toggle ---
ENABLE_BEST_TRACK_VIDEO = True  # Set to False to skip this entire cell

# --- Video Generation ---
GENERATE_BEST_TRACK_VIDEO = True  # Create video from best track
SAVE_BEST_TRACK_VIDEO = True      # Save video to disk
# BEST_TRACK_VIDEO_PATH = "best_track_visualization.mp4"

BEST_TRACK_VIDEO_PATH = os.path.join(str(VID_DIR), "best_track_visualization.mp4")

# --- Video Playback ---
PLAY_VIDEO_AFTER_GENERATION = True  # Auto-play video after creation
USE_WINDOWS_DEFAULT_PLAYER = True   # Use Windows default player (MPV or system default)
                                     # If False, tries to play inline in Jupyter

# MPV Player Options (Windows)
MPV_EXECUTABLE = "mpv"  # Assumes MPV is in system PATH
                        # If not, use full path: r"C:\Program Files\mpv\mpv.exe"
MPV_OPTIONS = [
    "--loop=no",           # Don't loop video (play once)
    "--keep-open=yes",     # Keep window open after playback
    "--ontop",             # Keep player window on top
    "--geometry=50%:50%",  # Center window on screen
    "--autofit=80%",       # Scale to 80% of screen size
]

# Fallback to system default player if MPV not found
USE_SYSTEM_DEFAULT_IF_MPV_MISSING = True

# --- Visualization Options ---
BBOX_COLOR = (0, 255, 0)           # Green bounding box (BGR format)
BBOX_THICKNESS = 3                 # Line thickness in pixels
TEXT_FONT = cv2.FONT_HERSHEY_SIMPLEX
TEXT_SCALE = 0.6                   # Font scale for captions
TEXT_THICKNESS = 2                 # Font thickness

# Color scheme
TEXT_COLOR_NORMAL = (0, 255, 255)   # Yellow text (normal frames)
TEXT_COLOR_GAP = (0, 0, 255)        # Red text (gap/predicted frames)
CENTROID_COLOR_NORMAL = (255, 0, 255)  # Magenta centroid (normal)
CENTROID_COLOR_GAP = (0, 0, 255)    # Red centroid (gap)
BEARING_ARROW_COLOR = (255, 255, 0) # Cyan bearing arrow

# --- Video Encoding ---
OUTPUT_CODEC = 'mp4v'  # Codec fourcc code ('mp4v', 'avc1', 'XVID')
OUTPUT_FPS = 25        # Frame rate of output video (DEFAULT: 25 fps for recorded video)

# --- Motion Analysis Options ---
ARROW_LENGTH = 50       # Bearing arrow length in pixels
SHOW_ACCELERATION = True  # Compute and display acceleration

# =================================================================================================
# INITIALIZATION
# =================================================================================================

print("=" * 70)
print("STEP 7.13: BEST TRACK VISUALIZATION (VIDEO) - ENHANCED")
print("=" * 70)

if not ENABLE_BEST_TRACK_VIDEO:
    print("\n⏭️  SKIPPED (ENABLE_BEST_TRACK_VIDEO = False)")
    print("   Set ENABLE_BEST_TRACK_VIDEO = True to generate visualization")
else:
    print("\n✅ ENABLED")
    print(f"   Generate video: {GENERATE_BEST_TRACK_VIDEO}")
    print(f"   Save to disk: {SAVE_BEST_TRACK_VIDEO}")
    if SAVE_BEST_TRACK_VIDEO:
        print(f"   Output path: {BEST_TRACK_VIDEO_PATH}")
    print(f"   Output FPS: {OUTPUT_FPS} (default for 25fps recorded video)")
    print(f"   Play after generation: {PLAY_VIDEO_AFTER_GENERATION}")
    if PLAY_VIDEO_AFTER_GENERATION:
        if USE_WINDOWS_DEFAULT_PLAYER:
            print(f"   Player: Windows default (MPV preferred)")
        else:
            print(f"   Player: Inline (Jupyter/Colab)")

# =================================================================================================
# MOTION METRICS COMPUTATION
# =================================================================================================

def compute_frame_motion_metrics(track: Track, 
                                 frame_idx: int, 
                                 prev_frame_idx: Optional[int] = None) -> Dict[str, Any]:
    """
    Compute motion metrics for a specific frame.
    
    Metrics:
    --------
    • Position (X, Y) - Current centroid position
    • Bearing - Direction of motion in degrees (0° = right, 90° = down)
    • Delta X, Delta Y - Change in position from previous frame
    • Acceleration - Change in velocity magnitude (px/frame²)
    
    Args:
        track: Track object
        frame_idx: Current frame index
        prev_frame_idx: Previous frame index (None for first frame)
    
    Returns:
        Dict with metrics: position, bearing, delta_x, delta_y, acceleration
    """
    
    metrics = {
        "position": (0, 0),
        "bearing": 0.0,
        "delta_x": 0,
        "delta_y": 0,
        "acceleration": 0.0,
        "is_gap": False
    }
    
    # Get current position
    if frame_idx in track.observations:
        metrics["position"] = track.observations[frame_idx]
        metrics["is_gap"] = False
    else:
        # Should not happen (we only process observed frames)
        metrics["is_gap"] = True
        return metrics
    
    curr_x, curr_y = metrics["position"]
    
    # Compute bearing
    frames = track.frames
    try:
        curr_idx = frames.index(frame_idx)
    except ValueError:
        curr_idx = -1
    
    if curr_idx > 0:
        # Get previous observation
        prev_frame = frames[curr_idx - 1]
        prev_x, prev_y = track.observations[prev_frame]
        
        # Compute displacement
        delta_x = curr_x - prev_x
        delta_y = curr_y - prev_y
        
        metrics["delta_x"] = delta_x
        metrics["delta_y"] = delta_y
        
        # Compute bearing (atan2 gives angle in radians)
        if delta_x != 0 or delta_y != 0:
            bearing_rad = math.atan2(delta_y, delta_x)
            metrics["bearing"] = math.degrees(bearing_rad)
        else:
            # No motion, use track mean heading
            metrics["bearing"] = track.mean_heading_deg if track.mean_heading_deg is not None else 0.0
    else:
        # First frame, use track mean heading
        metrics["bearing"] = track.mean_heading_deg if track.mean_heading_deg is not None else 0.0
    
    # Compute acceleration (change in velocity magnitude)
    # Requires two velocity measurements
    if curr_idx >= 2:
        # Get velocity from previous step
        prev_prev_frame_idx = frames[curr_idx - 2]
        
        if prev_prev_frame_idx in track.observations:
            prev_frame_idx = frames[curr_idx - 1]
            
            # Velocity at previous step
            prev_x = track.observations[prev_frame_idx][0]
            prev_y = track.observations[prev_frame_idx][1]
            prev_prev_x = track.observations[prev_prev_frame_idx][0]
            prev_prev_y = track.observations[prev_prev_frame_idx][1]
            
            dt_prev = prev_frame_idx - prev_prev_frame_idx
            if dt_prev > 0:
                v_prev_x = (prev_x - prev_prev_x) / dt_prev
                v_prev_y = (prev_y - prev_prev_y) / dt_prev
                v_prev_mag = math.hypot(v_prev_x, v_prev_y)
            else:
                v_prev_mag = 0.0
            
            # Velocity at current step
            dt_curr = frame_idx - prev_frame_idx
            if dt_curr > 0:
                v_curr_x = (curr_x - prev_x) / dt_curr
                v_curr_y = (curr_y - prev_y) / dt_curr
                v_curr_mag = math.hypot(v_curr_x, v_curr_y)
            else:
                v_curr_mag = 0.0
            
            # Acceleration (change in speed)
            # Units: (px/frame) / frame = px/frame²
            dt_avg = (dt_prev + dt_curr) / 2.0
            if dt_avg > 0:
                metrics["acceleration"] = (v_curr_mag - v_prev_mag) / dt_avg
            else:
                metrics["acceleration"] = 0.0
    
    return metrics


def detect_gap_frames(track: Track) -> set:
    """
    Detect frames that came immediately after a gap.
    
    These frames should be marked in RED to indicate potential
    prediction/recovery after occlusion.
    
    Args:
        track: Track object
    
    Returns:
        Set of frame indices that follow gaps
    """
    
    gap_frames = set()
    frames = track.frames
    
    # Loop through consecutive frame pairs
    for i in range(1, len(frames)):
        prev_frame = frames[i - 1]
        curr_frame = frames[i]
        gap_size = curr_frame - prev_frame - 1
        
        # If gap exists (missed frames between observations)
        if gap_size > 0:
            gap_frames.add(curr_frame)
    
    return gap_frames

# =================================================================================================
# FRAME EXTRACTION
# =================================================================================================

def extract_best_track_frames(video_path: str, track: Track) -> Dict[int, np.ndarray]:
    """
    Extract frames corresponding to best track's observations.
    
    Args:
        video_path: Path to source video
        track: Best track object
    
    Returns:
        Dict mapping frame_index → frame_image (BGR numpy array)
    """
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    
    # Get frame indices where track was observed
    target_frames = set(track.frames)
    
    frames_dict = {}
    frame_idx = 0
    
    print(f"\n📹 Extracting {len(target_frames)} frames from video...")
    
    # Read video sequentially
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Check if this frame is in track
        if frame_idx in target_frames:
            frames_dict[frame_idx] = frame.copy()
        
        frame_idx += 1
        
        # Early exit if we've collected all frames
        if len(frames_dict) == len(target_frames):
            break
    
    cap.release()
    
    print(f"   ✓ Extracted {len(frames_dict)} frames")
    
    return frames_dict

# =================================================================================================
# FRAME ANNOTATION
# =================================================================================================

def draw_track_annotation(frame: np.ndarray, 
                          track: Track, 
                          frame_idx: int,
                          total_frames: int,
                          metrics: Dict[str, Any],
                          is_gap_frame: bool) -> np.ndarray:
    """
    Draw enhanced annotations on frame.
    
    Annotations:
    ------------
    • Bounding box (green)
    • Centroid marker (magenta = normal, red = gap)
    • Bearing arrow (cyan)
    • Three-line caption:
        Line 1: Best Track | Frames | Confidence
        Line 2: Position: X | Y | Bearing (RED if gap)
        Line 3: Change X | Change Y | Acceleration (RED if gap)
    
    Args:
        frame: Frame image (BGR, modified in-place)
        track: Track object
        frame_idx: Current frame index
        total_frames: Total frames in track
        metrics: Motion metrics dict from compute_frame_motion_metrics()
        is_gap_frame: Whether this frame follows a gap
    
    Returns:
        Annotated frame (same as input, modified in-place)
    """
    
    # Get bounding box for this frame
    if frame_idx not in track.bboxes:
        print(f"⚠️  Warning: Frame {frame_idx} not in track bboxes")
        return frame

    # GS  ====================================================================================================
    x, y, w, h = track.bboxes[frame_idx]
    
    # Draw bounding box (always green)
   #  cv2.rectangle(frame, (x, y), (x + w, y + h), BBOX_COLOR, BBOX_THICKNESS)


    x, y, w, h = track.bboxes[frame_idx]

    # Optional: a little padding so the box doesn’t hug the pixels too tight
    pad = 20
    x1 = int(x - pad)
    y1 = int(y - pad)
    x2 = int(x + w + pad)
    y2 = int(y + h + pad)
    
    # Choose outline color based on gap status if you like
    bbox_color = CENTROID_COLOR_GAP if is_gap_frame else CENTROID_COLOR_NORMAL
    
    # Draw OUTLINE ONLY (thickness > 0, no fill)
    cv2.rectangle(frame, (x1, y1), (x2, y2), bbox_color, 2)
    
        

    # ====================================================================================================
    
    # Get position and metrics
    cx, cy = metrics["position"]
    bearing = metrics["bearing"]
    delta_x = metrics["delta_x"]
    delta_y = metrics["delta_y"]
    acceleration = metrics["acceleration"]
    
    # Choose colors based on gap status
    text_color = TEXT_COLOR_GAP if is_gap_frame else TEXT_COLOR_NORMAL
    centroid_color = CENTROID_COLOR_GAP if is_gap_frame else CENTROID_COLOR_NORMAL

    # =========================================================================================
    # ... bounding box drawing ...
    
    # Choose colors based on gap status (LINE ~310)
    text_color = TEXT_COLOR_GAP if is_gap_frame else TEXT_COLOR_NORMAL
    centroid_color = CENTROID_COLOR_GAP if is_gap_frame else CENTROID_COLOR_NORMAL





    
    # ... caption building (LINE ~320) ...
    gap_indicator = " [GAP FRAME]" if is_gap_frame else ""
    caption_line2 = f"Position: X={cx:.0f} | Y={cy:.0f} | Bearing: {bearing:.1f}°{gap_indicator}"

  # =========================================================================================
    
    # -----------------------------------------------------------------
    # CAPTION LINE 1: Track metadata
    # -----------------------------------------------------------------
    caption_line1 = f"Best Track (ID: {track.track_id}) | Frames: {frame_idx}/{track.frames[-1]} | Confidence: {track.confidence_score:.3f}"
    
    # -----------------------------------------------------------------
    # CAPTION LINE 2: Position and bearing (RED if gap)
    # -----------------------------------------------------------------
    gap_indicator = " [GAP FRAME]" if is_gap_frame else ""
    caption_line2 = f"Position: X={cx:.0f} | Y={cy:.0f} | Bearing: {bearing:.1f}°{gap_indicator}"
    
    # -----------------------------------------------------------------
    # CAPTION LINE 3: Delta position and acceleration (RED if gap)
    # -----------------------------------------------------------------
    # caption_line3 = f"Change: ΔX={delta_x:+.0f}px | ΔY={delta_y:+.0f}px | Accel: {acceleration:+.2f}px/f²"
    # caption_line3 = f"Change: X={delta_x:+.0f}px | Y={delta_y:+.0f}px | Accel: {acceleration:+.2f}px/f²"
    caption_line3 = f"Change: dX={delta_x:+.0f}px | dY={delta_y:+.0f}px | Accel: {acceleration:+.2f}px/f^2"
    # -----------------------------------------------------------------
    # Draw text backgrounds (semi-transparent black boxes)
    # -----------------------------------------------------------------
    text_x = 10
    text_y_line1 = 30
    text_y_line2 = 55
    text_y_line3 = 80
    padding = 5
    
    # Get text sizes
    (text_w1, text_h1), _ = cv2.getTextSize(caption_line1, TEXT_FONT, TEXT_SCALE, TEXT_THICKNESS)
    (text_w2, text_h2), _ = cv2.getTextSize(caption_line2, TEXT_FONT, TEXT_SCALE, TEXT_THICKNESS)
    (text_w3, text_h3), _ = cv2.getTextSize(caption_line3, TEXT_FONT, TEXT_SCALE, TEXT_THICKNESS)
    
    # Background rectangles
    for (line_y, line_h, line_w) in [
        (text_y_line1, text_h1, text_w1),
        (text_y_line2, text_h2, text_w2),
        (text_y_line3, text_h3, text_w3)
    ]:
# GS +++++++++++++++++++++++++++++++++++++++++++++++++++ Culprit ======================
        cv2.rectangle(frame, 
                     (text_x - padding, line_y - line_h - padding),
                     (text_x + line_w + padding, line_y + padding),
                     # (0, 0, 0), -1)  # Filled black rectangle        # solid fill
                      (0, 0, 0), 5)  # Filled black rectangle          # bounding 
    
    # -----------------------------------------------------------------
    # Draw text captions
    # -----------------------------------------------------------------
    # Line 1: Track info (always yellow)
    cv2.putText(frame, caption_line1, (text_x, text_y_line1), 
               TEXT_FONT, TEXT_SCALE, TEXT_COLOR_NORMAL, TEXT_THICKNESS, cv2.LINE_AA)
    
    # Line 2: Position and bearing (yellow or RED)
    cv2.putText(frame, caption_line2, (text_x, text_y_line2), 
               TEXT_FONT, TEXT_SCALE, text_color, TEXT_THICKNESS, cv2.LINE_AA)
    
    # Line 3: Delta and acceleration (yellow or RED)
    cv2.putText(frame, caption_line3, (text_x, text_y_line3), 
               TEXT_FONT, TEXT_SCALE, text_color, TEXT_THICKNESS, cv2.LINE_AA)
    
    # -----------------------------------------------------------------
    # Draw centroid marker (magenta or RED)
    # -----------------------------------------------------------------
    # gs 
    #  cv2.circle(frame, (cx, cy), 6, centroid_color, -1)  # Filled circle   Not wanted w/ small MD obect
    cv2.circle(frame, (cx, cy), 8, centroid_color, 3)   # Outer ring
    
    # -----------------------------------------------------------------
    # Draw bearing arrow (cyan, pointing in direction of motion)
    # -----------------------------------------------------------------
    if bearing is not None:
        # Convert bearing to radians
        bearing_rad = math.radians(bearing)
        
        # Compute arrow endpoint
        arrow_end_x = int(cx + ARROW_LENGTH * math.cos(bearing_rad))
        arrow_end_y = int(cy + ARROW_LENGTH * math.sin(bearing_rad))
        
        # Draw arrow with tip

        # GS  remove arrow
        # cv2.arrowedLine(frame, (cx, cy), (arrow_end_x, arrow_end_y),
         #              BEARING_ARROW_COLOR, 3, tipLength=0.3)
    
    return frame

# =================================================================================================
# VIDEO CREATION
# =================================================================================================

def create_best_track_video(video_path: str, 
                            track: Track, 
                            output_path: str,
                            fps: int = 25) -> bool:
    """
    Generate annotated video of best track.
    
    Workflow:
    ---------
    1. Extract frames from source video
    2. Detect gap frames (frames following missed detections)
    3. Compute motion metrics for each frame
    4. Annotate frames with enhanced captions
    5. Write to output video file
    
    Args:
        video_path: Path to source video
        track: Best track object
        output_path: Path to save output video
        fps: Frame rate of output video (default: 25 for recorded video)
    
    Returns:
        True if successful, False otherwise
    """
    
    print("\n🎬 Generating best track video...")
    
    # -----------------------------------------------------------------
    # Step 1: Extract frames
    # -----------------------------------------------------------------
    frames_dict = extract_best_track_frames(video_path, track)
    
    if not frames_dict:
        print("❌ No frames extracted")
        return False
    
    # -----------------------------------------------------------------
    # Step 2: Get frame dimensions
    # -----------------------------------------------------------------
    sample_frame = next(iter(frames_dict.values()))
    height, width = sample_frame.shape[:2]
    
    print(f"   Frame size: {width}×{height}")
    print(f"   Output FPS: {fps}")
    
    # -----------------------------------------------------------------
    # Step 3: Initialize video writer
    # -----------------------------------------------------------------
    fourcc = cv2.VideoWriter_fourcc(*OUTPUT_CODEC)
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    if not out.isOpened():
        print(f"❌ Failed to open video writer: {output_path}")
        return False
    
    # -----------------------------------------------------------------
    # Step 4: Detect gap frames
    # -----------------------------------------------------------------
    gap_frames = detect_gap_frames(track)
    
    if gap_frames:
        print(f"   Gap frames detected: {len(gap_frames)} (marked in RED)")
    
    # -----------------------------------------------------------------
    # Step 5: Process frames in temporal order
    # -----------------------------------------------------------------
    sorted_frames = sorted(frames_dict.keys())
    total_frames = len(sorted_frames)
    
    print(f"   Processing {total_frames} frames...")
    
    # Pre-compute motion metrics for all frames (with caching)
    motion_metrics_cache = {}
    
    # Loop to compute metrics
    for i, frame_idx in enumerate(sorted_frames):
        prev_frame_idx = sorted_frames[i - 1] if i > 0 else None
        metrics = compute_frame_motion_metrics(track, frame_idx, prev_frame_idx)
        motion_metrics_cache[frame_idx] = metrics
    
    # -----------------------------------------------------------------
    # Step 6: Annotate and write frames
    # -----------------------------------------------------------------
    # Loop through frames in temporal order
    for frame_idx in sorted_frames:
        frame = frames_dict[frame_idx].copy()
        
        # Get metrics from cache
        metrics = motion_metrics_cache[frame_idx]
        
        # Check if this is a gap frame
        is_gap_frame = frame_idx in gap_frames
        
        # Draw annotations
        annotated_frame = draw_track_annotation(
            frame, 
            track, 
            frame_idx, 
            total_frames,
            metrics,
            is_gap_frame
        )
        
        # Write to video
        out.write(annotated_frame)
    
    # -----------------------------------------------------------------
    # Step 7: Release writer
    # -----------------------------------------------------------------
    out.release()
    
    print(f"   ✓ Video saved to: {output_path}")
    
    if gap_frames:
        print(f"   ✓ {len(gap_frames)} gap frames marked in RED")
    
    return True

# =================================================================================================
# VIDEO PLAYBACK
# =================================================================================================

def play_video_windows(video_path: str) -> bool:
    """
    Play video using Windows default player (MPV preferred, system default fallback).
    
    Args:
        video_path: Path to video file
    
    Returns:
        True if player launched successfully, False otherwise
    
    Playback Strategy:
    ------------------
    1. Try MPV player (if MPV_EXECUTABLE in PATH or specified path exists)
    2. Fallback to Windows default player (os.startfile)
    3. Report failure if neither works
    """
    
    import subprocess
    import shutil
    
    print(f"\n▶️  Launching video player...")
    
    # Convert to absolute path
    abs_video_path = os.path.abspath(video_path)
    
    if not os.path.exists(abs_video_path):
        print(f"❌ Video file not found: {abs_video_path}")
        return False
    
    # -----------------------------------------------------------------
    # ATTEMPT 1: MPV Player
    # -----------------------------------------------------------------
    mpv_available = shutil.which(MPV_EXECUTABLE) is not None
    
    if mpv_available or os.path.exists(MPV_EXECUTABLE):
        try:
            print(f"   Attempting to launch MPV player...")
            
            # Build MPV command
            cmd = [MPV_EXECUTABLE] + MPV_OPTIONS + [abs_video_path]
            
            # Launch MPV (non-blocking)
            subprocess.Popen(
                cmd,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                creationflags=subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0
            )
            
            print(f"   ✓ MPV player launched")
            print(f"   Video: {abs_video_path}")
            return True
            
        except Exception as e:
            print(f"   ⚠️  MPV launch failed: {e}")
            
            if not USE_SYSTEM_DEFAULT_IF_MPV_MISSING:
                print(f"   USE_SYSTEM_DEFAULT_IF_MPV_MISSING = False")
                return False
            
            print(f"   Falling back to system default player...")
    
    else:
        print(f"   MPV player not found in PATH: {MPV_EXECUTABLE}")
        
        if not USE_SYSTEM_DEFAULT_IF_MPV_MISSING:
            print(f"   USE_SYSTEM_DEFAULT_IF_MPV_MISSING = False")
            print(f"   Install MPV: https://mpv.io/installation/")
            return False
        
        print(f"   Falling back to system default player...")
    
    # -----------------------------------------------------------------
    # ATTEMPT 2: Windows Default Player (os.startfile)
    # -----------------------------------------------------------------
    try:
        print(f"   Attempting to launch system default player...")
        
        # os.startfile uses Windows file associations
        os.startfile(abs_video_path)
        
        print(f"   ✓ System default player launched")
        print(f"   Video: {abs_video_path}")
        return True
        
    except Exception as e:
        print(f"   ❌ Failed to launch default player: {e}")
        return False


def play_video_inline(video_path: str) -> None:
    """
    Play video inline in Jupyter/Colab notebook (fallback method).
    
    Args:
        video_path: Path to video file
    
    Note:
        Requires IPython.display module (available in Jupyter/Colab)
    """
    
    try:
        from IPython.display import Video, display
        
        print(f"\n▶️  Playing video inline (Jupyter)...")
        
        # Display video
        display(Video(video_path, embed=True, width=800))
        
    except ImportError:
        print("\n⚠️  Cannot play inline: IPython.display not available")
        print(f"   Video saved to: {video_path}")
        print(f"   Open with external player to view")
    except Exception as e:
        print(f"\n⚠️  Error playing video: {e}")
        print(f"   Video saved to: {video_path}")

# =================================================================================================
# MAIN EXECUTION
# =================================================================================================

if ENABLE_BEST_TRACK_VIDEO and GENERATE_BEST_TRACK_VIDEO:
    
    # Check if we have gated tracks
    if not gated_tracks:
        print("\n❌ No gated tracks available")
        print("   Cannot generate video without valid tracks")
        print("   Check temporal gate settings or detection parameters")
    
    else:
        # Find best track (highest confidence score)
        best_track = max(gated_tracks, key=lambda t: t.confidence_score)
        
        print(f"\n🏆 Best Track Selected:")
        print(f"   Track ID: {best_track.track_id}")
        print(f"   Observed frames: {best_track.length}")
        print(f"   Gap frames: {best_track.gap_count}")
        print(f"   Span: {best_track.span} frames (frames {best_track.frames[0]}→{best_track.frames[-1]})")
        print(f"   Observation density: {best_track.observation_density:.1%}")
        print(f"   Confidence score: {best_track.confidence_score:.3f}")
        if best_track.mean_heading_deg is not None:
            print(f"   Mean bearing: {best_track.mean_heading_deg:.1f}°")
        
        # Generate video
        success = create_best_track_video(
            VIDEO_PATH,
            best_track,
            BEST_TRACK_VIDEO_PATH,
            fps=OUTPUT_FPS
        )
        
        if success:
            print("\n✅ Video generation complete")
            
            # Play video after generation
            if PLAY_VIDEO_AFTER_GENERATION:
                if USE_WINDOWS_DEFAULT_PLAYER:
                    # Use Windows player (MPV or system default)
                    play_success = play_video_windows(BEST_TRACK_VIDEO_PATH)
                    
                    if not play_success:
                        print(f"\n💡 Tip: Install MPV player for better experience:")
                        print(f"   https://mpv.io/installation/")
                        print(f"   Or manually open: {os.path.abspath(BEST_TRACK_VIDEO_PATH)}")
                else:
                    # Use inline player (Jupyter/Colab)
                    play_video_inline(BEST_TRACK_VIDEO_PATH)
            
            # Handle save option
            if not SAVE_BEST_TRACK_VIDEO:
                print(f"\n🗑️  Deleting temporary video (SAVE_BEST_TRACK_VIDEO = False)...")
                try:
                    import time
                    time.sleep(2)  # Wait for player to open file
                    if os.path.exists(BEST_TRACK_VIDEO_PATH):
                        os.remove(BEST_TRACK_VIDEO_PATH)
                        print(f"   ✓ Temporary file deleted")
                except Exception as e:
                    print(f"   ⚠️  Could not delete: {e}")
                    print(f"   File may be in use by video player")
        
        else:
            print("\n❌ Video generation failed")

elif ENABLE_BEST_TRACK_VIDEO and not GENERATE_BEST_TRACK_VIDEO:
    print("\n⏭️  Video generation disabled (GENERATE_BEST_TRACK_VIDEO = False)")


else:
    print("\n⏭️  Step 7.13 skipped (ENABLE_BEST_TRACK_VIDEO = False)")

print("\n" + "=" * 70)
# ------------------------------------------------------------------
# STEP 7.D – Run diagnostic best-track lens (if enabled)
# ------------------------------------------------------------------

from pathlib import Path
output_dir = Path(os.path.dirname(os.path.abspath(BEST_TRACK_VIDEO_PATH)) or ".")




# Compatibility wrapper so Step 7.D can reuse 7.13’s video builder
def build_best_track_video(
    video_path: str,
    track: "Track",
    track_id: int,
    output_dir: "Path",
    label: str,
    logger=None,
    config=None,
):
    """
    Wrapper so Step 7.D can call the 7.13 video generator using a consistent name.
    Produces a file named: <output_dir>/<label>_track_<ID>.mp4
    """
    output_path = output_dir / f"{label}_track_{track_id}.mp4"
    ok = create_best_track_video(
        video_path=video_path,
        track=track,
        output_path=str(output_path),
        fps=OUTPUT_FPS,
    )
    return output_path if ok else None




# ------------------------------------------------------------------
# STEP 7.D – Run diagnostic best-track lens (if enabled)
# ------------------------------------------------------------------
if "STEP7_DIAG" in globals() and STEP7_DIAG.get("enabled", False):
    print("\n[Step 7.D] Diagnostics enabled – calling run_step7_diag()...")
    run_step7_diag(
        artifacts=STEP7_ARTIFACTS,
        diag_config=STEP7_DIAG,
        video_path=VIDEO_PATH,
        output_dir=output_dir,   # same as used in 7.13 for video/plots
        logger=log if "log" in globals() else None,
        config=CONFIG if "CONFIG" in globals() else {},
    )
else:
    print("\n[Step 7.D] Diagnostics disabled or STEP7_DIAG not defined.")




## <font color = yellow> 7.13e – Diagnostic Best-Track Lens (run_step7_diag)

#### <font color = lime> This is a read-only diagnostic lens that answers:

    “If I ignore the fancy gating and just look at gap stats, which track looks best, and what does it actually look like in the video?”

##### This is genuinely useful when you’re debugging “did my gates throw away the real eagle?”

In [ ]:
# ======================================================================
# STEP 7.13e – DIAGNOSTIC BEST-TRACK LENS (READ-ONLY, OPTIONAL)
# ======================================================================



"""
    What this cell does:
    
        Prints which artifacts are present (sanity check).
        
        If all_tracks or gap_stats are missing/empty → prints and returns (no side-effects).
        
        If diag_config["make_raw_best_pre_gates_video"] is True:
        
        Builds a “raw pre-gates best track” score from gap_stats using:
        
        score = length + 50*density - 0.1*longest_gap
        
        
        Picks the highest-score track_id before any gating.
        
        Looks up that Track in all_tracks.
        
        Calls build_best_track_video(..., label="raw_pre_gates", ...) to write a debug video for that track.

"""





def run_step7_diag(
    artifacts: Dict[str, Any],
    diag_config: Dict[str, Any],
    video_path: str,
    output_dir: Path,
    logger,
    config: Dict[str, Any],
) -> None:
    """
    Minimal diagnostic lens:
      - Prints what artifacts are present.
      - Picks a 'raw best' track from pre-gate gap stats.
      - Writes a debug video for that track with label 'raw_pre_gates'.
    """
    print("\n" + "=" * 70)
    print("STEP 7.D – DIAGNOSTIC BEST-TRACK LENS")
    print("=" * 70)

    all_tracks = artifacts.get("all_tracks")
    gap_stats = artifacts.get("gap_stats")
    gating_cfg = artifacts.get("real_gating_config")
    gated_ids = artifacts.get("gated_track_ids")

    print(f"[Step 7.D] all_tracks present:    {all_tracks is not None}")
    print(f"[Step 7.D] gap_stats present:    {gap_stats is not None}")
    print(f"[Step 7.D] gating_config present:{gating_cfg is not None}")
    print(f"[Step 7.D] gated_track_ids:      {gated_ids if gated_ids is not None else 'None'}")

    # If we don't have tracks or gap_stats, bail with a clear message
    if not all_tracks or not gap_stats:
        print("[Step 7.D] Missing tracks or gap_stats – nothing to do.")
        return

    # ------------------------------------------------------------------
    # RAW PRE-GATES BEST TRACK VIDEO
    # ------------------------------------------------------------------
    if diag_config.get("make_raw_best_pre_gates_video", False):
        scored = []
        for row in gap_stats:
            try:
                tid = int(row["track_id"])
                length = float(row["length"])
                density = float(row["density"])
                longest_gap = float(row["longest_gap"])
            except Exception:
                continue

            # Simple heuristic:
            #   score = length + 50*density - 0.1*longest_gap
            score = length + 50.0 * density - 0.1 * longest_gap
            scored.append((score, tid))

        if not scored:
            print("[Step 7.D] No valid entries in gap_stats for RAW best.")
        else:
            scored.sort(reverse=True, key=lambda x: x[0])
            best_score, raw_best_id = scored[0]
            print(f"[Step 7.D] RAW pre-gates best track → ID={raw_best_id}, score={best_score:.3f}")

            track_obj = all_tracks.get(raw_best_id)
            if track_obj is None:
                print("[Step 7.D] RAW best track not found in all_tracks dict.")
            else:
                try:
                    raw_label = "raw_pre_gates"
                    raw_video_path = build_best_track_video(
                        video_path=video_path,
                        track=track_obj,
                        track_id=raw_best_id,
                        output_dir=output_dir,
                        label=raw_label,
                        logger=logger,
                        config=config,
                    )
                    print(f"✓ STEP 7.D: Wrote RAW pre-gates best-track video → {raw_video_path}")
                except Exception as e:
                    print(f"⚠ STEP 7.D: Failed to write RAW pre-gates video: {e}")

    print("\n[Step 7.D] Diagnostics complete.")


## <font color = yellow>  7.14: Export and Final Summary

In [ ]:
# =================================================================================================
# STEP 7.14: FINAL SUMMARY (WITH CONFIG CHANGE TRACKING)
# =================================================================================================
"""
STEP 7.14 – FINAL SUMMARY
=========================

PURPOSE:
--------
Provide a consolidated Step 7 summary with:
  • Configuration change report (vs defaults)
  • Track counts (total vs gated)
  • Gap statistics (all vs kept)
  • Length distributions (observed vs span)
  • Status breakdown
  • Top kept tracks with gap/density/score
  • Best-track quality snapshot (if scoring enabled)

USAGE:
------
Call after Step 7 pipeline has produced:
  • all_tracks     – canonical list of Track objects
  • gated_tracks   – subset that passed gating

Example:
    summarize_step7_results(all_tracks, gated_tracks, label="STANDARD APPROACH")
"""


def summarize_step7_results(
    all_tracks: List[Track],
    gated_tracks: List[Track],
    label: str = "",
) -> None:
    """
    Print comprehensive Step 7 summary with gap analysis and config changes.
    
    Args:
        all_tracks: All tracks from track-building step
        gated_tracks: Tracks that passed temporal/spatial gates
        label: Optional label for output (e.g., run name or approach)
    """
    # =========================================================================
    # SECTION 1: CONFIGURATION CHANGE REPORT
    # =========================================================================
    print("\n" + "=" * 70)
    print("CONFIGURATION SUMMARY")
    print("=" * 70)
    
    config_changes = detect_config_changes()
    print(f"\n{config_changes['summary']}")
    
    # =========================================================================
    # SECTION 2: TRACK SUMMARY
    # =========================================================================
    total_tracks = len(all_tracks)
    kept_tracks = len(gated_tracks)
    
    print("\n" + "=" * 70)
    print(f"STEP 7.14: TRACK SUMMARY {f'({label})' if label else ''}")
    print("=" * 70)
    
    print("\n📊 Track Statistics:")
    print(f"   Total tracks built: {total_tracks}")
    keep_pct = 100 * kept_tracks / total_tracks if total_tracks else 0.0
    print(f"   Tracks kept (gated): {kept_tracks} ({keep_pct:.1f}%)")
    
    # -----------------------------------------------------------------
    # Gap Statistics (ALL TRACKS)
    # -----------------------------------------------------------------
    if all_tracks:
        total_gaps = sum(t.gap_count for t in all_tracks)
        tracks_with_gaps = sum(1 for t in all_tracks if t.gap_count > 0)
        avg_obs_density = (
            sum(t.observation_density for t in all_tracks) / len(all_tracks)
        )
        
        total_observed = sum(t.length for t in all_tracks)
        total_span = sum(t.span for t in all_tracks)
        
        print("\n🔗 Gap Statistics (ALL TRACKS):")
        print(
            f"   Tracks with gaps: {tracks_with_gaps}/{total_tracks} "
            f"({100*tracks_with_gaps/total_tracks:.1f}%)"
        )
        print(f"   Total gap frames: {total_gaps}")
        print(f"   Total observed frames: {total_observed}")
        print(f"   Total track span (obs+gaps): {total_span}")
        if total_span > 0:
            print(f"   Overall density: {total_observed/total_span:.1%}")
        print(f"   Avg track density: {avg_obs_density:.1%}")
    
    # -----------------------------------------------------------------
    # Gap Statistics (KEPT TRACKS)
    # -----------------------------------------------------------------
    if gated_tracks:
        kept_gaps = sum(t.gap_count for t in gated_tracks)
        kept_observed = sum(t.length for t in gated_tracks)
        kept_span = sum(t.span for t in gated_tracks)
        kept_avg_density = (
            sum(t.observation_density for t in gated_tracks) / len(gated_tracks)
        )
        
        print("\n🔗 Gap Statistics (KEPT TRACKS):")
        print(f"   Total gap frames: {kept_gaps}")
        print(f"   Total observed frames: {kept_observed}")
        print(f"   Total track span: {kept_span}")
        if kept_span > 0:
            print(f"   Overall density: {kept_observed/kept_span:.1%}")
        print(f"   Avg track density: {kept_avg_density:.1%}")
    
    # -----------------------------------------------------------------
    # Length Distribution
    # -----------------------------------------------------------------
    if all_tracks:
        lengths_all = [t.length for t in all_tracks]
        spans_all = [t.span for t in all_tracks]
        print("\n📏 Track Lengths (ALL):")
        print(
            f"   Observed: min={min(lengths_all)}, max={max(lengths_all)}, "
            f"mean={sum(lengths_all)/len(lengths_all):.1f}"
        )
        print(
            f"   Span:     min={min(spans_all)}, max={max(spans_all)}, "
            f"mean={sum(spans_all)/len(spans_all):.1f}"
        )
    
    if gated_tracks:
        lengths_kept = [t.length for t in gated_tracks]
        spans_kept = [t.span for t in gated_tracks]
        print("\n📏 Track Lengths (KEPT):")
        print(
            f"   Observed: min={min(lengths_kept)}, max={max(lengths_kept)}, "
            f"mean={sum(lengths_kept)/len(lengths_kept):.1f}"
        )
        print(
            f"   Span:     min={min(spans_kept)}, max={max(spans_kept)}, "
            f"mean={sum(spans_kept)/len(spans_kept):.1f}"
        )
    
    # -----------------------------------------------------------------
    # Status Breakdown
    # -----------------------------------------------------------------
    status_counts: Dict[str, int] = {}
    for t in all_tracks:
        st = t.status
        status_counts[st] = status_counts.get(st, 0) + 1
    
    print("\n🚦 Track Status Breakdown:")
    for st, cnt in sorted(status_counts.items()):
        pct = 100 * cnt / total_tracks if total_tracks else 0.0
        print(f"   {st:15s}: {cnt:3d} ({pct:5.1f}%)")
    
    # -----------------------------------------------------------------
    # Per-Track Gap Breakdown (Top 10 Kept Tracks)
    # -----------------------------------------------------------------
    if gated_tracks:
        print("\n🔍 Gap Breakdown (Top 10 Kept Tracks):")
        print("   {:>5} | {:>4} | {:>4} | {:>4} | {:>7} | {:>6}".format(
            "Track", "Obs", "Gaps", "Span", "Density", "Conf"
        ))
        print("   " + "-"*5 + "-+-" + "-"*4 + "-+-" + "-"*4 + "-+-" +
              "-"*4 + "-+-" + "-"*7 + "-+-" + "-"*6)
        
        ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
        
        for t in ranked[:10]:
            print(
                f"   {t.track_id:5d} | {t.length:4d} | {t.gap_count:4d} | "
                f"{t.span:4d} | {t.observation_density:6.1%} | {t.confidence_score:6.3f}"
            )
    
    # -----------------------------------------------------------------
    # Quality Ranking (Best Track)
    # -----------------------------------------------------------------
    if USE_TRACK_QUALITY_SCORING and gated_tracks:
        print("\n" + "=" * 70)
        print("STEP 7: QUALITY RANKING (Gap-Aware)")
        print("=" * 70)
        
        ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
        best = ranked[0]
        
        print("\n🏆 BEST TRACK:")
        print(f"   Track ID:          {best.track_id}")
        print(f"   Observed frames:   {best.length}")
        print(f"   Gap frames:        {best.gap_count}")
        print(f"   Total span:        {best.span} (frames {best.frames[0]}→{best.frames[-1]})")
        print(f"   Observation density: {best.observation_density:.1%}")
        print(f"   Quality score:       {best.quality_score:.3f}")
        print(f"   Confidence score:    {best.confidence_score:.3f}")
        
        if best.mean_heading_deg is not None:
            print(f"   Mean heading:       {best.mean_heading_deg:.1f}°")
        if best.mean_y is not None:
            print(f"   Mean Y:             {best.mean_y:.1f}px")
        print(f"   Corridor occupancy: {best.y_band_fraction:.1%}")
    
    print("\n" + "=" * 70)


# -----------------------------------------------------------------
# Example (leave commented; call from end-of-pipeline cell)
# -----------------------------------------------------------------
# summarize_step7_results(all_tracks, gated_tracks, label="STANDARD APPROACH")



## <font color = lime> 7.15A: <font color = lime> Optional - validate plot accuracy

    Sanity check that visualization plots display the correct track with expected properties.
    
    WHEN TO USE:
    ------------
    ✅ First-time analysis of new video (validate algorithm worked correctly)
    ✅ After parameter tuning (verify changes didn't break tracking)
    ✅ Debugging unexpected results (spatial extent too small, wrong track plotted)
    ✅ Production deployment (automated validation in batch processing)

    ✅ Works for any video (no hardcoded values)
    ✅ User adjusts thresholds in config (not in code)
    ✅ Thresholds have semantic meaning ("minimum movement")

In [ ]:
# =================================================================================================
# 🔍 STEP 7.15A: PLOT & TRACK VALIDATION (OPTIONAL DIAGNOSTIC)
# =================================================================================================

"""
OPTIONAL VALIDATION
===================

PURPOSE:
--------
Sanity check that visualization plots display the correct track with expected properties.

WHEN TO USE:
------------
✅ First-time analysis of new video (validate algorithm worked correctly)
✅ After parameter tuning (verify changes didn't break tracking)
✅ Debugging unexpected results (spatial extent too small, wrong track plotted)
✅ Production deployment (automated validation in batch processing)

WHEN TO SKIP:
-------------
⏭️ Routine re-runs of same video with same parameters
⏭️ Computational efficiency critical (skip non-essential diagnostics)

CHECKS PERFORMED:
-----------------
1. Best track ID consistency (plot label matches ranked track)
2. Spatial extent validation (object moved reasonable distance)
3. Temporal coherence (frame span matches expectations)
4. Confidence score sanity (not suspiciously perfect/low)
"""

# =================================================================================================
# USER CONFIGURATION
# =================================================================================================

ENABLE_VALIDATION = True  # Master toggle (set False to skip entire cell)

# Validation thresholds (adjust based on expected object motion)
MIN_SPATIAL_DISPLACEMENT_X = 100  # Minimum horizontal travel (pixels)
MIN_SPATIAL_DISPLACEMENT_Y = 20   # Minimum vertical travel (pixels)
MIN_CONFIDENCE_SCORE = 0.3        # Minimum acceptable confidence
MAX_CONFIDENCE_SCORE = 0.999      # Maximum before suspecting bug (1.0 = perfect)

# Print verbosity
VERBOSE_OUTPUT = True  # Print detailed diagnostics

# =================================================================================================
# VALIDATION LOGIC
# =================================================================================================

if not ENABLE_VALIDATION:
    print("\n⏭️  Plot validation skipped (ENABLE_VALIDATION = False)")
else:
    print("\n" + "=" * 70)
    print("STEP 7.15A: PLOT & TRACK VALIDATION")
    print("=" * 70)
    
    # -----------------------------------------------------------------
    # CHECK 0: Prerequisites (Handle Empty Track List)
    # -----------------------------------------------------------------
    if not gated_tracks:
        print("\n❌ VALIDATION FAILED: No gated tracks available")
        print("\n   Possible Causes:")
        print("   1. All tracks rejected by temporal gates (too short, low density, etc.)")
        print("   2. Detection parameters too strict (no motion detected)")
        print("   3. Video contains no target objects")
        
        print("\n   Diagnostic Information:")
        if 'all_tracks' in dir() and all_tracks:
            print(f"   • Total tracks built: {len(all_tracks)}")
            
            # Show rejection breakdown
            status_counts = {}
            for t in all_tracks:
                st = t.status
                status_counts[st] = status_counts.get(st, 0) + 1
            
            print(f"   • Rejection breakdown:")
            for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
                pct = 100 * count / len(all_tracks)
                print(f"      - {status}: {count} ({pct:.1f}%)")
            
            # Show length distribution
            lengths = [t.length for t in all_tracks]
            print(f"   • Track lengths: min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.1f}")
        else:
            print(f"   • Total tracks built: 0 (no detections or tracking failed)")
        
        print("\n   Recommendations:")
        print("   1. Relax MIN_TRACK_LENGTH_FRAMES (current: {})".format(MIN_TRACK_LENGTH_FRAMES if 'MIN_TRACK_LENGTH_FRAMES' in dir() else 'unknown'))
        print("   2. Increase BUFFER (current: {})".format(BUFFER if 'BUFFER' in dir() else 'unknown'))
        print("   3. Lower MIN_OBSERVATION_DENSITY (current: {})".format(MIN_OBSERVATION_DENSITY if 'MIN_OBSERVATION_DENSITY' in dir() else 'unknown'))
        print("   4. Check video contains visible target objects")
        print("   5. Review Cell 7.9 output for specific rejection reasons")
        
        print("\n" + "=" * 70)
        
    else:
        # -----------------------------------------------------------------
        # Tracks exist - Proceed with validation
        # -----------------------------------------------------------------
        
        # Get best track (safe now that we checked gated_tracks is not empty)
        ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
        best = ranked[0]
        
        validation_passed = True
        warnings = []
        
        # -----------------------------------------------------------------
        # CHECK 1: Track Metadata
        # -----------------------------------------------------------------
        if VERBOSE_OUTPUT:
            print(f"\n🏆 Best Track (from Step 7.13):")
            print(f"   Track ID: {best.track_id}")
            print(f"   Frames: {best.frames[0]} → {best.frames[-1]}")
            print(f"   Observed: {best.length} frames")
            print(f"   Span: {best.span} frames")
            print(f"   Gaps: {best.gap_count} frames")
            print(f"   Observation density: {best.observation_density:.1%}")
            print(f"   Confidence: {best.confidence_score:.3f}")
        
        # -----------------------------------------------------------------
        # CHECK 2: Spatial Extent
        # -----------------------------------------------------------------
        positions = [best.observations[f] for f in best.frames]
        x_coords = [p[0] for p in positions]
        y_coords = [p[1] for p in positions]
        
        x_displacement = max(x_coords) - min(x_coords)
        y_displacement = max(y_coords) - min(y_coords)
        
        if VERBOSE_OUTPUT:
            print(f"\n📐 Spatial Extent:")
            print(f"   X range: {min(x_coords)} → {max(x_coords)} (Δ={x_displacement} px)")
            print(f"   Y range: {min(y_coords)} → {max(y_coords)} (Δ={y_displacement} px)")
        
        # Validate X displacement
        if x_displacement < MIN_SPATIAL_DISPLACEMENT_X:
            validation_passed = False
            warnings.append(
                f"⚠️  Horizontal displacement too small: {x_displacement}px "
                f"(expected ≥{MIN_SPATIAL_DISPLACEMENT_X}px)\n"
                f"      → Object may be stationary or track is noise"
            )
        
        # Validate Y displacement
        if y_displacement < MIN_SPATIAL_DISPLACEMENT_Y:
            warnings.append(
                f"⚠️  Vertical displacement small: {y_displacement}px "
                f"(expected ≥{MIN_SPATIAL_DISPLACEMENT_Y}px)\n"
                f"      → Object flying level (normal) or track may be horizontal line noise"
            )
        
        # -----------------------------------------------------------------
        # CHECK 3: Confidence Score Sanity
        # -----------------------------------------------------------------
        if best.confidence_score < MIN_CONFIDENCE_SCORE:
            validation_passed = False
            warnings.append(
                f"⚠️  Confidence score very low: {best.confidence_score:.3f} "
                f"(expected ≥{MIN_CONFIDENCE_SCORE})\n"
                f"      → Track quality poor, may not be target object"
            )
        
        if best.confidence_score > MAX_CONFIDENCE_SCORE:
            warnings.append(
                f"⚠️  Confidence score suspiciously perfect: {best.confidence_score:.3f} "
                f"(expected <{MAX_CONFIDENCE_SCORE})\n"
                f"      → Check for scoring bug or overfitting"
            )
        
        # -----------------------------------------------------------------
        # CHECK 4: Frame Coverage
        # -----------------------------------------------------------------
        if best.observation_density < MIN_OBSERVATION_DENSITY:
            warnings.append(
                f"⚠️  Observation density below threshold: {best.observation_density:.1%} "
                f"(expected ≥{MIN_OBSERVATION_DENSITY:.0%})\n"
                f"      → Track has many gaps, passed gating but low quality"
            )
        
        # -----------------------------------------------------------------
        # CHECK 5: Very Short Track Warning
        # -----------------------------------------------------------------
        if best.length < 5:
            warnings.append(
                f"⚠️  Best track is very short: {best.length} frames\n"
                f"      → Likely fragmentation issue (identity switching)\n"
                f"      → Consider implementing Hungarian algorithm or relaxing gates further"
            )
        
        # -----------------------------------------------------------------
        # SUMMARY
        # -----------------------------------------------------------------
        print(f"\n{'='*70}")
        print("VALIDATION SUMMARY")
        print(f"{'='*70}")
        
        if validation_passed and not warnings:
            print(f"\n✅ ALL CHECKS PASSED")
            print(f"   Best track (ID: {best.track_id}) is valid for visualization")
            print(f"   Spatial extent: {x_displacement}×{y_displacement} px")
            print(f"   Confidence: {best.confidence_score:.3f}")
        
        elif not validation_passed:
            print(f"\n❌ VALIDATION FAILED")
            print(f"   Critical issues detected with best track (ID: {best.track_id})")
            print(f"\n   Issues:")
            for warning in warnings:
                print(f"      {warning}")
            print(f"\n   Recommendations:")
            print(f"      1. Check detection parameters (diff_threshold, min_area)")
            print(f"      2. Review temporal gates (may be too strict)")
            print(f"      3. Visually inspect video at frames {best.frames[0]}-{best.frames[-1]}")
            if best.length < 5:
                print(f"      4. For multi-object scenarios: Implement Hungarian algorithm")
                print(f"      5. For fragmented tracking: Lower MIN_TRACK_LENGTH_FRAMES")
        
        else:  # Warnings but passed
            print(f"\n⚠️  PASSED WITH WARNINGS")
            print(f"   Best track (ID: {best.track_id}) is valid but has minor issues")
            print(f"\n   Warnings:")
            for warning in warnings:
                print(f"      {warning}")
        
        print(f"\n{'='*70}")

# =================================================================================================
# VERIFY PLOT CORRESPONDS TO BEST TRACK (SAFE VERSION)
# =================================================================================================

print("\n" + "=" * 70)
print("PLOT VALIDATION")
print("=" * 70)

# Check if tracks exist before proceeding
if not gated_tracks:
    print("\n⚠️  Cannot validate plot: No gated tracks available")
    print("   (See validation summary above for details)")
    
else:
    # Get best track (safe - already checked gated_tracks is not empty)
    ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
    best = ranked[0]
    
    print(f"\n🏆 Best Track from Cell 7.13:")
    print(f"   Track ID: {best.track_id}")
    print(f"   Frames: {best.frames[0]} → {best.frames[-1]}")
    print(f"   Observed: {best.length} frames")
    print(f"   Span: {best.span} frames")
    print(f"   Confidence: {best.confidence_score:.3f}")
    
    # Check spatial extent
    positions = [best.observations[f] for f in best.frames]
    x_coords = [p[0] for p in positions]
    y_coords = [p[1] for p in positions]
    
    x_displacement = max(x_coords) - min(x_coords)
    y_displacement = max(y_coords) - min(y_coords)
    
    print(f"\n📐 Spatial Extent:")
    print(f"   X range: {min(x_coords)} → {max(x_coords)} (Δ={x_displacement} px)")
    print(f"   Y range: {min(y_coords)} → {max(y_coords)} (Δ={y_displacement} px)")
    
    # Verify plot matches expectations
    if x_displacement > MIN_SPATIAL_DISPLACEMENT_X:
        print(f"\n✅ CONFIRMED: Significant spatial extent detected")
        print(f"   Track {best.track_id} shows substantial horizontal motion ({x_displacement}px)")
    else:
        print(f"\n⚠️  WARNING: Limited spatial extent")
        print(f"   Track {best.track_id} has minimal horizontal motion ({x_displacement}px)")
        print(f"   Expected >={MIN_SPATIAL_DISPLACEMENT_X}px for valid track")
    
    # Additional check for multi-eagle fragmentation scenario
    if best.length < 5 and len(gated_tracks) > 5:
        print(f"\n⚠️  FRAGMENTATION DETECTED:")
        print(f"   Best track is only {best.length} frames long")
        print(f"   But {len(gated_tracks)} tracks passed gating")
        print(f"   → Likely identity switching (see multi-eagle video analysis)")
        print(f"   → Consider implementing Hungarian algorithm")

print("\n" + "=" * 70)

print("\n✓ Validation complete")

## <font color = yellow> 7.15B: TRACK GAP DETAILED EXPORT

In [ ]:
# =================================================================================================
# STEP 7.15B: TRACK GAP DETAILED EXPORT + PLOTS (WORKING, SINGLE-CELL)
# =================================================================================================

import os, csv, math
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd
import matplotlib.pyplot as plt


print("\nSTEP 7.15B: TRACK GAP DETAILED EXPORT + PLOTS")

# ---------------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------------
PLOT_ENABLE = True
PLOT_TRACK_ID: Optional[int] = None      # None => use best gated track
PLOT_SHOW_INLINE = True                 # True => show plots inline; also saves PNGs
EXPORT_TOP_N = 3
MAX_ROWS_TERMINAL = 50



# OUT_DIR = str(output_dir) if "output_dir" in globals() else "."
# os.makedirs(OUT_DIR, exist_ok=True)
# CSV_PATH = os.path.join(OUT_DIR, "plotting_data_best_track.csv")


# from pathlib import Path
# from datetime import datetime

# ---- Resolve run-scoped output dirs (preferred) ----
if "LOG_DIR" in globals() and "IMG_DIR" in globals():
    LOG_DIR_PATH = Path(LOG_DIR)
    IMG_DIR_PATH = Path(IMG_DIR)
else:
    # Fallback: create a run-stamped folder under outputs/Step7_full
    PROJECT_ROOT = Path(r"C:\Axis_code_projects\OF_vs_Step7")
    MODULE_NAME = "Step7_full"
    RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
    video_stem = Path(VIDEO_PATH).stem if "VIDEO_PATH" in globals() else "unknown_clip"
    RUN_DIR = PROJECT_ROOT / "outputs" / MODULE_NAME / f"step7_{video_stem}_{RUN_TS}"
    LOG_DIR_PATH = RUN_DIR / "logs"
    IMG_DIR_PATH = RUN_DIR / "images"

LOG_DIR_PATH.mkdir(parents=True, exist_ok=True)
IMG_DIR_PATH.mkdir(parents=True, exist_ok=True)

CSV_PATH = str(LOG_DIR_PATH / "plotting_data_best_track.csv")

print(f"[7.15B] LOG_DIR: {LOG_DIR_PATH}")
print(f"[7.15B] IMG_DIR: {IMG_DIR_PATH}")
print(f"[7.15B] CSV_PATH: {CSV_PATH}")




# ---------------------------------------------------------------------------------
# PRINT CONFIG CHANGES
# ---------------------------------------------------------------------------------
config_changes = detect_config_changes()
print("\n" + "=" * 70)
print("⏱️ CONFIGURATION Change SUMMARY")
print("=" * 70)
print(config_changes["summary"])


# ---------------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------------
def _safe_track_id_choice(tracks: List["Track"]) -> Optional[int]:
    if not tracks:
        return None
    ranked = sorted(tracks, key=lambda t: (getattr(t, "confidence_score", 0.0) or 0.0), reverse=True)
    return ranked[0].track_id



# def _savefig(filename: str):
#     path = os.path.join(OUT_DIR, filename)
#     plt.tight_layout()
#     plt.savefig(path, dpi=140)
#     if PLOT_SHOW_INLINE:
#         plt.show()
#     else:
#         plt.close()
#     print(f"✓ Saved plot → {path}")

def _savefig(filename: str):
    path = str(IMG_DIR_PATH / filename)
    plt.tight_layout()
    plt.savefig(path, dpi=140)
    if PLOT_SHOW_INLINE:
        plt.show()
    else:
        plt.close()
    print(f"✓ Saved plot → {path}")

    

def _shade_gaps(ax, df_track: pd.DataFrame):
    """Shade contiguous GAP / predicted runs along x-axis."""
    if df_track.empty:
        return
    df_track = df_track.sort_values("frame")
    is_gap = (df_track["status"] == "GAP") | (df_track["is_predicted"] == True)
    frames = df_track["frame"].to_numpy()
    gap = is_gap.to_numpy()

    start = None
    for i in range(len(frames)):
        if gap[i] and start is None:
            start = frames[i]
        if (not gap[i]) and start is not None:
            end = frames[i - 1]
            ax.axvspan(start, end, alpha=0.15)
            start = None
    if start is not None:
        ax.axvspan(start, frames[-1], alpha=0.15)


def generate_frame_details(track: "Track") -> List[Dict[str, Any]]:
    """
    Frame-by-frame rows across the track span.
    Observed frames use compute_frame_motion_metrics().
    Gap frames use predict_position() + synthetic deltas.
    """
    details: List[Dict[str, Any]] = []
    if not track.frames:
        return details

    start_frame = track.frames[0]
    end_frame = track.frames[-1]

    observed_frames = set(track.frames)
    gap_frames = set(range(start_frame, end_frame + 1)) - observed_frames

    # cache metrics for observed frames
    motion_metrics_cache: Dict[int, Dict[str, Any]] = {}
    for i, frame_idx in enumerate(track.frames):
        prev_frame_idx = track.frames[i - 1] if i > 0 else None
        motion_metrics_cache[frame_idx] = compute_frame_motion_metrics(track, frame_idx, prev_frame_idx)

    prev_obs_frame: Optional[int] = None

    for frame_idx in range(start_frame, end_frame + 1):
        is_obs = frame_idx in observed_frames
        status = "OBSERVED" if is_obs else "GAP"

        if is_obs:
            x, y = track.observations[frame_idx]
            is_pred = False
        else:
            if prev_obs_frame is not None:
                x, y = predict_position(track, frame_idx)
                is_pred = True
            else:
                x, y = 0, 0
                is_pred = True

        if is_obs:
            metrics = motion_metrics_cache[frame_idx]
            bearing = metrics.get("bearing", float("nan"))
            dx = metrics.get("delta_x", 0.0)
            dy = metrics.get("delta_y", 0.0)
            accel = metrics.get("acceleration", 0.0)
        else:
            bearing = getattr(track, "mean_heading_deg", 0.0) or 0.0
            dx = 0.0
            dy = 0.0
            accel = 0.0
            if prev_obs_frame is not None and prev_obs_frame in track.observations:
                px, py = track.observations[prev_obs_frame]
                dx = x - px
                dy = y - py

        disp = math.hypot(dx, dy) if (dx != 0 or dy != 0) else 0.0

        details.append({
            "track_id": int(track.track_id),
            "frame": int(frame_idx),
            "status": status,
            "x": float(x),
            "y": float(y),
            "bearing": float(bearing) if bearing is not None else float("nan"),
            "delta_x": float(dx),
            "delta_y": float(dy),
            "displacement": float(disp),
            "acceleration": float(accel),
            "is_predicted": bool(is_pred) if not is_obs else False,
            "confidence_score": float(getattr(track, "confidence_score", float("nan"))),
        })

        if is_obs:
            prev_obs_frame = frame_idx

    return details


def print_track_details_table(track: "Track", df_track: pd.DataFrame, max_rows: int = 50):
    print("\n" + "=" * 70)
    print(f"BEST TRACK FRAME-BY-FRAME DETAILS (Track ID: {track.track_id})")
    print("=" * 70)
    print("\nFrame | Status    |    X  |    Y  | Bearing |  ΔX |  ΔY | Disp | Accel")
    print("------+-----------+-------+-------+---------+-----+-----+------+-------")

    total_rows = len(df_track)
    if total_rows == 0:
        print("   (no rows)")
        return

    # truncate
    if total_rows <= max_rows:
        df_show = df_track
        head_tail = False
    else:
        head = df_track.head(max_rows // 2)
        tail = df_track.tail(max_rows - len(head))
        df_show = pd.concat([head, tail], axis=0)
        head_tail = True

    def _fmt_row(r):
        star = "*" if r["is_predicted"] else " "
        x_str = f"{r['x']:5.0f}{star}"
        y_str = f"{r['y']:5.0f}{star}"
        b = r["bearing"]
        bearing_str = f"{b:5.1f}°{star}" if pd.notna(b) else "  --   "
        dx = r["delta_x"]
        dy = r["delta_y"]
        disp = r["displacement"]
        accel = r["acceleration"]

        dx_str = f"{dx:+3.0f}{star}" if dx != 0 else " --"
        dy_str = f"{dy:+3.0f}{star}" if dy != 0 else " --"
        disp_str = f"{disp:4.1f}{star}" if disp != 0 else " -- "
        accel_str = f"{accel:+5.2f}" if accel != 0 else "  --  "
        return f"{int(r['frame']):5d} | {r['status']:9s} | {x_str} | {y_str} | {bearing_str} | {dx_str} | {dy_str} | {disp_str} | {accel_str}"

    for _, r in df_show.iterrows():
        print(_fmt_row(r))

    if head_tail:
        print(f"  ... ({total_rows - len(df_show)} rows omitted) ...")

    print("\nLegend:")
    print("  * = Predicted value (gap frame)")
    print(f"\nTotal frames: {total_rows}")
    print(f"Observed: {int((df_track['status'] == 'OBSERVED').sum())}")
    print(f"Gaps: {int((df_track['status'] == 'GAP').sum())}")


def export_csv_with_headers(df_all: pd.DataFrame, video_filename: str, config_summary: str, out_csv: str):
    """Write a simple, clean CSV (NO weird commented sections)."""
    df_all = df_all.copy()
    df_all.insert(0, "video", video_filename)
    df_all.insert(1, "config_summary", config_summary.replace("\n", " | "))

    df_all.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"\n✓ Wrote CSV → {out_csv}")
    print(f"   rows = {len(df_all)}   tracks = {df_all['track_id'].nunique()}")


def plot_pack(df_all: pd.DataFrame, track_id: int):
    dft = df_all[df_all["track_id"] == track_id].copy().sort_values("frame")
    if dft.empty:
        print(f"❌ Plot pack: track_id {track_id} not in dataframe")
        return

    # Observed-only for the “did we REALLY reverse?” test
    d_obs = dft[(dft["status"] == "OBSERVED") & (dft["is_predicted"] == False)].copy().sort_values("frame")
    if len(d_obs) >= 2:
        d_obs["dx_obs_only"] = d_obs["x"].diff()
    else:
        d_obs["dx_obs_only"] = float("nan")

    # 1) X observed-only
    plt.figure()
    plt.title(f"Track {track_id}: X vs Frame (Observed only)")
    plt.xlabel("Frame")
    plt.ylabel("X (px)")
    plt.plot(d_obs["frame"], d_obs["x"])
    _savefig(f"plot_track{track_id}_X_observed.png")

    # 2) ΔX observed-only recomputed
    plt.figure()
    plt.title(f"Track {track_id}: ΔX vs Frame (Observed-only, recomputed)")
    plt.xlabel("Frame")
    plt.ylabel("ΔX (px per observed step)")
    plt.bar(d_obs["frame"], d_obs["dx_obs_only"])
    plt.axhline(0.0)
    _savefig(f"plot_track{track_id}_dX_observed_only.png")

    # 3) X full with gap shading
    plt.figure()
    ax = plt.gca()
    plt.title(f"Track {track_id}: X vs Frame (Observed + GAP rows)")
    plt.xlabel("Frame")
    plt.ylabel("X (px)")
    plt.plot(dft["frame"], dft["x"])
    _shade_gaps(ax, dft)
    _savefig(f"plot_track{track_id}_X_with_gap_shading.png")

    # 4) ΔX from detail rows, with gap shading
    plt.figure()
    ax = plt.gca()
    plt.title(f"Track {track_id}: ΔX vs Frame (detail delta_x, GAP shaded)")
    plt.xlabel("Frame")
    plt.ylabel("ΔX (px/frame)")
    plt.bar(dft["frame"], dft["delta_x"])
    plt.axhline(0.0)
    _shade_gaps(ax, dft)
    _savefig(f"plot_track{track_id}_dX_with_gap_shading.png")

    print("\n✅ Plot pack complete.")
    print("   Decisive reversal proof = plot_track*_dX_observed_only.png")


# ---------------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------------
if "gated_tracks" not in globals() or not gated_tracks:
    print("\n⚠️ No gated_tracks available. Nothing to export/plot.")
else:
    ranked_tracks = sorted(gated_tracks, key=lambda t: (getattr(t, "confidence_score", 0.0) or 0.0), reverse=True)
    top_tracks = ranked_tracks[:EXPORT_TOP_N]

    # Build detail rows in-memory (no CSV parsing required)
    all_rows: List[Dict[str, Any]] = []
    for t in top_tracks:
        all_rows.extend(generate_frame_details(t))

    df_all = pd.DataFrame(all_rows).sort_values(["track_id", "frame"]).reset_index(drop=True)

    # Terminal view (best track only)
    best_track = top_tracks[0]
    df_best = df_all[df_all["track_id"] == best_track.track_id].copy().sort_values("frame")
    print_track_details_table(best_track, df_best, max_rows=MAX_ROWS_TERMINAL)

    # Export a real CSV (clean format)
    video_filename = os.path.basename(VIDEO_PATH) if "VIDEO_PATH" in globals() else "unknown_video"
    export_csv_with_headers(df_all, video_filename, config_changes["summary"], CSV_PATH)

    # Plot
    if PLOT_ENABLE:
        tid = PLOT_TRACK_ID if PLOT_TRACK_ID is not None else _safe_track_id_choice(top_tracks)
        if tid is None:
            print("⚠️ Plot pack skipped: no track id available")
        else:
            plot_pack(df_all, tid)

print("\n" + "=" * 70)
print("STEP 7.15B complete.")


In [ ]:
type(all_tracks)


In [ ]:
for t in all_tracks:
    print(t.track_id, t.status, t.length if hasattr(t, "length") else None)


### <font color = lime> Filter / get the confirmed tracks

In [ ]:
for t in all_tracks:
    if t.status == "confirmed":
        print(t.track_id, t.length if hasattr(t, "length") else None)



In [ ]:
def status_rank(t):
    # Higher is better
    order = {"confirmed": 3, "low_density": 2, "too_short": 1}
    return order.get(t.status, 0)

best_track = max(
    all_tracks,
    key=lambda t: (status_rank(t), t.confidence_score, getattr(t, "length", 0))
)

best_track_id = best_track.track_id
print(f"Best: id & most frames :", best_track_id, best_track.status, best_track.length)


In [ ]:
best_track

## <font color = yellow> 7.15_R: Create/save output files (df) for R

In [ ]:
import pandas as pd

def track_to_df(track):
    rows = []

    # union of all frame indices present in observations or bboxes
    all_frames = sorted(
        set(track.observations.keys()) |
        set(track.bboxes.keys())
    )

    for f in all_frames:
        obs = track.observations.get(f, (None, None))
        bbox = track.bboxes.get(f, (None, None, None, None))

        rows.append({
            "track_id": track.track_id,
            "frame": f,
            "x": obs[0],
            "y": obs[1],
            "bbox_x": bbox[0],
            "bbox_y": bbox[1],
            "bbox_w": bbox[2],
            "bbox_h": bbox[3],
            "created_frame": track.created_frame,
            "last_observed_frame": track.last_observed_frame,
            "last_updated_frame": track.last_updated_frame,
            "miss_count": track.miss_count,
            "status": track.status,
            "is_gap": f in track.gap_frames,
            "was_interpolated": track.was_interpolated,
            "quality_score": track.quality_score,
            "confidence_score": track.confidence_score,
            "mean_y": track.mean_y,
            "min_y": track.min_y,
            "max_y": track.max_y,
        })

    return pd.DataFrame(rows)


In [ ]:
df_best_track = track_to_df(best_track)
df_best_track.to_csv(os.path.join(str(LOG_DIR), "best_track.csv"), index=False)


In [ ]:
import os
import csv

data = [
    {"track_id": 1, "length": 42, "density": 0.88},
    {"track_id": 2, "length": 17, "density": 0.55},
]


out_path = os.path.join(str(LOG_DIR), "R_output.csv")

with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=data[0].keys())
    writer.writeheader()
    writer.writerows(data)

print(f"✓ R_output saved → {out_path}")



## <font color = yellow> 7.16: A second bird

#### <font color = lime> What “trailing bird” means for this notebook:

    For this notebook’s first pass, we’ll define a trailing candidate as a track that:
    
    Direction match
    
    |heading(primary) − heading(candidate)| < MAX_HEADING_DIFF (e.g. 20–30°).
    
    Same lane / band
    
    |mean_y(primary) − mean_y(candidate)| < MAX_Y_OFFSET (e.g. 40 px).
    
    Temporal lag, not random overlap
    
    Candidate’s start frame is after primary’s start frame by between MIN_LAG_FRAMES and MAX_LAG_FRAMES
    (e.g. 5–80 frames).
    
    Their spans overlap or are adjacent enough that both exist for part of the clip.
    
    Spatial ordering consistent with direction
    
    If primary is flying left → right:
    
    candidate’s x is mostly left of primary’s x at matched times
    
    If primary is flying right → left:
    
    candidate’s x is mostly right of primary’s x.
    
    We don’t need perfect alignment, just a strong majority of frames.
    
    Reasonable quality
    
    candidate confidence >= some MIN_FOLLOWER_CONF (say 0.2–0.3)
    
    candidate length >= MIN_FOLLOWER_LENGTH (e.g. 8–10 frames).

In [ ]:
# The first bird
# primary = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)[0]


## <font color = yellow> 7.16a: Trailing-bird analysis config (optional)

#### <font color = lime> No interaction with other configs; just read these locally.

In [ ]:
# ======================================================================
# 7.16a – Trailing-bird analysis config (optional)
# ======================================================================

TRAILING_CFG = dict(
    enabled= True,              # master toggle
    max_heading_diff_deg=25.0, # how parallel the paths must be
    max_y_offset_px=40.0,      # same lane constraint
    min_lag_frames=5,          # follower should start at least this many frames later
    max_lag_frames=80,         # ...but not absurdly late
    min_follower_length=8,     # track must be long enough
    min_follower_conf=0.25,    # avoid tiny junk tracks
    min_same_side_fraction=0.7 # fraction of frames where follower is “behind” primary
)



# ======================================================================
# 7.16a – Trailing-bird analysis config (optional)
# ======================================================================

from typing import Dict, Any

TRAILING_CFG: Dict[str, Any] = dict(
    enabled= False,               # Master toggle for trailing-bird diagnostics

    # Geometric similarity
    max_heading_diff_deg=25.0,  # Max allowed heading difference (deg)
    max_y_offset_px=40.0,       # Max allowed mean Y separation (same lane)

    # Temporal relationship
    min_lag_frames=5,           # Follower should start at least this many frames later
    max_lag_frames=80,          # ...but not absurdly late

    # Track quality
    min_follower_length=8,      # Minimum length for follower track (frames)
    min_follower_conf=0.25,     # Minimum confidence for follower track

    # “Behind-ness” constraint
    min_same_side_fraction=0.7, # Fraction of overlapping frames where follower is behind

    # Overall acceptance threshold for composite follower score (0–1)
    min_follower_score=0.5,
)

print("\n[7.16a] Trailing-bird analysis config loaded.")
print("        TRAILING_CFG['enabled'] =", TRAILING_CFG["enabled"])


## <font color = yellow> 7.16b: Candidate Finder



In [ ]:
# ======================================================================
# 7.16b – Trailing-bird candidate finder (primary + follower analysis)
# ======================================================================



print("\n" + "=" * 70)
print("STEP 7.16 – TRAILING-BIRD CANDIDATE ANALYSIS")
print("=" * 70)

if not TRAILING_CFG.get("enabled", False):
    print("\n⏭️  Trailing-bird analysis disabled (TRAILING_CFG['enabled'] = False)")
else:
    if not gated_tracks or len(gated_tracks) < 2:
        print("\n⚠️  Trailing-bird analysis skipped:")
        print("    Need at least 2 gated tracks (have", len(gated_tracks) if gated_tracks else 0, ")")
    else:
        # -----------------------------------------------------------------
        # Helper functions
        # -----------------------------------------------------------------
        def get_primary_track(tracks: List["Track"]) -> "Track":
            ranked = sorted(tracks, key=lambda t: t.confidence_score, reverse=True)
            return ranked[0]

        def estimate_direction_sign(track: "Track") -> int:
            """Return +1 for left→right, -1 for right→left, 0 if ambiguous."""
            if not track.frames:
                return 0
            f0 = track.frames[0]
            f1 = track.frames[-1]
            p0 = track.observations.get(f0)
            p1 = track.observations.get(f1)
            if not p0 or not p1:
                return 0
            dx = p1[0] - p0[0]
            if abs(dx) < 1e-3:
                return 0
            return 1 if dx > 0 else -1

        def heading_diff_deg(a: "Track", b: "Track") -> float:
            ha = a.mean_heading_deg
            hb = b.mean_heading_deg
            if ha is None or hb is None:
                return 180.0
            d = abs(ha - hb) % 360.0
            return d if d <= 180.0 else 360.0 - d

        def mean_y_offset(a: "Track", b: "Track") -> float:
            if a.mean_y is None or b.mean_y is None:
                return 1e6
            return abs(a.mean_y - b.mean_y)

        def lag_frames(primary: "Track", follower: "Track") -> int:
            return follower.frames[0] - primary.frames[0]

        def overlap_frames(primary: "Track", follower: "Track") -> List[int]:
            return sorted(set(primary.frames) & set(follower.frames))

        def trailing_fraction(
            primary: "Track",
            follower: "Track",
            direction_sign: int,
            frames: List[int],
        ) -> float:
            """Fraction of frames where follower is behind primary along X."""
            if direction_sign == 0 or not frames:
                return 0.0
            trailing_count = 0
            valid = 0
            for f in frames:
                p_pos = primary.observations.get(f)
                q_pos = follower.observations.get(f)
                if not p_pos or not q_pos:
                    continue
                valid += 1
                x_p = p_pos[0]
                x_q = q_pos[0]
                if direction_sign > 0:  # left→right
                    if x_q < x_p:
                        trailing_count += 1
                else:  # right→left
                    if x_q > x_p:
                        trailing_count += 1
            if valid == 0:
                return 0.0
            return trailing_count / valid

        def normalize_clamped(value: float, max_value: float) -> float:
            """Map 0..max_value to 1..0 linearly, clip to [0,1]."""
            if max_value <= 0:
                return 0.0
            r = 1.0 - (value / max_value)
            return max(0.0, min(1.0, r))

        def lag_score(
            lag: int,
            min_lag: int,
            max_lag: int,
        ) -> float:
            """Score lag within [min_lag, max_lag] range (1 in range center, 0 outside)."""
            if lag < min_lag or lag > max_lag or max_lag <= min_lag:
                return 0.0
            mid = 0.5 * (min_lag + max_lag)
            half_range = 0.5 * (max_lag - min_lag)
            if half_range <= 0:
                return 1.0
            d = abs(lag - mid)
            r = 1.0 - (d / half_range)
            return max(0.0, min(1.0, r))

        # -----------------------------------------------------------------
        # Primary track and direction
        # -----------------------------------------------------------------
        primary = get_primary_track(gated_tracks)
        direction_sign = estimate_direction_sign(primary)

        print(f"\nPrimary (best) track: ID={primary.track_id}")
        print(f"   Frames: {primary.frames[0]} → {primary.frames[-1]}")
        print(f"   Length: {primary.length}  |  Conf: {primary.confidence_score:.3f}")
        print(
            f"   Mean heading: {primary.mean_heading_deg:.1f}°"
            if primary.mean_heading_deg is not None else "   Mean heading: (none)"
        )
        print(
            f"   Mean Y: {primary.mean_y:.1f}px"
            if primary.mean_y is not None else "   Mean Y: (none)"
        )
        if direction_sign == 0:
            print("\n⚠️  Direction ambiguous (little net X motion); "
                  "trailing-bird check will use weaker evidence.")
        else:
            direction_str = "left→right" if direction_sign > 0 else "right→left"
            print(f"   Estimated horizontal direction: {direction_str}")

        cfg = TRAILING_CFG
        min_lag = cfg["min_lag_frames"]
        max_lag = cfg["max_lag_frames"]

        # -----------------------------------------------------------------
        # Evaluate candidates
        # -----------------------------------------------------------------
        candidate_rows: List[Dict[str, Any]] = []
        best_candidate: Optional["Track"] = None
        best_score: float = -1.0

        for cand in gated_tracks:
            if cand.track_id == primary.track_id:
                continue  # skip self

            # Quality pre-filter
            if cand.length < cfg["min_follower_length"]:
                reason = "too short"
                base_row = dict(track_id=cand.track_id, reason=reason)
            elif cand.confidence_score < cfg["min_follower_conf"]:
                reason = "low confidence"
                base_row = dict(track_id=cand.track_id, reason=reason)
            else:
                base_row = dict(track_id=cand.track_id, reason=None)

            # Core metrics
            h_diff = heading_diff_deg(primary, cand)
            y_off = mean_y_offset(primary, cand)
            lag = lag_frames(primary, cand)
            ov_frames = overlap_frames(primary, cand)
            same_side = trailing_fraction(primary, cand, direction_sign, ov_frames)

            # Scores
            heading_score = normalize_clamped(
                h_diff, cfg["max_heading_diff_deg"]
            )
            lane_score = normalize_clamped(
                y_off, cfg["max_y_offset_px"]
            )
            lag_s = lag_score(lag, min_lag, max_lag)
            same_side_s = same_side  # already 0–1

            # Composite follower score
            # (weights can be tuned later)
            follower_score = (
                0.3 * heading_score +
                0.3 * lane_score +
                0.2 * lag_s +
                0.2 * same_side_s
            )

            row = dict(
                track_id=cand.track_id,
                length=cand.length,
                confidence=cand.confidence_score,
                heading_diff=h_diff,
                y_offset=y_off,
                lag_frames=lag,
                overlap_count=len(ov_frames),
                same_side_fraction=same_side,
                heading_score=heading_score,
                lane_score=lane_score,
                lag_score=lag_s,
                same_side_score=same_side_s,
                follower_score=follower_score,
                reason=base_row["reason"],
            )
            candidate_rows.append(row)

            # Eligibility for "best" follower
            passes_basic = (
                row["reason"] is None and
                lag >= min_lag and lag <= max_lag
            )

            if passes_basic and follower_score > best_score:
                best_score = follower_score
                best_candidate = cand

        # -----------------------------------------------------------------
        # Persist results to STEP7_ARTIFACTS
        # -----------------------------------------------------------------
        trailing_result: Dict[str, Any] = dict(
            primary_id=primary.track_id,
            primary_direction_sign=direction_sign,
            candidates=candidate_rows,
            follower_id=None,
            follower_score=None,
            accepted=False,
            acceptance_reasons=[],
        )

        # Decide final follower acceptance
        if best_candidate is not None:
            # Find metrics row for best candidate
            best_row = next(
                r for r in candidate_rows
                if r["track_id"] == best_candidate.track_id
            )

            reasons = []

            if best_row["heading_diff"] > cfg["max_heading_diff_deg"]:
                reasons.append("heading_diff too large")
            if best_row["y_offset"] > cfg["max_y_offset_px"]:
                reasons.append("Y offset too large")
            if not (cfg["min_lag_frames"] <= best_row["lag_frames"] <= cfg["max_lag_frames"]):
                reasons.append("lag outside allowed range")
            if best_row["same_side_fraction"] < cfg["min_same_side_fraction"]:
                reasons.append("follower not consistently behind primary")
            if best_row["follower_score"] < cfg["min_follower_score"]:
                reasons.append("composite follower_score below threshold")

            accepted = (len(reasons) == 0)

            trailing_result["follower_id"] = best_candidate.track_id
            trailing_result["follower_score"] = best_row["follower_score"]
            trailing_result["accepted"] = accepted
            trailing_result["acceptance_reasons"] = reasons

        # Store in shared artifacts bucket
        if "STEP7_ARTIFACTS" in globals():
            STEP7_ARTIFACTS["trailing_analysis"] = trailing_result
        else:
            print("\n⚠️  STEP7_ARTIFACTS not found in globals() – result not stored.")

        print("\n[7.16b] Trailing-bird candidate analysis complete.")
        print(f"        Candidates evaluated: {len(candidate_rows)}")


## <font color = yellow> 🧩 Cell 7.16c – Trailing-bird diagnostic report

In [ ]:
# ======================================================================
# 7.16c – Trailing-bird diagnostic report (human-readable)
# ======================================================================

print("\n" + "=" * 70)
print("STEP 7.16 – TRAILING-BIRD DIAGNOSTIC REPORT")
print("=" * 70)

if not TRAILING_CFG.get("enabled", False):
    print("\n⏭️  Trailing-bird analysis disabled; no report generated.")
else:
    if "STEP7_ARTIFACTS" not in globals():
        print("\n⚠️  STEP7_ARTIFACTS not available; run Step 7.5a first.")
    else:
        result = STEP7_ARTIFACTS.get("trailing_analysis")

        if not result:
            print("\n⚠️  No trailing_analysis result found.")
            print("    (Run 7.16b before this cell.)")
        else:
            primary_id = result["primary_id"]
            follower_id = result["follower_id"]
            candidates = result["candidates"]

            print(f"\nPrimary track ID: {primary_id}")
            print(f"Candidates evaluated: {len(candidates)}")

            # Per-candidate summary table
            if candidates:
                print("\nCandidate summary:")
                print("   {:>5} | {:>4} | {:>6} | {:>7} | {:>6} | {:>5} | {:>5} | {:>6}".format(
                    "ID", "Len", "Conf", "hdgΔ°", "YΔpx", "Lag", "Over", "Score"
                ))
                print("   " + "-"*5 + "-+-" + "-"*4 + "-+-" + "-"*6 + "-+-" +
                      "-"*7 + "-+-" + "-"*6 + "-+-" + "-"*5 + "-+-" + "-"*6)
                for row in sorted(candidates, key=lambda r: r["follower_score"], reverse=True):
                    print("   {track_id:5d} | {length:4d} | {confidence:6.3f} | "
                          "{heading_diff:7.1f} | {y_offset:6.1f} | "
                          "{lag_frames:5d} | {overlap_count:5d} | {follower_score:6.3f}".format(
                              **row
                          ))

            print("\n" + "-" * 70)

            if follower_id is None:
                print("Conclusion:")
                print("   ❌ No trailing-bird candidate selected.")
                print("   Most likely single-eagle clip or fragmented secondary tracks.")
            else:
                print("Conclusion:")
                if result["accepted"]:
                    print(f"   ✅ Trailing-bird detected: Track {follower_id}")
                    print(f"   Follower score: {result['follower_score']:.3f}")
                else:
                    print(f"   ❌ Best candidate (Track {follower_id}) did not meet thresholds.")
                    print(f"   Follower score: {result['follower_score']:.3f}")
                    if result["acceptance_reasons"]:
                        print("\n   Issues:")
                        for r in result["acceptance_reasons"]:
                            print(f"      - {r}")

print("\n" + "=" * 70)


In [ ]:
stopper: Archieves Beyond

## <font color = teal> 7.20 Archieves: dead / uni-purpose code

## <font color = yellow> 7.20a: Diagnostic tool (Used to eval if/when connecting 2 long tracks)

#### <font color = lime> Focus on best and second-best tracks ... ask the question:
#### <font color = yellow> Deep debug tool for “should I stitch second-best to best?

- “Could these actually be two halves of the same bird’s path?”
- Looks at the gated tracks (already scored / filtered) to determine if a longer track should exist.
- Explore that question with and without retro-interpolation, and give a recommendation.
- Legacy / advanced diagnostic – track stitching experiment.

In [ ]:
# ======================================================================
# STEP 7.20a — LEGACY / ADVANCED DIAGNOSTIC
# TRACK STITCHING ANALYSIS (WITH RETRO-INTERPOLATION UPGRADE)
# ----------------------------------------------------------------------
# Status: Optional research diagnostic
# Purpose: Examines whether two top tracks can be stitched together
#          using predicted-to-actual spatial gap, heading consistency,
#          Y-lane consistency, and retro-interpolation smoothing.
# Impact:  Does NOT modify the pipeline; read-only.
# ======================================================================


# --- User Toggle ---
ENABLE_STITCHING_DIAGNOSTIC = True  # Set to False to skip this cell

print("\n" + "=" * 70)
print("TRACK STITCHING DIAGNOSTIC (WITH RETRO-INTERPOLATION)")
print("=" * 70)

if not ENABLE_STITCHING_DIAGNOSTIC:
    print("\n⏭️  Stitching diagnostic disabled (ENABLE_STITCHING_DIAGNOSTIC = False)")
else:
    if not gated_tracks:
        print("\n❌ No gated tracks available for stitching analysis")
    
    elif len(gated_tracks) < 2:
        print("\n⚠️  Only one gated track - no stitching candidates")
        print(f"   Track ID: {gated_tracks[0].track_id}")
        print(f"   Frames: {gated_tracks[0].frames[0]}→{gated_tracks[0].frames[-1]}")
        print(f"   Observed: {gated_tracks[0].length}, Confidence: {gated_tracks[0].confidence_score:.3f}")
    
    else:
        # =============================================================================
        # FUNCTION DEFINITIONS
        # =============================================================================
        
        def retro_interpolate_track(track: Track) -> Track:
            """
            Fill gaps in track with linearly interpolated positions.
            
            Process:
            --------
            1. Identify gap frames (missing observations between first and last frame)
            2. For each gap, compute interpolated position using linear interpolation
               between surrounding observations
            3. Add interpolated positions to track.observations (marked as estimated)
            4. Recompute observation density and gap count
            
            Args:
                track: Track object (will be modified in-place)
            
            Returns:
                Modified track with interpolated positions
            
            Example:
            --------
            Before: frames [10, 11, 14, 15] (gap at 12, 13)
                    positions [(100,200), (110,205), (130,215), (140,220)]
            
            After:  frames [10, 11, 12, 13, 14, 15]
                    positions [(100,200), (110,205), (120,210), (125,212), (130,215), (140,220)]
                                                      ^^^^^^^^  ^^^^^^^^  (interpolated)
            """
            
            if not track.observations or len(track.frames) < 2:
                return track  # Can't interpolate single-point track
            
            # Get all frames in span
            first_frame = track.frames[0]
            last_frame = track.frames[-1]
            all_frames_in_span = set(range(first_frame, last_frame + 1))
            
            # Identify gap frames
            observed_frames = set(track.frames)
            gap_frames = sorted(all_frames_in_span - observed_frames)
            
            if not gap_frames:
                return track  # No gaps to fill
            
            # For each gap frame, interpolate position
            for gap_frame in gap_frames:
                # Find surrounding observed frames
                prev_observed = [f for f in track.frames if f < gap_frame]
                next_observed = [f for f in track.frames if f > gap_frame]
                
                if not prev_observed or not next_observed:
                    continue  # Can't interpolate at edges
                
                # Get closest surrounding observations
                frame_before = max(prev_observed)
                frame_after = min(next_observed)
                
                pos_before = track.observations[frame_before]
                pos_after = track.observations[frame_after]
                
                # Linear interpolation
                dt_total = frame_after - frame_before
                dt_to_gap = gap_frame - frame_before
                
                if dt_total == 0:
                    continue  # Should never happen
                
                t = dt_to_gap / dt_total
                
                interpolated_x = int(pos_before[0] + t * (pos_after[0] - pos_before[0]))
                interpolated_y = int(pos_before[1] + t * (pos_after[1] - pos_before[1]))
                
                # Add interpolated position to observations
                track.observations[gap_frame] = (interpolated_x, interpolated_y)
                
                # Create synthetic bounding box
                if frame_before in track.bboxes and frame_after in track.bboxes:
                    bbox_before = track.bboxes[frame_before]
                    bbox_after = track.bboxes[frame_after]
                    
                    bbox_x = int(bbox_before[0] + t * (bbox_after[0] - bbox_before[0]))
                    bbox_y = int(bbox_before[1] + t * (bbox_after[1] - bbox_before[1]))
                    bbox_w = int(bbox_before[2] + t * (bbox_after[2] - bbox_before[2]))
                    bbox_h = int(bbox_before[3] + t * (bbox_after[3] - bbox_before[3]))
                    
                    track.bboxes[gap_frame] = (bbox_x, bbox_y, bbox_w, bbox_h)
            
            return track
        
        
        def analyze_stitching_candidates(track_A: Track, track_B: Track, 
                                         use_interpolation: bool = True) -> Dict[str, Any]:
            """
            Analyze compatibility for stitching two tracks.
            
            Args:
                track_A: First track (earlier in time)
                track_B: Second track (later in time)
                use_interpolation: If True, apply retro-interpolation before analysis
            
            Returns:
                Dict with compatibility metrics and recommendation
            """
            
            # Apply retro-interpolation if enabled
            if use_interpolation:
                import copy
                track_A = copy.deepcopy(track_A)
                track_B = copy.deepcopy(track_B)
                track_A = retro_interpolate_track(track_A)
                track_B = retro_interpolate_track(track_B)
            
            # Check temporal ordering
            if track_A.frames[-1] >= track_B.frames[0]:
                return {
                    "compatible": False,
                    "reason": "Tracks overlap",
                    "details": None
                }
            
            # Compute gap between tracks
            gap_frames = track_B.frames[0] - track_A.frames[-1] - 1
            
            # Compute velocity from Track A
            positions_A = [track_A.observations[f] for f in track_A.frames]
            frames_A = track_A.frames
            
            total_vx = 0
            total_vy = 0
            count = 0
            
            for i in range(1, len(frames_A)):
                dx = positions_A[i][0] - positions_A[i-1][0]
                dy = positions_A[i][1] - positions_A[i-1][1]
                dt = frames_A[i] - frames_A[i-1]
                
                if dt > 0:
                    vx = dx / dt
                    vy = dy / dt
                    total_vx += vx
                    total_vy += vy
                    count += 1
            
            if count > 0:
                avg_vx = total_vx / count
                avg_vy = total_vy / count
            else:
                avg_vx = 0.0
                avg_vy = 0.0
            
            # Predict position at Track B start
            last_pos = positions_A[-1]
            last_frame = frames_A[-1]
            target_frame = track_B.frames[0]
            dt_pred = target_frame - last_frame
            
            predicted_x = last_pos[0] + avg_vx * dt_pred
            predicted_y = last_pos[1] + avg_vy * dt_pred
            
            actual_pos = track_B.observations[track_B.frames[0]]
            
            spatial_gap = math.hypot(
                actual_pos[0] - predicted_x,
                actual_pos[1] - predicted_y
            )
            
            # Compute heading difference
            heading_diff = abs(track_A.mean_heading_deg - track_B.mean_heading_deg) if (
                track_A.mean_heading_deg is not None and track_B.mean_heading_deg is not None
            ) else 180.0
            
            # Compute Y-position difference
            y_diff = abs(track_A.mean_y - track_B.mean_y) if (
                track_A.mean_y is not None and track_B.mean_y is not None
            ) else 1000.0
            
            # Thresholds
            SPATIAL_THRESHOLD = 150
            HEADING_THRESHOLD = 30
            Y_THRESHOLD = 80
            MAX_TEMPORAL_GAP = 15
            
            spatial_ok = spatial_gap < SPATIAL_THRESHOLD
            heading_ok = heading_diff < HEADING_THRESHOLD
            y_ok = y_diff < Y_THRESHOLD
            temporal_ok = gap_frames <= MAX_TEMPORAL_GAP
            
            compatible = spatial_ok and heading_ok and y_ok and temporal_ok
            
            return {
                "compatible": compatible,
                "reason": "All checks passed" if compatible else "Failed compatibility checks",
                "details": {
                    "gap_frames": gap_frames,
                    "velocity": (avg_vx, avg_vy),
                    "predicted_position": (predicted_x, predicted_y),
                    "actual_position": actual_pos,
                    "spatial_gap": spatial_gap,
                    "heading_diff": heading_diff,
                    "y_diff": y_diff,
                    "checks": {
                        "spatial": spatial_ok,
                        "heading": heading_ok,
                        "y_position": y_ok,
                        "temporal": temporal_ok
                    },
                    "thresholds": {
                        "spatial": SPATIAL_THRESHOLD,
                        "heading": HEADING_THRESHOLD,
                        "y_position": Y_THRESHOLD,
                        "temporal": MAX_TEMPORAL_GAP
                    },
                    "interpolation_applied": use_interpolation
                }
            }
        
        
        def compute_velocity_diagnostic(track: Track) -> Tuple[float, float]:
            """
            DIAGNOSTIC: Inspect velocity computation for a track.
            
            Returns:
                (avg_vx, avg_vy) - Average velocity in px/frame
            """
            
            print(f"\n🔍 Track {track.track_id} Velocity Diagnostic:")
            print(f"   Frames: {track.frames[:10]}{'...' if len(track.frames) > 10 else ''}")
            print(f"   Observations: {len(track.observations)}")
            print(f"   Current velocity: {track.current_velocity}")
            
            if track.current_velocity == (0.0, 0.0) or track.current_velocity is None:
                print("\n   ⚠️  WARNING: Zero velocity detected!")
                print("   → Prediction will not extrapolate (stays at last position)")
            
            # Manual velocity calculation
            if len(track.frames) >= 2:
                print(f"\n📐 Manual Velocity Calculation:")
                
                positions = [track.observations[f] for f in track.frames]
                frames = track.frames
                
                total_vx = 0
                total_vy = 0
                count = 0
                
                for i in range(1, min(len(frames), 6)):  # Show first 5 steps
                    dx = positions[i][0] - positions[i-1][0]
                    dy = positions[i][1] - positions[i-1][1]
                    dt = frames[i] - frames[i-1]
                    
                    if dt > 0:
                        vx = dx / dt
                        vy = dy / dt
                        total_vx += vx
                        total_vy += vy
                        count += 1
                        print(f"   Frame {frames[i-1]}→{frames[i]}: v=({vx:.2f}, {vy:.2f}) px/frame")
                
                # Compute over ALL frames
                total_vx = 0
                total_vy = 0
                count = 0
                for i in range(1, len(frames)):
                    dx = positions[i][0] - positions[i-1][0]
                    dy = positions[i][1] - positions[i-1][1]
                    dt = frames[i] - frames[i-1]
                    if dt > 0:
                        total_vx += dx / dt
                        total_vy += dy / dt
                        count += 1
                
                if count > 0:
                    avg_vx = total_vx / count
                    avg_vy = total_vy / count
                    print(f"\n   Average velocity (all frames): ({avg_vx:.2f}, {avg_vy:.2f}) px/frame")
                    return (avg_vx, avg_vy)
            
            return (0.0, 0.0)
        
        
        def diagnose_stitching_velocity(track_a: Track, track_b: Track) -> None:
            """
            DIAGNOSTIC: Check why spatial gap is large between two tracks.
            """
            
            print("\n" + "=" * 70)
            print("VELOCITY DIAGNOSTIC FOR STITCHING")
            print("=" * 70)
            
            # Compute velocities
            print(f"\n📊 Track {track_a.track_id} (Predecessor):")
            vel_a_manual = compute_velocity_diagnostic(track_a)
            
            print(f"\n📊 Track {track_b.track_id} (Successor):")
            vel_b_manual = compute_velocity_diagnostic(track_b)
            
            # Predict Track A → Track B
            last_frame_a = track_a.frames[-1]
            first_frame_b = track_b.frames[0]
            dt_pred = first_frame_b - last_frame_a
            
            print(f"\n🎯 Prediction from Track {track_a.track_id} to Track {track_b.track_id}:")
            print(f"   Time gap: {dt_pred} frames")
            
            # Method 1: Stored velocity
            if track_a.current_velocity is not None:
                vx_stored, vy_stored = track_a.current_velocity
                last_pos_a = track_a.observations[last_frame_a]
                
                pred_x_stored = last_pos_a[0] + vx_stored * dt_pred
                pred_y_stored = last_pos_a[1] + vy_stored * dt_pred
                
                print(f"\n   Method 1: Using track.current_velocity")
                print(f"      Velocity: ({vx_stored:.2f}, {vy_stored:.2f})")
                print(f"      Predicted at frame {first_frame_b}: ({pred_x_stored:.0f}, {pred_y_stored:.0f})")
            else:
                print(f"\n   Method 1: FAILED (current_velocity is None)")
                pred_x_stored, pred_y_stored = track_a.observations[last_frame_a]
            
            # Method 2: Manual velocity
            vx_manual, vy_manual = vel_a_manual
            last_pos_a = track_a.observations[last_frame_a]
            
            pred_x_manual = last_pos_a[0] + vx_manual * dt_pred
            pred_y_manual = last_pos_a[1] + vy_manual * dt_pred
            
            print(f"\n   Method 2: Using manually computed velocity")
            print(f"      Velocity: ({vx_manual:.2f}, {vy_manual:.2f})")
            print(f"      Predicted at frame {first_frame_b}: ({pred_x_manual:.0f}, {pred_y_manual:.0f})")
            
            # Actual position
            actual_pos_b = track_b.observations[first_frame_b]
            print(f"\n   Actual position at frame {first_frame_b}: {actual_pos_b}")
            
            # Spatial gaps
            gap_stored = math.hypot(actual_pos_b[0] - pred_x_stored, actual_pos_b[1] - pred_y_stored)
            gap_manual = math.hypot(actual_pos_b[0] - pred_x_manual, actual_pos_b[1] - pred_y_manual)
            
            print(f"\n📏 Spatial Gaps:")
            print(f"   Using stored velocity: {gap_stored:.1f} px")
            print(f"   Using manual velocity: {gap_manual:.1f} px")
            
            if gap_manual < 100:
                print(f"\n   ✅ With corrected velocity, gap is ACCEPTABLE (<100px)")
                print(f"   → Root cause: track.current_velocity was incorrect")
                print(f"   → Fix: update_velocity() in Cell 7.4")
            else:
                print(f"\n   ❌ Even with corrected velocity, gap TOO LARGE (>100px)")
                print(f"   → Tracks likely represent DIFFERENT objects")
            
            print("\n" + "=" * 70)
        
        # =============================================================================
        # PRE-STITCHING DIAGNOSTIC
        # =============================================================================
        
        print("\n" + "=" * 70)
        print("PRE-STITCHING DIAGNOSTIC")
        print("=" * 70)
        
        ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
        track_best = ranked[0]
        track_second = ranked[1] if len(ranked) > 1 else None
        
        print(f"\n🔍 Best Track (ID {track_best.track_id}):")
        print(f"   Frame range: {track_best.frames[0]}→{track_best.frames[-1]}")
        print(f"   Observed frames: {track_best.length}")
        print(f"   Gap frames: {track_best.gap_count}")
        
        if track_second:
            print(f"\n🔍 Second Best Track (ID {track_second.track_id}):")
            print(f"   Frame range: {track_second.frames[0]}→{track_second.frames[-1]}")
            print(f"   Observed frames: {track_second.length}")
            print(f"   Gap frames: {track_second.gap_count}")
            
            # Check overlap
            frames_best = set(track_best.frames)
            frames_second = set(track_second.frames)
            overlap = frames_best & frames_second
            
            if overlap:
                print(f"\n⚠️  OVERLAP DETECTED:")
                print(f"   {len(overlap)} frames where both tracks exist")
                print(f"   Overlap frames: {sorted(list(overlap))[:20]}")
                
                sample_frames = sorted(list(overlap))[:5]
                print(f"\n   Spatial separation in overlap region:")
                
                for frame in sample_frames:
                    pos_best = track_best.observations.get(frame)
                    pos_second = track_second.observations.get(frame)
                    
                    if pos_best and pos_second:
                        dist = math.hypot(pos_second[0] - pos_best[0], pos_second[1] - pos_best[1])
                        print(f"      Frame {frame}: distance={dist:.1f}px")
                
                print(f"\n   → If distance < 15px: Likely duplicate (same object)")
                print(f"   → If distance > 100px: Different objects")
            
            else:
                print(f"\n✅ NO OVERLAP - Tracks are sequential")
                gap_between = track_second.frames[0] - track_best.frames[-1] - 1
                print(f"   Gap between tracks: {gap_between} frames")
                print(f"   → Stitching may be possible")
        
        # =============================================================================
        # MAIN STITCHING ANALYSIS
        # =============================================================================
        
        if len(ranked) >= 2:
            track_A = ranked[1]  # Second best (earlier)
            track_B = ranked[0]  # Best (later)
            
            print("\n" + "=" * 70)
            print("STITCHING ANALYSIS")
            print("=" * 70)
            
            print(f"\n📊 Track {track_A.track_id} (BEFORE Interpolation):")
            print(f"   Span: {track_A.frames[0]}→{track_A.frames[-1]} ({track_A.length} observed)")
            print(f"   Density: {track_A.observation_density:.1%}, Gaps: {track_A.gap_count}")
            print(f"   Mean Y: {track_A.mean_y:.1f}, Heading: {track_A.mean_heading_deg:.1f}°")
            
            print(f"\n📊 Track {track_B.track_id} (BEFORE Interpolation):")
            print(f"   Span: {track_B.frames[0]}→{track_B.frames[-1]} ({track_B.length} observed)")
            print(f"   Density: {track_B.observation_density:.1%}, Gaps: {track_B.gap_count}")
            print(f"   Mean Y: {track_B.mean_y:.1f}, Heading: {track_B.mean_heading_deg:.1f}°")
            
            # Without interpolation
            print("\n" + "-" * 70)
            print("ANALYSIS WITHOUT INTERPOLATION")
            print("-" * 70)
            
            result_no_interp = analyze_stitching_candidates(track_A, track_B, use_interpolation=False)
            
            if result_no_interp["compatible"]:
                print(f"\n✅ COMPATIBLE (without interpolation)")
            else:
                print(f"\n❌ NOT COMPATIBLE (without interpolation)")
                print(f"   Reason: {result_no_interp['reason']}")
            
            if result_no_interp["details"]:
                details = result_no_interp["details"]
                print(f"\n🔗 METRICS:")
                print(f"   Spatial gap: {details['spatial_gap']:.1f} px (threshold: {details['thresholds']['spatial']})")
                print(f"   Heading diff: {details['heading_diff']:.1f}° (threshold: {details['thresholds']['heading']})")
                print(f"   Y diff: {details['y_diff']:.1f} px (threshold: {details['thresholds']['y_position']})")
                
                print(f"\n✅ CHECKS:")
                for check_name, passed in details['checks'].items():
                    status = "✅ PASS" if passed else "❌ FAIL"
                    print(f"   {check_name.capitalize():12s}: {status}")
                
                # Velocity diagnostic if spatial gap large
                if details['spatial_gap'] > 100:
                    diagnose_stitching_velocity(track_A, track_B)
            
            # With interpolation
            print("\n" + "-" * 70)
            print("ANALYSIS WITH RETRO-INTERPOLATION")
            print("-" * 70)
            
            result_with_interp = analyze_stitching_candidates(track_A, track_B, use_interpolation=True)
            
            if result_with_interp["compatible"]:
                print(f"\n✅ ✅ ✅ COMPATIBLE (with interpolation) ✅ ✅ ✅")
                
                print(f"\n📦 STITCHED TRACK PROJECTION:")
                import copy
                track_A_interp = copy.deepcopy(track_A)
                track_B_interp = copy.deepcopy(track_B)
                track_A_interp = retro_interpolate_track(track_A_interp)
                track_B_interp = retro_interpolate_track(track_B_interp)
                
                combined_observed = len(track_A_interp.frames) + len(track_B_interp.frames)
                combined_span = track_B_interp.frames[-1] - track_A_interp.frames[0] + 1
                
                print(f"   Combined observed: {combined_observed} frames")
                print(f"   Combined span: {combined_span} frames")
                print(f"   Combined density: {combined_observed/combined_span:.1%}")
                
                print(f"\n💡 RECOMMENDATION:")
                print(f"   ✅ Stitch Track {track_A.track_id} → Track {track_B.track_id}")
                print(f"   ✅ Apply retro-interpolation to fill internal gaps")
            else:
                print(f"\n❌ NOT COMPATIBLE (even with interpolation)")
                print(f"   Reason: {result_with_interp['reason']}")

print("\n" + "=" * 70)

## <font color = yellow> 7.20b: VERIFY PLOT CORRESPONDS TO BEST TRACK - deprecated

In [ ]:
# =================================================================================================
# 7.20b: VERIFY PLOT CORRESPONDS TO BEST TRACK - deprecated

# =================================================================================================

# print("=" * 70)

# print("PLOT VALIDATION")
# print("=" * 70)

# # Get best track
# ranked = sorted(gated_tracks, key=lambda t: t.confidence_score, reverse=True)
# best = ranked[0]

# print(f"\n🏆 Best Track from Cell 7.13:")
# print(f"   Track ID: {best.track_id}")
# print(f"   Frames: {best.frames[0]} → {best.frames[-1]}")
# print(f"   Observed: {best.length} frames")
# print(f"   Span: {best.span} frames")
# print(f"   Confidence: {best.confidence_score:.3f}")

# print(f"\n📊 Plot shows:")
# print(f"   Track ID: T0")
# print(f"   Confidence: 0.95")

# # Check spatial extent
# positions = [best.observations[f] for f in best.frames]
# x_coords = [p[0] for p in positions]
# y_coords = [p[1] for p in positions]

# print(f"\n📐 Spatial Extent:")
# print(f"   X range: {min(x_coords)} → {max(x_coords)} (Δ={max(x_coords)-min(x_coords)} px)")
# print(f"   Y range: {min(y_coords)} → {max(y_coords)} (Δ={max(y_coords)-min(y_coords)} px)")

# print(f"\n✅ Plot trajectory matches:")
# print(f"   Expected X span: ~200-400 px")
# print(f"   Expected Y span: ~200-250 px")
# print(f"   Observed X: {min(x_coords)}-{max(x_coords)} px")
# print(f"   Observed Y: {min(y_coords)}-{max(y_coords)} px")

# if max(x_coords) - min(x_coords) > 150:  # Significant X displacement
#     print(f"\n   ✅ CONFIRMED: Plot shows best track (Track {best.track_id})")
# else:
#     print(f"\n   ⚠️  WARNING: Spatial extent seems small for plotted trajectory")

# print("=" * 70)

In [ ]:
# calls the full pipeline (again)
# notebook tracking is NO LONGER calling update_velocity() (deprecated)

def update_velocity(track: Track,
                    current_frame: int,
                    centroid: Tuple[float, float]) -> None:
    """
    Legacy hook for velocity-based prediction.

    In the current simplified Step 7 design we are NOT using a smoothed
    velocity / acceleration model. Association is based on nearest-neighbor
    to the last observed centroid, plus MAX_MISS for temporal windowing.

    This function is intentionally a no-op and only exists so that
    build_tracks_standard(...) can safely call update_velocity(...)
    without raising NameError.
    """
    # Intentionally do nothing
    return


all_tracks, gated_tracks = run_step7_pipeline(
    VIDEO_PATH, DIFF_THRESHOLD, MIN_AREA, CENTROID_THRESHOLD
)

## <font color = lime> 7.20c: cleanup 7.9 (one time cleanup)

In [ ]:
def step7_artifacts_cleanup():
    g = globals()
    if "STEP7_ARTIFACTS" not in g or not isinstance(g["STEP7_ARTIFACTS"], dict):
        g["STEP7_ARTIFACTS"] = {}

    A = g["STEP7_ARTIFACTS"]

    # Ensure bins exist
    A.setdefault("pre", {})
    A.setdefault("post", {})
    A.setdefault("viz", {})
    A.setdefault("_legacy", {})  # keep old junk so nothing is lost

    # --- Move known loose keys into bins (safe + reversible) ---
    move_map = {
        # pre
        "all_tracks": ("pre", "all_tracks"),
        "gap_stats": ("pre", "gap_stats"),

        # post
        "gated_tracks": ("post", "gated_tracks"),
        "gated_track_ids": ("post", "gated_track_ids"),
        "best_track_id": ("post", "best_track_id"),
        "quality_scores": ("post", "quality_scores"),

        # viz
        "ORIGINAL_BEST_TRACK": ("viz", "best_track_original"),
        "INTERPOLATED_BEST_TRACK": ("viz", "best_track_interpolated"),
        "gated_tracks_retro": ("viz", "gated_tracks_retro"),
        "plot_data_csv": ("viz", "plot_data_csv"),
        "plots": ("viz", "plots"),
    }

    # Move anything in root that matches move_map
    root_keys = list(A.keys())
    for k in root_keys:
        if k in ("pre", "post", "viz", "_legacy"):
            continue

        if k in move_map:
            bin_name, new_key = move_map[k]
            # Don’t overwrite if already present; stash into _legacy instead
            if new_key in A[bin_name]:
                A["_legacy"][k] = A.pop(k)
            else:
                A[bin_name][new_key] = A.pop(k)
        else:
            # Unknown junk: preserve it but get it out of the root
            A["_legacy"][k] = A.pop(k)

    # --- Normalize shapes (dict keyed by track_id is ideal) ---
    # all_tracks
    if "all_tracks" in A["pre"]:
        at = A["pre"]["all_tracks"]
        if isinstance(at, list):
            A["pre"]["all_tracks"] = {t.track_id: t for t in at}
    # gated_tracks
    if "gated_tracks" in A["post"]:
        gt = A["post"]["gated_tracks"]
        if isinstance(gt, list):
            A["post"]["gated_tracks"] = {t.track_id: t for t in gt}

    # Report
    print("\nSTEP7_ARTIFACTS cleanup complete.")
    print("  root keys now :", sorted([k for k in A.keys()]))
    print("  pre keys      :", sorted(A["pre"].keys()))
    print("  post keys     :", sorted(A["post"].keys()))
    print("  viz keys      :", sorted(A["viz"].keys()))
    print("  legacy keys   :", sorted(A["_legacy"].keys()))

step7_artifacts_cleanup()


In [ ]:
A = STEP7_ARTIFACTS
print("pre/all_tracks:", "all_tracks" in A.get("pre", {}), "len=", len(A["pre"].get("all_tracks", {})))
print("post/gated_tracks:", "gated_tracks" in A.get("post", {}), "len=", len(A["post"].get("gated_tracks", {})))
print("viz keys:", sorted(A.get("viz", {}).keys()))
print("legacy keys:", sorted(A.get("_legacy", {}).keys()))
